# QuantumJam 2026 · Curva de energía de $\mathrm{H_4}$ con VQE

Cadena lineal de 4 H con espaciado uniforme $d$, base STO-3G, neutra y singlete: 4 orbitales espaciales → 8 espín-orbitales → **8 qubits** (Jordan-Wigner), 4 electrones. Calculamos con VQE la curva $E(d)$ para $d \in [0{,}6;\ 3{,}0]$ Å, con $R_e$, $D_e = E(3{,}0\ \text{Å}) - E(R_e)$ y una barra de error $\pm\sigma$ por punto, y la comparamos con CCSD y con la energía exacta (FCI). Enunciado completo en `docs/challenge/`.

### El problema: por qué la zona estirada es difícil

Cerca del equilibrio ($R_e \approx 0{,}89$ Å) el estado fundamental es casi un solo determinante (Hartree-Fock pesa 0,985 a 0,6 Å) y CCSD es prácticamente exacto (< 0,01 mHa de error). Al estirar la cadena, los 4 electrones terminan en cuatro orbitales 1s casi desacoplados, un singlete con muchos determinantes de peso parecido: el estado se vuelve **multirreferencia** (HF pesa 0,58 a 1,8 Å y 0,27 a 3,0 Å). Un método de una sola referencia como CCSD no lo describe bien, y como no es variacional, **cae por debajo de FCI**:

| $d$ (Å) | 0,9 | 1,6 | **1,8** | 2,0 | 2,4 | 2,6 | 3,0 |
|---|---|---|---|---|---|---|---|
| $E_\text{CCSD} - E_\text{FCI}$ (mHa) | +0,01 | −3,0 | **−9,1** | −18,3 | −30,6 | −30,6 | −23,0 |

(`docs/analysis/data/h4_reference.json`; CCSD(T) es peor todavía). Por eso el desafío pide, para $d \geq 1{,}8$ Å, $|E_\text{VQE} - E_\text{FCI}| + \sigma < |E_\text{CCSD} - E_\text{FCI}|$; 1,8 Å es el punto más exigente porque ahí el margen es el más chico (9,1 mHa). La meta máxima es precisión química, $< 1{,}6$ mHa. CCSD también subestima $D_e$ en ~23 mHa (≈ 290 contra 313 mHa de FCI). Un detalle más: desde ~2,4 Å el primer excitado es un triplete a pocos mHa (0,46 mHa a 3,0 Å), así que la energía sola no alcanza para saber si el estado es el correcto.

### Qué hacemos

| Paso | Sección | Qué sale |
|---|---|---|
| Referencias clásicas HF, CCSD, FCI (solo para comparar y elegir puntos de partida) | 1 | tabla `refs` |
| Hamiltoniano de 8 qubits por Jordan-Wigner, validado contra HF y FCI | 2 | `qubit_hamiltonian(d)` |
| Ansatz **2-QUpCCGSD podado**: 19 excitaciones de qubit como rotaciones de Givens, 465 `cz`, conserva $N$ y $S_z$ | 3 | `build_ansatz()` |
| Búsqueda de parámetros con statevector (permitido), *warm start* entre distancias vecinas y varios arranques | 4 | `params` |
| Energía **reportada**: medición con `SamplerV2` de `qiskit_aer`, shots explícitos y $\sigma$ desde los bitstrings | 4 | tabla `vqe` |
| Curva, $R_e$, $D_e$ y tabla por punto (energía, $\sigma$, shots, `cz`) | 5 | gráficos y tablas |
| Confiabilidad en la zona estirada **sin mirar FCI** ($\langle S^2\rangle$, varianza, robustez) y recién después la comparación con CCSD y FCI | 6 | evidencia y veredicto |
| Resumen de resultados, decisiones, qué no funcionó y limitaciones | Resultados clave, 7 | |

Reglas que respetamos en todo el notebook: el algoritmo es VQE; **ninguna energía reportada sale del statevector** (el statevector solo se usa para buscar parámetros y validar el ansatz); HF, CCSD y FCI son referencias, nunca resultados.

### Cómo leer y correr el notebook

- **Colab:** abrir el notebook y *Entorno de ejecución → Ejecutar todas*. La primera celda instala las dependencias con las versiones fijadas (las mismas que `requirements.txt`). Si Colab pide reiniciar el entorno después del `%pip`, reiniciar y volver a ejecutar todo. Es autocontenido: no importa nada del repo.
- **Local:** `nix develop -c jupyter lab` (mismas versiones; ver `README` del repo).
- La celda **0.1** es el contrato que comparten todas las secciones: grilla `D`, geometría, convención de qubits (`HF_BITSTRING = "00110011"`: qubit $p$ = $(p,\alpha)$, qubit $4+p$ = $(p,\beta)$), transpilación y conteo de `cz` (base `cz, sx, x, rz`, qubits en línea) y los nombres de las tablas `refs`, `params`, `vqe`.
- Cada sección termina con un markdown de **decisiones**: qué elegimos, por qué y qué probamos que no funcionó. La sección 7 los resume y analiza las limitaciones.
- Todas las energías son **totales** (incluyen la repulsión nuclear), en Hartree; los errores, en mHa.
- **Resultados clave** ($R_e$, $D_e$, ¿le ganamos a CCSD?, ¿precisión química?, `cz` y shots) en la celda *Resultados clave*, justo antes de la sección 7.

## 0. Instalación

In [ ]:
%pip install -q qiskit==2.5.2 qiskit-aer==0.17.2 pyscf==2.14.0 qiskit-ibm-runtime==0.47.0 numpy scipy pandas matplotlib

## 0.1 Contrato compartido

Nombres, firmas y convenciones que usa todo el notebook. Cada sección reemplaza su `NotImplementedError`; cambios a este contrato se avisan en el issue.

In [ ]:
# Contrato compartido: nombres, firmas y convenciones que usa todo el notebook.
# Cada sección implementa su parte; nadie cambia esto sin avisar en el issue correspondiente.
import numpy as np
import pandas as pd
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import SparsePauliOp
from qiskit.transpiler import CouplingMap

# --- Grilla y geometría -------------------------------------------------------
BASIS = "sto-3g"
D = np.round(np.concatenate([np.arange(0.6, 1.75, 0.1), np.arange(1.8, 3.01, 0.2)]), 2)  # Å, 19 puntos
STRETCHED = 1.8    # Å: desde acá hay que ganarle a CCSD
DISSOCIATED = 3.0  # Å: D_e = E(3,0) - E(R_e)
CHEM_ACC = 1.6e-3  # Ha


def h4_geometry(d: float) -> str:
    """Cadena lineal de 4 H sobre el eje z, separados por d Å (formato PySCF)."""
    return "; ".join(f"H 0 0 {i * d:.6f}" for i in range(4))


# --- Convención de qubits (Jordan-Wigner) --------------------------------------
# Orbitales espaciales MO de RHF, p = 0..3 en orden de energía.
# qubit p     = espín-orbital (p, α)
# qubit 4 + p = espín-orbital (p, β)
# |1> = ocupado. Qiskit es little-endian: el label de un estado se lee q7 ... q0.
N_ORB, N_QUBITS, N_ALPHA, N_BETA = 4, 8, 2, 2
HF_BITSTRING = "00110011"  # q0, q1 (α 0,1) y q4, q5 (β 0,1) ocupados


def alpha_qubit(p: int) -> int:
    return p


def beta_qubit(p: int) -> int:
    return N_ORB + p


# --- Transpilación y conteo de cz (igual para todos) ---------------------------
BASIS_GATES = ["cz", "sx", "x", "rz"]


def transpile_line(qc: QuantumCircuit) -> QuantumCircuit:
    return transpile(qc, basis_gates=BASIS_GATES, coupling_map=CouplingMap.from_line(qc.num_qubits),
                     optimization_level=3, seed_transpiler=0)


def cz_count(qc: QuantumCircuit) -> int:
    return transpile_line(qc).count_ops().get("cz", 0)


# --- Firmas que implementa cada sección ----------------------------------------
# Tablas compartidas (energías totales en Ha, con repulsión nuclear):
#   refs   : DataFrame  d, e_hf, e_ccsd, e_fci           (#1)
#   params : dict       d -> np.ndarray θ*               (#5)
#   vqe    : DataFrame  d, energy, sigma, shots, cz      (#6)

def references(d: float) -> dict:
    """#1. {'d', 'e_hf', 'e_ccsd', 'e_fci'}."""
    raise NotImplementedError


def qubit_hamiltonian(d: float) -> tuple[SparsePauliOp, float]:
    """#2. (H electrónico como SparsePauliOp de 8 qubits en la convención de arriba, e_nuc)."""
    raise NotImplementedError


def build_ansatz() -> QuantumCircuit:
    """#4. Circuito parametrizado de 8 qubits que arranca en HF_BITSTRING."""
    raise NotImplementedError


## 1. Referencias clásicas (HF, CCSD, FCI)

Energías de referencia en toda la grilla `D` con PySCF. **FCI** es la respuesta exacta en la base STO-3G, contra la que se mide todo. **CCSD** es el método clásico a superar en la zona estirada. Ninguna de las dos es un resultado del VQE: solo sirven para comparar y para elegir puntos de partida.

Esta celda define `references(d)` y la tabla `refs` (`d, e_hf, e_ccsd, e_fci`), que también se guarda en `results/classical.csv`. Usa el mismo procedimiento de RHF que `h4_scf` de la sección 2 (`conv_tol = 1e-11`). Las energías no dependen del signo de los orbitales. Para cualquier cosa que dependa de los orbitales (vector CI, amplitudes), usar `h4_scf(d)`.

**Decisiones**

- Se usa **RHF**, no UHF. Desde ~1,2 Å RHF tiene una inestabilidad RHF → UHF: la solución UHF baja la energía pero rompe la simetría de espín. Queremos un singlete con orbitales restringidos, que es lo que usa el Hamiltoniano de qubits. El chequeo de **estabilidad interna** (RHF → RHF) pasa en todos los puntos: no hay una solución RHF más baja que la encontrada.
- Tolerancias ajustadas (`1e-11` en SCF, `1e-10` en CCSD, `1e-12` en FCI). El código verifica la convergencia de RHF y CCSD en cada punto y que el estado FCI sea singlete (⟨S²⟩ = 0). Todos convergen con el `init_guess` por defecto, así que no hizo falta arrancar desde la densidad del punto vecino.

**Qué muestran los números**

- Cerca del equilibrio (0,6–1,1 Å) CCSD está a menos de 0,01 mHa de FCI.
- Desde ~1,1 Å CCSD **deja de ser variacional** y queda por debajo de FCI. El error es −9,1 mHa a 1,8 Å (el punto más exigente de la zona juzgada), llega a −30,6 mHa a 2,6 Å y vuelve a −23,0 mHa a 3,0 Å.
- Para ganarle a CCSD hace falta `|E_VQE − E_FCI| + σ < 9,1 mHa` a 1,8 Å y < 18 mHa desde 2,0 Å.

In [ ]:
# 1. Referencias clásicas: HF, CCSD y FCI en toda la grilla D (#1).
# Solo sirven para comparar y elegir puntos de partida; nunca son resultados.
# Las energías no dependen del signo de los orbitales: los orbitales "oficiales" (signo fijo,
# los del Hamiltoniano) son los de h4_scf(d) en la sección 2. Acá se usa el mismo procedimiento.
import os
import matplotlib.pyplot as plt
from pyscf import cc, fci, gto, scf


def references(d: float) -> dict:
    """#1. {'d', 'e_hf', 'e_ccsd', 'e_fci'} en Ha (energías totales)."""
    mol = gto.M(atom=h4_geometry(d), basis=BASIS, unit="Angstrom", charge=0, spin=0, verbose=0)
    mf = scf.RHF(mol)
    mf.conv_tol = 1e-11
    mf.kernel()
    # Al estirar, el SCF podría caer en un punto silla. Si la solución tiene una inestabilidad
    # interna (RHF -> RHF), seguimos la dirección inestable y reconvergemos (en esta grilla no pasa).
    for _ in range(5):
        mo, _, stable, _ = mf.stability(return_status=True)
        if stable:
            break
        mf.kernel(dm0=mf.make_rdm1(mo, mf.mo_occ))
    assert mf.converged, f"RHF no convergió en d={d}"

    ccsd = cc.CCSD(mf)
    ccsd.conv_tol, ccsd.max_cycle = 1e-10, 500
    ccsd.kernel()
    assert ccsd.converged, f"CCSD no convergió en d={d}"

    solver = fci.FCI(mf)
    solver.conv_tol = 1e-12
    e_fci, ci = solver.kernel()
    s2, _ = solver.spin_square(ci, N_ORB, (N_ALPHA, N_BETA))
    assert abs(s2) < 1e-6, f"FCI no es singlete en d={d}: <S²>={s2:.3g}"
    return {"d": float(d), "e_hf": mf.e_tot, "e_ccsd": ccsd.e_tot, "e_fci": e_fci}


refs = pd.DataFrame([references(d) for d in D])

# Chequeos de cordura: FCI es exacto en la base (variacional), CCSD es casi exacto cerca del equilibrio.
assert (refs.e_fci <= refs.e_hf + 1e-10).all()
near_eq = refs[(refs.d >= 0.8) & (refs.d <= 1.0)]
assert ((near_eq.e_ccsd - near_eq.e_fci).abs() < CHEM_ACC).all()

os.makedirs("results", exist_ok=True)
refs.to_csv("results/classical.csv", index=False)

err_ccsd = 1e3 * (refs.e_ccsd - refs.e_fci)  # mHa, con signo
print(refs.assign(ccsd_menos_fci_mHa=err_ccsd).to_string(
    index=False, formatters={"d": "{:.2f}".format, "ccsd_menos_fci_mHa": "{:+.3f}".format}, float_format="%.6f"))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.2))
ax1.plot(refs.d, refs.e_hf, "s-", ms=4, label="RHF")
ax1.plot(refs.d, refs.e_ccsd, "^-", ms=4, label="CCSD")
ax1.plot(refs.d, refs.e_fci, "o-", ms=4, color="k", label="FCI (exacta)")
ax1.axvspan(STRETCHED, D.max() + 0.05, color="0.92", zorder=0, label=f"zona estirada (d ≥ {STRETCHED} Å)")
ax1.set(xlabel="d (Å)", ylabel="E (Ha)", title="Curva de energía de H₄ (STO-3G)")
ax1.legend()

ax2.semilogy(refs.d, err_ccsd.abs(), "^-", ms=4, color="C1", label="|E_CCSD − E_FCI|")
ax2.axhline(1e3 * CHEM_ACC, ls="--", color="C2", label="precisión química (1,6 mHa)")
ax2.axvspan(STRETCHED, D.max() + 0.05, color="0.92", zorder=0)
ax2.set(xlabel="d (Å)", ylabel="error (mHa)", title="Error de CCSD: lo que hay que superar")
ax2.legend()
plt.tight_layout()
plt.show()

## 2. Hamiltoniano de qubits

In [ ]:
import copy
from functools import lru_cache

from pyscf import ao2mo, fci, gto, scf
from pyscf.fci import cistring


def _so(p: int, spin: int) -> int:
    return alpha_qubit(p) if spin == 0 else beta_qubit(p)


@lru_cache(maxsize=None)
def _jw_ladder(n_qubits: int) -> tuple[tuple[SparsePauliOp, ...], tuple[SparsePauliOp, ...]]:
    annihilation = []
    for j in range(n_qubits):
        z_string = list(range(j))
        annihilation.append(SparsePauliOp.from_sparse_list(
            [("Z" * j + "X", z_string + [j], 0.5), ("Z" * j + "Y", z_string + [j], 0.5j)],
            num_qubits=n_qubits))
    creation = [a.adjoint() for a in annihilation]
    return tuple(creation), tuple(annihilation)


def jordan_wigner(h1_so: np.ndarray, h2_so: np.ndarray | None = None, atol: float = 1e-12,
                  hermitian: bool = True) -> SparsePauliOp:
    n = h1_so.shape[0]
    cdag, a = _jw_ladder(n)
    terms = [SparsePauliOp("I" * n, 0.0)]
    for p, q in np.argwhere(np.abs(h1_so) > atol):
        terms.append(h1_so[p, q] * cdag[p] @ a[q])
    if h2_so is not None:
        for p, q, r, s in np.argwhere(np.abs(h2_so) > atol):
            if p == r or q == s:
                continue
            terms.append(0.5 * h2_so[p, q, r, s] * cdag[p] @ cdag[r] @ a[s] @ a[q])
    op = SparsePauliOp.sum(terms).simplify(atol=atol)
    if not hermitian:
        return op
    if np.max(np.abs(op.coeffs.imag), initial=0.0) > 1e-10:
        raise ValueError("Operador no hermítico (coeficientes complejos): revisar índices o usar hermitian=False")
    return SparsePauliOp(op.paulis, op.coeffs.real)


def spatial_to_spin_orbitals(h1: np.ndarray, eri: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    n_orb = h1.shape[0]
    n_so = 2 * n_orb
    h1_so = np.zeros((n_so, n_so))
    h2_so = np.zeros((n_so, n_so, n_so, n_so))
    for s1 in (0, 1):
        i = [_so(p, s1) for p in range(n_orb)]
        h1_so[np.ix_(i, i)] = h1
        for s2 in (0, 1):
            k = [_so(p, s2) for p in range(n_orb)]
            h2_so[np.ix_(i, i, k, k)] = eri
    return h1_so, h2_so


def _distance_key(d: float) -> float:
    return round(float(d), 6)


@lru_cache(maxsize=None)
def _h4_scf_cached(d: float) -> scf.hf.RHF:
    mol = gto.M(atom=h4_geometry(d), basis=BASIS, unit="Angstrom", charge=0, spin=0, verbose=0)
    mf = scf.RHF(mol)
    mf.conv_tol = 1e-11
    mf.kernel()
    if not mf.converged:
        raise RuntimeError(f"RHF no convergió en d = {d}")
    c = mf.mo_coeff
    significant = np.abs(c) > 1e-4
    if not significant.any(axis=0).all():
        raise RuntimeError(f"Orbital sin coeficientes > 1e-4 en d = {d}: no se puede fijar el signo")
    first = np.argmax(significant, axis=0)
    mf.mo_coeff = c * np.sign(c[first, np.arange(c.shape[1])])
    return mf


def h4_scf(d: float) -> scf.hf.RHF:
    mf = copy.copy(_h4_scf_cached(_distance_key(d)))
    for attr in ("mo_coeff", "mo_energy", "mo_occ"):
        setattr(mf, attr, getattr(mf, attr).copy())
    return mf


@lru_cache(maxsize=None)
def _qubit_hamiltonian_cached(d: float) -> tuple[SparsePauliOp, float]:
    mf = _h4_scf_cached(d)
    mol, c = mf.mol, mf.mo_coeff
    h1 = c.T @ mf.get_hcore() @ c
    eri = ao2mo.restore(1, ao2mo.full(mol, c), c.shape[1])
    h1_so, h2_so = spatial_to_spin_orbitals(h1, eri)
    return jordan_wigner(h1_so, h2_so), mol.energy_nuc()


def qubit_hamiltonian(d: float) -> tuple[SparsePauliOp, float]:
    op, e_nuc = _qubit_hamiltonian_cached(_distance_key(d))
    return op.copy(), e_nuc


_STRINGS = cistring.make_strings(range(N_ORB), N_ALPHA)
SECTOR = np.array([int(a) | (int(b) << N_ORB) for a in _STRINGS for b in _STRINGS])


def sector_ground_energy(op: SparsePauliOp) -> float:
    m = op.to_matrix()
    return np.linalg.eigvalsh(m[np.ix_(SECTOR, SECTOR)])[0].real


def hf_energy_from_op(op: SparsePauliOp) -> float:
    i = int(HF_BITSTRING, 2)
    return op.to_matrix()[i, i].real


def validate_hamiltonian(d: float) -> dict:
    op, e_nuc = qubit_hamiltonian(d)
    mf = h4_scf(d)
    e_fci = fci.FCI(mf).kernel()[0]
    e_hf_op = hf_energy_from_op(op) + e_nuc
    e_gs_op = sector_ground_energy(op) + e_nuc
    return {"d": float(d), "n_paulis": len(op), "e_hf": mf.e_tot, "e_fci": e_fci,
            "err_hf": abs(e_hf_op - mf.e_tot), "err_fci": abs(e_gs_op - e_fci)}


ham_check = pd.DataFrame([validate_hamiltonian(d) for d in D])
assert (ham_check[["err_hf", "err_fci"]] < 1e-8).all().all(), ham_check
ham_check


### Decisiones (#2)

**Firma.** `qubit_hamiltonian(d) -> (op, e_nuc)`, como dice el contrato 0.1 ([ADR-0001](https://github.com/Santiago-Garrote/QJ/blob/main/docs/adr/0001-firma-qubit-hamiltonian.md)). `op` es solo la parte electrónica: energía total = $\langle op \rangle + e_\text{nuc}$. El estado HF es la constante `HF_BITSTRING = "00110011"` porque no depende de $d$.

**Orden de qubits.** Qubit $p$ = espín-orbital $(p, \alpha)$, qubit $4+p$ = $(p, \beta)$, con $p = 0..3$ los orbitales RHF ordenados por energía. `HF_BITSTRING` deja ocupados q0, q1, q4, q5 (Qiskit es little-endian: el label se lee q7 … q0).

**Hamiltoniano.** Integrales en base MO de RHF: $h_{pq} = C^T h^\text{core} C$ y $(pq|rs)$ con `ao2mo` (notación química). En espín-orbitales:

$$\hat H = \sum_{PQ} h_{PQ}\, a_P^\dagger a_Q + \tfrac12 \sum_{PQRS} (PQ|RS)\, a_P^\dagger a_R^\dagger a_S a_Q,$$

con el espín conservado dentro de cada par $(P,Q)$ y $(R,S)$. Jordan-Wigner a mano: $a_j = Z_0 \cdots Z_{j-1}\,(X_j + iY_j)/2$ ([ADR-0002](https://github.com/Santiago-Garrote/QJ/blob/main/docs/adr/0002-jordan-wigner-a-mano.md)). `jordan_wigner(h1_so, h2_so)` es genérica: sirve también para $N$, $S_z$, $S^2$ (#8) y para el tapering (#14). Por defecto exige un operador hermítico (coeficientes reales) y falla si no lo es, porque eso delata un error de índices; para operadores no hermíticos como $S^+ = \sum_p a^\dagger_{p\alpha} a_{p\beta}$ usar `hermitian=False`, que devuelve los coeficientes complejos.

**Signo de los orbitales.** PySCF puede devolver un orbital con el signo invertido entre una distancia y la siguiente. El espectro no cambia, pero los coeficientes de $\hat H$ sí, y eso rompe el *warm start* de #5. Se fija el signo para que el primer coeficiente de cada orbital con $|c| > 10^{-4}$ sea positivo (si algún orbital no tiene ninguno, se lanza un error en vez de anularlo) ([ADR-0003](https://github.com/Santiago-Garrote/QJ/blob/main/docs/adr/0003-signo-orbitales-y-cache.md)). Verificado: ningún término de Pauli cambia de signo de golpe entre distancias vecinas (los únicos cambios, cerca de 0,67 Å, son coeficientes que cruzan cero de forma suave).

**Orbitales RHF en toda la grilla.** RHF es inestable hacia UHF desde ~1,1 Å (`docs/analysis/state-of-the-art.md`; verificado con `mf.stability(external=True)`), pero se mantienen los orbitales RHF: solo definen la base (FCI no cambia), el estado de referencia sigue siendo un singlete y ⟨S²⟩ = 0 sigue sirviendo como diagnóstico en la sección 6 (#8). Con orbitales UHF la referencia quedaría contaminada de espín.

**Caché.** RHF y el Hamiltoniano se calculan una sola vez por distancia (`lru_cache` en funciones privadas). Las funciones públicas devuelven **copias**: `qubit_hamiltonian` devuelve `op.copy()` y `h4_scf` una copia del `mf` con sus propios `mo_coeff`, `mo_energy` y `mo_occ`, así que modificar lo devuelto (o correr `mf.kernel()` / `mf.stability()` sobre él) no altera el caché ni desincroniza orbitales y Hamiltoniano. La clave del caché es `d` redondeada a 6 decimales: `qubit_hamiltonian(0.1 * 7)` reutiliza el de `0.7`.

**Validación** (tabla `ham_check`, las 19 distancias):

| Prueba | Criterio | Resultado |
|---|---|---|
| $\langle \text{HF}\vert op \vert \text{HF}\rangle + e_\text{nuc}$ vs. $E_\text{HF}$ de PySCF | < 1e-8 Ha | máx. ≈ 2e-14 Ha |
| Mínimo de `op` en el sector $N=4$, $S_z=0$ (36 estados) + $e_\text{nuc}$ vs. FCI | < 1e-8 Ha | máx. ≈ 2e-14 Ha |
| Términos de Pauli | — | 185 (184 más la identidad) en todas las distancias, mismo conjunto de Paulis |

`SECTOR` son los 36 índices de la base computacional con 2 α y 2 β, **en el orden `ci[ia, ib]` de PySCF** (`cistring.make_strings`): un vector CI `ci` de PySCF corresponde a `psi[SECTOR] = ci.ravel()` (signos verificados contra `fci.direct_spin1.energy` en estados al azar, error < 3e-14 Ha). Es la misma definición que usa la sección 3 (#4).

La validación contra FCI se hace en el sector $N=4$, $S_z=0$: el mínimo del espectro completo (256 estados) puede tener otro número de electrones. FCI se calcula acá mismo con los orbitales de `h4_scf`; cuando esté `refs` (#1) se puede comparar también contra `refs.e_fci`.

**Qué no funcionó.** La primera versión armaba los productos con `A.compose(B)`, que en Qiskit es la matriz $B \cdot A$, no $A \cdot B$: quedaba $a_Q a_P^\dagger$ en vez de $a_P^\dagger a_Q$, o sea el Hamiltoniano con partículas y huecos intercambiados. Con 4 electrones en 8 espín-orbitales (medio llenado) ese operador tiene **el mismo espectro en el sector** $N=4$, así que la prueba contra FCI pasaba igual; solo la prueba de HF lo detectó (el estado HF aparecía en `11001100`). Se usa `@` (`A @ B` = $A \cdot B$). Por eso las dos pruebas son necesarias.

## 3. Ansatz

### Decisión (#4)

Elegimos un **2-QUpCCGSD podado**: excitaciones de pares $(p_\uparrow p_\downarrow)\to(q_\uparrow q_\downarrow)$ y simples $p\to q$ (mismo ángulo en α y β), como excitaciones de **qubit** (sin cadenas Z), en dos capas, con 19 de las 24 excitaciones.

Comparación con statevector (error vs. FCI en mHa a 0,9 / 1,8 / 3,0 Å; `cz` tras transpilar a `cz, sx, x, rz` en línea):

| Ansatz | `cz` | Parámetros | Error (mHa) |
|---|---|---|---|
| UCCSD (fermiónico) | 1277 | 26 | 0,03 / 3,3 / 0,46 |
| UCCSD de qubits | 1247 | 26 | 0,003 / 3,0 / 0,21 |
| 1-UpCCGSD | 474 | 12 | 14 / 11 / 0,29 |
| 2-UpCCGSD (fermiónico) | 982 | 24 | exacto |
| 2-QUpCCGSD (Givens) | 590 | 24 | exacto |
| **2-QUpCCGSD podado (elegido)** | **465** | **19** | **≤ 0,03 en toda la grilla** |
| 2-QUpCCGSD podado más agresivo | 411 | 17 | ≤ 0,5 (en 6 distancias) |
| QEB-ADAPT-VQE (un circuito por distancia) | 461–573 | 16–19 | ≤ 0,01 |
| Hardware-efficient (RY·RZ + CZ en línea), 6 capas | 42 | 112 | 53 / 23 / 0,67 |
| Hardware-efficient, 8 capas (optimizando fidelidad con FCI) | 56 | 144 | 1 − F = 1e-5 / 1,5e-3 / 2e-4 |
| *CCSD (referencia)* | – | – | 0,007 / −9,1 / −23 |

Por qué:

- **Llega a FCI en la zona estirada.** UCCSD falla ahí (3,3 mHa a 1,8 Å, fuera de precisión química) por la misma razón que CCSD: el estado se vuelve multirreferencia. Las excitaciones generalizadas de pares sí lo capturan.
- **Conserva N = 4 y S_z = 0**: cada excitación mueve electrones dentro del mismo bloque de espín. Lo verificamos numéricamente abajo, y sirve para el argumento de confiabilidad (#8).
- **Menos `cz`**: cada excitación es una rotación de Givens entre dos estados de la base, implementada con un multiplexor de RY en código Gray (14 CX por doble) en lugar de `PauliEvolutionGate` (837 → 590 `cz`). Después podamos de a una las excitaciones de menor ángulo mientras el error siguiera < 0,1 mHa en 6 distancias (590 → 465 `cz`).

Qué no funcionó o descartamos:

- UCCSD y 1-UpCCGSD: no llegan a precisión química en la zona estirada.
- Sintetizar las excitaciones con `PauliEvolutionGate`: correcto pero caro en `cz`.
- **Hardware-efficient** (RY·RZ por qubit + cadena de CZ): muy barato en `cz` (42–56), pero optimizando la energía queda trabado lejos de FCI (53 mHa a 0,9 Å y 23 mHa a 1,8 Å con 6 capas y 6 arranques). Con 8 capas *puede* representar el estado (fidelidad 1 − 1,5e-3 a 1,8 Å si se optimiza contra el vector FCI, cosa que en un VQE real no se puede), así que el problema es el paisaje de optimización, no la expresividad. Además no conserva N ni S_z (fuga de hasta 0,4 %), lo que complica #8. Queda como opción para los extras con ruido si alguien encuentra cómo optimizarlo.
- **QEB-ADAPT-VQE** (sugerido en #4): con un pool de 90 excitaciones de qubit converge a ≤ 0,01 mHa con 16–19 excitaciones, pero necesita 461–573 `cz` (sus dobles αβ genéricas tocan qubits lejanos en la línea) y un circuito distinto por distancia. Incluso cortado justo al llegar a precisión química necesita 467 `cz` a 1,8 Å. No mejora al ansatz elegido.
- La versión de 17 excitaciones (411 `cz`) queda como alternativa si los extras con ruido necesitan menos compuertas, a costa de errores de hasta 0,5 mHa.

Limitaciones:

- **El paisaje de energía tiene mínimos locales.** Con un solo arranque, algunas distancias quedan a 1 mHa o hasta decenas de mHa; con 30 arranques el mejor llega a ≤ 0,003 mHa. Con los 10 arranques de la validación de abajo, el peor punto queda entre 0,04 y 0,13 mHa según la corrida (diferencias numéricas mínimas cambian a qué mínimo cae cada arranque). La optimización (#5) tiene que usar multi-start.
- **Contaminación de espín.** Conservar N y S_z no garantiza S² = 0: las excitaciones de qubit no están adaptadas al espín, así que si la optimización cae en otro mínimo el estado puede mezclarse con el triplete, que en la zona estirada está casi degenerado (brecha de 0,5 mHa a 3,0 Å, así que la energía no lo delata). Con el Hamiltoniano de #2 (orbitales con signo fijo) la validación llega al singlete: ⟨S²⟩ ≤ 3e-7. Con un stand-in previo cuyos orbitales no tenían el signo fijo, el warm start entre distancias caía en otros mínimos y vimos ⟨S²⟩ de hasta 3e-2 (~1,5 % de triplete) desde 2,0 Å. Por eso la tabla de abajo reporta ⟨S²⟩ y la brecha, #8 tiene que reportarlo, y #5 puede asegurarlo con una penalización λ⟨S²⟩.
- **465 `cz` es mucho para hardware ruidoso.** Con 0,1 % de error por `cz`, la probabilidad de al menos una falla es 1 − 0,999⁴⁶⁵ ≈ 37 %. Para el desafío central (sin ruido) no importa; para los extras con ruido conviene la versión podada más agresiva o menos qubits (#14).

La validación de abajo usa el Hamiltoniano de #2 (`qubit_hamiltonian`) restringido a los 36 estados con N = 4 y S_z = 0, y toma como FCI su autovalor mínimo. Mientras #2 no estaba, la exploración usó un stand-in armado con PySCF (misma convención de signos, verificado en #4); con el Hamiltoniano de #2 los resultados son los mismos.


In [ ]:
# Ansatz elegido (#4): 2-QUpCCGSD podado, 19 excitaciones de qubit como rotaciones de Givens.
import itertools
from qiskit.circuit import ParameterVector


def givens_single(qc: QuantumCircuit, t, i: int, k: int) -> None:
    """exp(θ(Q†_k Q_i − h.c.)): rota |q_i=1, q_k=0⟩ hacia |q_i=0, q_k=1⟩."""
    qc.cx(k, i)
    qc.cry(2 * t, i, k)
    qc.cx(k, i)


def _ucry(qc: QuantumCircuit, t, controls: list, target: int, pattern: tuple) -> None:
    """RY(2θ) en target solo cuando controls == pattern. Multiplexor Gray: 2^n CX."""
    n = 2 ** len(controls)
    gray = [m ^ (m >> 1) for m in range(n)]
    idx = sum(b << c for c, b in enumerate(pattern))
    for m in range(n):
        qc.ry(2 * t * (-1) ** bin(idx & gray[m]).count("1") / n, target)
        qc.cx(controls[(gray[m] ^ gray[(m + 1) % n]).bit_length() - 1], target)


def givens_double(qc: QuantumCircuit, t, i: int, j: int, k: int, l: int) -> None:
    """exp(θ(Q†_k Q†_l Q_j Q_i − h.c.)): rota |ij=11, kl=00⟩ hacia |ij=00, kl=11⟩ (14 CX)."""
    for m in (i, j, l):
        qc.cx(k, m)
    _ucry(qc, t, [i, j, l], k, (1, 1, 0))
    for m in (l, j, i):
        qc.cx(k, m)


# (capa, tipo, p, q). "D": par (p↑, p↓) → (q↑, q↓). "S": simple p → q, en α y en β con el mismo θ.
ANSATZ_OPS = [
    (0, "D", 0, 2), (0, "D", 1, 2), (0, "D", 1, 3), (0, "D", 2, 3),
    (0, "S", 0, 1), (0, "S", 0, 3), (0, "S", 1, 2), (0, "S", 1, 3), (0, "S", 2, 3),
    (1, "D", 0, 1), (1, "D", 0, 3), (1, "D", 1, 2), (1, "D", 1, 3), (1, "D", 2, 3),
    (1, "S", 0, 2), (1, "S", 0, 3), (1, "S", 1, 2), (1, "S", 1, 3), (1, "S", 2, 3),
]


def build_ansatz() -> QuantumCircuit:
    """#4. Circuito de 8 qubits que arranca en HF_BITSTRING; un parámetro por excitación."""
    th = ParameterVector("θ", len(ANSATZ_OPS))
    qc = QuantumCircuit(N_QUBITS, name="QUpCCGSD-19")
    for q, bit in enumerate(reversed(HF_BITSTRING)):
        if bit == "1":
            qc.x(q)
    for t, (_, kind, p, q) in zip(th, ANSATZ_OPS):
        if kind == "D":
            givens_double(qc, t, alpha_qubit(p), beta_qubit(p), alpha_qubit(q), beta_qubit(q))
        else:
            givens_single(qc, t, alpha_qubit(p), alpha_qubit(q))
            givens_single(qc, t, beta_qubit(p), beta_qubit(q))
    return qc


ansatz = build_ansatz()
print(f"{ansatz.name}: {ansatz.num_parameters} parámetros, {cz_count(ansatz)} cz (transpilado a cz/sx/x/rz en línea)")


In [ ]:
# Validación del ansatz en toda la grilla, con statevector (permitido para elegir parámetros y ansatz).
# Usa el Hamiltoniano de #2 (`qubit_hamiltonian`) restringido al sector N=4, S_z=0 (36 estados):
# el ansatz conserva N y S_z, así que simular ahí es exacto y mucho más rápido que con 256 amplitudes.
from pyscf.fci import cistring, spin_op
from scipy.optimize import minimize
from qiskit.quantum_info import Statevector

_STRINGS = cistring.make_strings(range(N_ORB), N_ALPHA)
SECTOR = np.array([int(a) | (int(b) << N_ORB) for a in _STRINGS for b in _STRINGS])  # índices de qubits


def sector_hamiltonian(d: float) -> tuple[np.ndarray, float]:
    """H total (con e_nuc) de #2 en el sector de 36 estados, y su autovalor mínimo (= E_FCI)."""
    op, e_nuc = qubit_hamiltonian(d)
    M = op.to_matrix(sparse=True)[SECTOR][:, SECTOR].toarray()
    assert np.abs(M.imag).max() < 1e-12
    H = M.real + e_nuc * np.eye(len(SECTOR))
    return H, np.linalg.eigvalsh(H)[0]


def _qubit_excitation(create: tuple, annihilate: tuple) -> np.ndarray:
    """K = T − T† en el sector, con T = Q†...Q... (operadores de qubit, sin cadenas Z)."""
    def q(j, dag):
        lab = lambda p: "I" * (N_QUBITS - j - 1) + p + "I" * j
        return SparsePauliOp([lab("X"), lab("Y")], [0.5, -0.5j if dag else 0.5j])
    T = SparsePauliOp("I" * N_QUBITS)
    for c in create:
        T = T @ q(c, True)
    for a in annihilate:
        T = T @ q(a, False)
    M = (T - T.adjoint()).simplify().to_matrix(sparse=True)[SECTOR][:, SECTOR].toarray()
    return M.real


class SectorAnsatz:
    """El mismo ansatz simulado en el sector de 36 estados, con gradiente adjunto exacto (rápido)."""

    def __init__(self, ops):
        self.groups = []
        for _, kind, p, q in ops:
            if kind == "D":
                self.groups.append([_qubit_excitation((alpha_qubit(q), beta_qubit(q)), (beta_qubit(p), alpha_qubit(p)))])
            else:
                self.groups.append([_qubit_excitation((alpha_qubit(q),), (alpha_qubit(p),)),
                                    _qubit_excitation((beta_qubit(q),), (beta_qubit(p),))])
        self.n = len(self.groups)
        self.hf = (SECTOR == int(HF_BITSTRING, 2)).astype(float)

    def _U(self, k, t):  # cada generador cumple K³ = −K, así que exp(tK) = 1 + sin(t)K + (1 − cos(t))K²
        I = np.eye(len(SECTOR))
        U = I
        for K in self.groups[k]:
            U = (I + np.sin(t) * K + (1 - np.cos(t)) * K @ K) @ U
        return U

    def state(self, x):
        v = self.hf
        for k in range(self.n):
            v = self._U(k, x[k]) @ v
        return v

    def energy_grad(self, x, H):
        Us = [self._U(k, x[k]) for k in range(self.n)]
        v = self.hf
        for U in Us:
            v = U @ v
        lam = H @ v
        e, g = v @ lam, np.empty(self.n)
        for k in reversed(range(self.n)):
            g[k] = 2 * lam @ (sum(self.groups[k]) @ v)
            v, lam = Us[k].T @ v, Us[k].T @ lam
        return e, g


sector_ansatz = SectorAnsatz(ANSATZ_OPS)

# El circuito real y la simulación en el sector tienen que dar el mismo estado.
_x = np.random.default_rng(0).normal(size=ansatz.num_parameters)
_full = Statevector(ansatz.assign_parameters(_x)).data
assert np.isclose(np.linalg.norm(_full[SECTOR]), 1), "el ansatz se sale del sector N=4, S_z=0"
assert np.isclose(abs(np.vdot(_full[SECTOR], sector_ansatz.state(_x))), 1), "circuito ≠ simulación en el sector"

# El paisaje de energía tiene mínimos locales (algunos arranques quedan a decenas de mHa): por eso
# probamos varios arranques por distancia (cero, el óptimo de la distancia anterior y 8 al azar).
rng = np.random.default_rng(0)
rows, x0 = [], np.zeros(sector_ansatz.n)
for d in D:
    H, e_fci = sector_hamiltonian(d)
    starts = [np.zeros(sector_ansatz.n), x0] + [rng.normal(scale=0.7, size=sector_ansatz.n) for _ in range(8)]
    best = min((minimize(sector_ansatz.energy_grad, s, args=(H,), jac=True, method="BFGS", options={"gtol": 1e-10})
                for s in starts), key=lambda r: r.fun)
    x0 = best.x
    s2 = spin_op.spin_square0(sector_ansatz.state(best.x).reshape(6, 6), N_ORB, (N_ALPHA, N_BETA))[0]
    gap = np.diff(np.linalg.eigvalsh(H)[:2])[0]
    rows.append({"d": d, "e_ansatz": best.fun, "e_fci": e_fci, "err_mHa": (best.fun - e_fci) * 1e3,
                 "S2": s2, "gap_mHa": gap * 1e3})
ansatz_check = pd.DataFrame(rows)
print(f"error máximo vs. FCI en la grilla: {ansatz_check.err_mHa.abs().max():.3f} mHa "
      f"(zona estirada: {ansatz_check[ansatz_check.d >= STRETCHED].err_mHa.abs().max():.3f} mHa); "
      f"⟨S²⟩ máximo: {ansatz_check.S2.max():.1e} (singlete puro = 0, triplete = 2)")
ansatz_check


## 4. VQE

In [ ]:
# VQE, parte 1 (#5): optimización de θ*(d) en toda la grilla, con statevector (permitido para buscar
# parámetros). Las energías de esta celda son solo diagnóstico: la energía reportada la mide #6.
# Reutiliza de la sección 3 `sector_hamiltonian`, `sector_ansatz` (gradiente adjunto exacto), `ansatz`,
# `minimize`, `Statevector` y `spin_op`.
import time

_t0 = time.perf_counter()
H_SECTOR = {float(d): sector_hamiltonian(d) for d in D}  # d -> (H con e_nuc, E_FCI); E_FCI solo para comparar

# S² en el sector (mismo orden ci[ia, ib] que SECTOR). Se minimiza E(θ) + λ⟨S²⟩(θ): ⟨S²⟩ es otro valor
# esperado, así que el gradiente adjunto sirve igual con H + λS². En el singlete la penalización vale 0
# y no mueve el mínimo; sin ella, en la zona estirada hay mínimos locales contaminados con triplete a
# solo ~0,02 mHa de FCI (la brecha singlete-triplete es de 0,5 mHa a 3,0 Å), que la energía no delata.
S2_SECTOR = np.column_stack([spin_op.contract_ss(e.reshape(6, 6), N_ORB, (N_ALPHA, N_BETA)).ravel()
                             for e in np.eye(len(SECTOR))])
_v = sector_ansatz.state(np.random.default_rng(2).normal(size=sector_ansatz.n))
assert np.allclose(S2_SECTOR, S2_SECTOR.T)
assert np.isclose(_v @ S2_SECTOR @ _v, spin_op.spin_square0(_v.reshape(6, 6), N_ORB, (N_ALPHA, N_BETA))[0])
LAMBDA_S2 = 0.05  # Ha: ⟨S²⟩ = 0,03 (1,5 % de triplete) pasa a costar 1,5 mHa
COST = {d: H + LAMBDA_S2 * S2_SECTOR for d, (H, _) in H_SECTOR.items()}

# El gradiente adjunto de la sección 3 coincide con diferencias finitas centradas.
_x, _C = np.random.default_rng(1).normal(size=sector_ansatz.n), COST[STRETCHED]
_fd = [(sector_ansatz.energy_grad(_x + h, _C)[0] - sector_ansatz.energy_grad(_x - h, _C)[0]) / 2e-6
       for h in 1e-6 * np.eye(sector_ansatz.n)]
assert np.allclose(sector_ansatz.energy_grad(_x, _C)[1], _fd, atol=1e-7), "gradiente adjunto ≠ diferencias finitas"


def optimize(d: float, x0: np.ndarray):
    """BFGS con gradiente adjunto exacto sobre E + λ⟨S²⟩. |∇| < 1e-6 deja el costo a ~1e-12 Ha del mínimo local."""
    return minimize(sector_ansatz.energy_grad, x0, args=(COST[d],), jac=True, method="BFGS", options={"gtol": 1e-6})


def diagnostics(d: float, x: np.ndarray) -> dict:
    """Energía statevector (sin penalización), error vs. FCI y ⟨S²⟩ (0 = singlete, 2 = triplete)."""
    v = sector_ansatz.state(x)
    H, e_fci = H_SECTOR[d]
    return {"e_sv": v @ H @ v, "err_mHa": (v @ H @ v - e_fci) * 1e3, "S2": v @ S2_SECTOR @ v}


# 1) Warm start en las dos direcciones + multi-start: en cada d se arranca desde θ* de la distancia vecina,
#    desde N_PERTURB perturbaciones de ese θ* y desde N_RANDOM puntos al azar; se queda el mejor.
N_PERTURB, N_RANDOM = 2, 3
rng = np.random.default_rng(5)
best, starts_log = {}, []  # best: d -> (resultado de minimize, origen)
for direction, order in (("0,6→3,0", D), ("3,0→0,6", D[::-1])):
    x = np.zeros(sector_ansatz.n)  # θ = 0 es HF: el primer punto se salva con los arranques al azar
    for d in map(float, order):
        starts = ([("warm", x)] + [("warm+ruido", x + rng.normal(scale=0.3, size=x.size)) for _ in range(N_PERTURB)]
                  + [("azar", rng.normal(scale=0.7, size=x.size)) for _ in range(N_RANDOM)])
        runs = []
        for kind, x0 in starts:
            r = optimize(d, x0)
            runs.append((r, f"{direction} {kind}"))
            starts_log.append({"d": d, "direction": direction, "start": kind, **diagnostics(d, r.x),
                               "nit": r.nit, "converged": r.success})
        r, origin = min(runs, key=lambda t: t[0].fun)
        x = r.x
        if d not in best or r.fun < best[d][0].fun:
            best[d] = (r, origin)
starts_log = pd.DataFrame(starts_log)

# 2) Pulido: cada d se reoptimiza desde el θ* final de sus dos vecinos, hasta que nada mejore.
#    Corrige los puntos donde ninguna de las dos pasadas encontró el mismo mínimo que sus vecinos.
#    Cada ronda recorre la grilla en el sentido opuesto a la anterior, así una mejora se propaga entera.
grid = list(map(float, D))
for n_polish in range(1, 21):
    improved = 0
    for i in (range(len(grid)) if n_polish % 2 else reversed(range(len(grid)))):
        for j in (i - 1, i + 1):
            if 0 <= j < len(grid):
                r = optimize(grid[i], best[grid[j]][0].x)
                if r.fun < best[grid[i]][0].fun - 1e-9:
                    best[grid[i]], improved = (r, f"pulido desde {grid[j]}"), improved + 1
    if not improved:
        break
assert not improved, "el pulido no terminó en 20 rondas"

params = {d: best[d][0].x for d in grid}  # contrato 0.1: d -> θ* (np.ndarray, en el orden de build_ansatz)
opt = pd.DataFrame([{"d": d, **diagnostics(d, r.x), "e_fci": H_SECTOR[d][1], "origin": origin, "nit": r.nit,
                     "nfev": r.nfev, "converged": r.success, "grad_norm": np.linalg.norm(r.jac)}
                    for d, (r, origin) in best.items()]).sort_values("d", ignore_index=True)
t_opt = time.perf_counter() - _t0

# 3) Paisaje en la zona estirada (para #8): resultado desde 10 arranques distintos (HF, warm desde la
#    distancia vecina ya optimizada y 8 al azar), cada uno por separado.
MULTISTART_D = (1.8, 2.4, 3.0)
ms_rows = []
for d in MULTISTART_D:
    neighbor = grid[grid.index(d) - 1]
    starts = ([("HF (θ=0)", np.zeros(sector_ansatz.n)), (f"warm desde {neighbor}", params[neighbor])]
              + [(f"azar {i}", rng.normal(scale=0.7, size=sector_ansatz.n)) for i in range(8)])
    for kind, x0 in starts:
        r = optimize(d, x0)
        ms_rows.append({"d": d, "start": kind, **diagnostics(d, r.x), "nit": r.nit, "converged": r.success})
multistart = pd.DataFrame(ms_rows)


def vqe_state(d: float) -> Statevector:
    """Statevector de 8 qubits del circuito real en θ*(d) (para ⟨S²⟩ y la varianza en #8)."""
    return Statevector(ansatz.assign_parameters(params[float(d)]))


# El circuito real con el Hamiltoniano completo de 8 qubits da la misma energía que la simulación en el sector.
for d in (0.9, STRETCHED, 2.4, DISSOCIATED):
    op, e_nuc = qubit_hamiltonian(d)
    e_full = vqe_state(d).expectation_value(op).real + e_nuc
    assert abs(e_full - opt.set_index("d").e_sv[d]) < 1e-9, (d, e_full)

stretched = opt[opt.d >= STRETCHED]
assert len(params) == len(D) and opt.converged.all()
assert (opt.S2 < 1e-3).all(), opt  # singlete en todas las distancias (#8)
assert (stretched.err_mHa.abs() < CHEM_ACC * 1e3).all(), stretched
jump = np.abs(np.diff(opt.err_mHa)).max()  # suavidad: un salto del error entre vecinos delata un mínimo local
print(f"{time.perf_counter() - _t0:.0f} s ({t_opt:.0f} s optimización, {n_polish} rondas de pulido). "
      f"Error máx. vs. FCI: {opt.err_mHa.abs().max():.4f} mHa (zona estirada: {stretched.err_mHa.abs().max():.4f} mHa); "
      f"salto máx. del error entre vecinos: {jump:.4f} mHa; ⟨S²⟩ máx.: {opt.S2.max():.1e}")
print("Multi-start en la zona estirada (10 arranques por distancia, sin elegir):")
print(multistart.groupby("d").agg(err_min=("err_mHa", "min"), err_mediana=("err_mHa", "median"),
                                  err_max=("err_mHa", "max"), S2_max=("S2", "max")).to_string(float_format="%.3g"))
opt

### Decisiones (#5)

**Qué deja esta celda.** `params = {d: θ*}` (contrato 0.1, un `np.ndarray` de 19 ángulos por distancia, en el orden de `build_ansatz`), la tabla `opt` (energía statevector, error vs. FCI, ⟨S²⟩, iteraciones, convergencia y de qué arranque salió cada θ*), `starts_log` (todos los arranques del barrido), `multistart` (10 arranques sin elegir a 1,8 / 2,4 / 3,0 Å) y `vqe_state(d)`, que regenera el statevector de 8 qubits del circuito real en θ*(d) para ⟨S²⟩ y la varianza (#8). Las energías de `opt` son **solo diagnóstico**: la tabla `vqe` del contrato (energía, σ, shots, `cz`) la arma #6 midiendo con `qiskit_aer`, nunca desde el statevector.

**Resultado** (una corrida, semilla fija):

| | Valor |
|---|---|
| Error máximo vs. FCI, toda la grilla | 0,003 mHa (exacto a < 1e-7 mHa hasta 2,2 Å) |
| Error máximo vs. FCI, $d \geq 1{,}8$ Å | 0,003 mHa (criterio: 1,6 mHa) |
| Salto máximo del error entre distancias vecinas | 0,002 mHa (curva suave) |
| ⟨S²⟩ máximo | 2e-5 (singlete) |
| Convergencia | 19/19 (BFGS, $\lvert\nabla\rvert < 10^{-6}$), 45–140 iteraciones por punto |
| Tiempo de la celda | 60–75 s local (~2,5 min el notebook entero); sin caché |

Verificado: el circuito real (`ansatz` con θ*) y el Hamiltoniano de 8 qubits de #2 dan la misma energía que la simulación en el sector (< 1e-9 Ha, a 0,9 / 1,8 / 2,4 / 3,0 Å).

**Simulador.** La búsqueda usa `sector_ansatz` de la sección 3: el ansatz conserva N y S_z, así que simular en los 36 estados del sector es exacto y mucho más rápido que con 256 amplitudes.

**Optimizador: BFGS con gradiente adjunto exacto.** El gradiente completo sale de una pasada hacia adelante y otra hacia atrás sobre el estado (verificado contra diferencias finitas). La regla de *parameter-shift* de 2 términos no sirve acá: el generador de cada excitación tiene autovalores {−1, 0, 1} (frecuencias 1 y 2: 4 evaluaciones con la regla generalizada) y las simples comparten ángulo en α y β (autovalores −2…2: 8 evaluaciones), o sea 9·4 + 10·8 = 116 circuitos por gradiente. Además cada θ aparece en varias compuertas (8 `ry` por doble). Con statevector, el gradiente adjunto es exacto y cuesta del orden de una evaluación de la energía.

**Warm start en las dos direcciones + multi-start + pulido.**

1. Se recorre la grilla 0,6 → 3,0 y 3,0 → 0,6. En cada $d$ se arranca desde θ* de la distancia anterior (*warm*), desde 2 perturbaciones de ese θ* (ruido gaussiano σ = 0,3) y desde 3 puntos al azar (σ = 0,7), y se queda el de menor costo. Entre las dos pasadas se guarda el mejor por punto.
2. Pulido: cada $d$ se reoptimiza desde el θ* final de sus dos vecinos, alternando el sentido del recorrido, hasta que ningún punto mejore (> 1e-9 Ha; 3 rondas).

Cada pieza aporta: el mejor por punto de una sola dirección queda a 0,06 mHa (ida) o 0,11 mHa (vuelta); combinando las dos y puliendo, a 0,003 mHa. Por tipo de arranque, en todo el barrido: desde *warm* el 95 % de las corridas llega a precisión química y el 47 % a < 0,01 mHa; desde el azar, el 75 % y el 10 %.

**Penalización de espín: se minimiza $E(\theta) + \lambda \langle S^2\rangle(\theta)$, con λ = 0,05 Ha.** ⟨S²⟩ es otro valor esperado, así que el gradiente adjunto se aplica a $H + \lambda S^2$ sin cambios. En el singlete exacto la penalización vale 0 y el mínimo no se mueve; `opt.e_sv` es la energía sin penalizar. Sin ella, con otra semilla el barrido terminó con ⟨S²⟩ = 3e-2 a 2,8 Å (≈ 1,5 % de triplete) y un error de solo 0,02 mHa: el triplete está a 0,5 mHa a 3,0 Å, así que la energía no lo delata (ver la limitación de la sección 3). Con la penalización, en tres semillas ⟨S²⟩ ≤ 2e-4 y el error ≤ 0,03 mHa.

**Multi-start en la zona estirada** (`multistart`, error vs. FCI en mHa, 10 arranques por distancia):

| $d$ (Å) | HF (θ = 0) | *warm* desde el vecino | mejor de 8 al azar | mediana de 8 al azar | peor de 8 al azar |
|---|---|---|---|---|---|
| 1,8 | 153 | < 1e-8 | 0,006 | 0,3 | 63 |
| 2,4 | 230 | 0,007 | 0,02 | 2,7 | 10,5 |
| 3,0 | 287 | 0,002 | 0,0002 | 0,12 | 1,3 |

**Qué no funcionó.**

- **Arrancar desde HF (θ = 0)**: cae en un punto estacionario a 10 mHa (0,6 Å) y hasta 287 mHa (3,0 Å). Por eso el primer punto de cada pasada depende de los arranques al azar.
- **Un solo arranque**: uno al azar queda fuera de precisión química en ~25 % de las corridas (hasta decenas de mHa) e incluso uno *warm* en ~5 %, como ya había visto #4.
- **L-BFGS-B**: llega a los mismos mínimos, pero tarda 2–4× más (con `ftol` relativo de 1e-13 necesita 300–900 iteraciones). **Ajustar la tolerancia** (`gtol` de 1e-6 a 1e-10, o L-BFGS-B con `ftol = 1e-15`) desde θ* no baja el error ni un µHa: los errores residuales de ~1e-3 mHa son mínimos locales reales, no falta de convergencia.
- **Reescribir el gradiente con productos matriz-vector** (en vez de armar cada $e^{\theta K}$): solo 15 % más rápido, porque domina el costo de Python; se reutiliza el de la sección 3.

**Limitaciones.** El paisaje tiene mínimos locales hasta el nivel del µHa, así que los últimos dígitos de `params` dependen de la semilla (el error se mantiene ≤ 0,04 mHa en todas las semillas probadas, con y sin penalización). Diferencias numéricas mínimas (del orden de 1e-13 Ha) cambian a qué mínimo cae un arranque al azar: entre dos corridas del mismo código, el mejor de los 8 al azar a 3,0 Å pasó de 0,0002 a 0,002 mHa. Elegir por $E + \lambda\langle S^2\rangle$ evita que uno de esos empates elija un mínimo contaminado (queda λ⟨S²⟩ por encima); entre mínimos singlete solo cambian los µHa. Por lo mismo, θ* cambia entre corridas aunque la energía casi no: para comparar corridas usar energías y ⟨S²⟩, no los ángulos.

### 4.2 Medición de la energía con `SamplerV2`: energía, σ, shots y `cz` (#6)

Con los parámetros óptimos $\theta^*(d)$ fijos (de la optimización de arriba, #5), acá producimos la **energía reportada** de
cada punto de la grilla. La regla es dura: el número que va a la tabla sale de **medir circuitos** con
`qiskit_aer.primitives.SamplerV2` con una cantidad explícita de shots, nunca del statevector.

Receta, en una línea: partimos $H = c_I + \sum_k c_k P_k$ en grupos de Paulis que conmutan qubit a qubit (QWC), medimos cada
grupo en su base con `SamplerV2`, calculamos el valor **por shot** del grupo entero y combinamos
$E = e_\text{nuc} + c_I + \sum_g \bar h_g$, $\;\sigma^2 = \sum_g \operatorname{Var}_g / N_g$. Los shots se reparten entre grupos
como $N_g \propto \sqrt{\operatorname{Var}_g}$ (reparto óptimo) usando una corrida piloto, y el total por punto se dimensiona
a partir del margen de CCSD.

Salida: la tabla del contrato `vqe` con columnas `d, energy, sigma, shots, cz`, y la función reutilizable `measure_energy(...)`.

In [ ]:
# --- 4.2.0 Perillas de la medición ---------------------------------------------------------------
import os
import time
import itertools

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit, transpile
from qiskit.circuit import Parameter, ParameterVector
from qiskit.circuit.library import PauliEvolutionGate
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit_aer import AerSimulator
from qiskit_aer.primitives import SamplerV2, EstimatorV2

# SHOTS_SCALE multiplica TODOS los presupuestos de shots de esta sección (tabla y validaciones).
# 1.0 = corrida completa (lo reportado). Para una prueba rápida: SHOTS_SCALE = 0.02 (σ crece como 1/sqrt(escala)).
# También se puede fijar desde afuera con la variable de entorno QJ_SHOTS_SCALE.
SHOTS_SCALE = float(os.environ.get("QJ_SHOTS_SCALE", 1.0))

SEED = 2026                    # semilla base: todo lo de esta sección es reproducible
SIGMA_DEFAULT = 3.0e-3         # Ha: σ objetivo "sano" en toda la curva
SIGMA_MARGIN_FRACTION = 0.2    # en d >= STRETCHED: σ objetivo <= 1/5 del margen |E_CCSD - E_FCI|
PILOT_SHOTS_PER_GROUP = 500    # corrida piloto (solo para estimar Var_g y repartir; no entra en E ni σ)
MIN_GROUP_SHOTS = 100          # piso de shots por grupo en la corrida principal
MAX_SHOTS_PER_POINT = 2_000_000  # tope de seguridad por punto (si se alcanza, se avisa)

_T0_MEDICION = time.time()
print(f"SHOTS_SCALE = {SHOTS_SCALE}  (1.0 = corrida reportada)")

#### Piezas que todavía no están en `main`: stand-ins que se apagan solos

Esta sección necesita `references(d)` (#1), `qubit_hamiltonian(d)` (#2), `build_ansatz()` (#4) y la tabla `params` (#5).
Para no esperar, cada pieza se pide primero a la función real del contrato; si levanta `NotImplementedError` (o la
tabla `params` no existe), se usa un **stand-in** definido acá (`_standin_*`) y se imprime un aviso de una línea.
Cuando los compañeros suban su código, se usa la pieza real automáticamente, sin tocar esta sección.

- **Referencias (stand-in de #1):** RHF, CCSD y FCI con PySCF. Solo se usan para dimensionar shots (margen de CCSD) y para
  validar el Hamiltoniano; nunca como resultado.
- **Hamiltoniano (stand-in de #2):** Jordan-Wigner escrito a mano en la convención del contrato (adaptado de
  `scripts/h4_reference.py` de la rama de análisis), validado contra FCI diagonalizando en el sector $N=4$, $S_z=0$.
- **Ansatz y parámetros (stand-in de #4 y #5):** UCCSD de un paso (26 excitaciones de espín-orbital, conserva $N_\alpha$ y
  $N_\beta$) que arranca en `HF_BITSTRING`, con $\theta$ ajustado por statevector (permitido para *buscar* parámetros).
  **No llega a FCI** en la zona estirada (UCCSD de un paso se traba a unos mHa); alcanza para construir y validar la medición.

In [ ]:
# --- 4.2.1 Stand-ins de #1 y #2 (se apagan solos cuando existan references / qubit_hamiltonian) ---
from pyscf import ao2mo, cc, fci, gto, scf
from pyscf import lib as pyscf_lib

# Reproducibilidad bit a bit: PySCF reparte sumas entre hilos de OpenMP y el orden de suma cambia de corrida en corrida
# (diferencias de ~1e-14 que el ajuste de θ amplifica a ~1e-9 Ha y que terminan moviendo el reparto de shots). Con 1 hilo
# el resultado es idéntico. Para H4 en STO-3G los cálculos son diminutos: un hilo no es más lento (los hilos solo agregan costo).
pyscf_lib.num_threads(1)

STANDIN_USED = {}  # qué stand-ins se usaron en esta corrida (queda en el resumen final)


def _notice_standin(key, msg):
    if key not in STANDIN_USED:
        STANDIN_USED[key] = msg
        print(f"[stand-in] {msg}")


_STANDIN_SCF_CACHE = {}


def _standin_scf(d):
    """RHF de PySCF para H4 a distancia d (cacheado: lo comparten los stand-ins de #1 y #2)."""
    d = float(round(d, 4))
    if d not in _STANDIN_SCF_CACHE:
        mol = gto.M(atom=h4_geometry(d), basis=BASIS, unit="Angstrom", verbose=0)
        mf = scf.RHF(mol)
        mf.conv_tol = 1e-10
        mf.run()
        _STANDIN_SCF_CACHE[d] = (mol, mf)
    return _STANDIN_SCF_CACHE[d]


def _standin_references(d):
    """STAND-IN de #1: {'d','e_hf','e_ccsd','e_fci'} con PySCF (solo para comparar y dimensionar shots)."""
    mol, mf = _standin_scf(d)
    mycc = cc.CCSD(mf)
    mycc.max_cycle = 500
    mycc.run()
    e_fci = fci.FCI(mf).kernel()[0]
    return {"d": float(d), "e_hf": float(mf.e_tot), "e_ccsd": float(mycc.e_tot), "e_fci": float(e_fci)}


# Jordan-Wigner en la convención del contrato: qubit p = (p, α), qubit 4+p = (p, β); labels de Qiskit se leen q7...q0.
def _standin_annihilation(j, n=N_QUBITS):
    def label(p):
        s = ["I"] * n
        for k in range(j):
            s[k] = "Z"
        s[j] = p
        return "".join(reversed(s))
    return SparsePauliOp([label("X"), label("Y")], [0.5, 0.5j])


_STANDIN_A = [_standin_annihilation(j) for j in range(N_QUBITS)]
_STANDIN_AD = [a.adjoint() for a in _STANDIN_A]
_STANDIN_OPCACHE = {}  # productos de operadores de JW: no dependen de d, se calculan una vez


def _standin_fop(*idx):
    """a†_{i0} a†_{i1} ... a_{..}: idx = (('+', p), ('-', q), ...) -> SparsePauliOp (cacheado)."""
    if idx not in _STANDIN_OPCACHE:
        op = None
        for kind, p in idx:
            f = _STANDIN_AD[p] if kind == "+" else _STANDIN_A[p]
            op = f if op is None else op @ f
        _STANDIN_OPCACHE[idx] = op.simplify(atol=1e-12)
    return _STANDIN_OPCACHE[idx]


def _standin_qubit_hamiltonian(d):
    """STAND-IN de #2: (H electrónico JW de 8 qubits en la base MO de RHF, e_nuc)."""
    mol, mf = _standin_scf(d)
    C = mf.mo_coeff
    h1 = C.T @ mf.get_hcore() @ C
    eri = ao2mo.restore(1, ao2mo.kernel(mol, C), C.shape[1])  # (pq|rs), notación de químicos
    so = lambda p, s: p + N_ORB * s
    acc = []
    for p, q in itertools.product(range(N_ORB), repeat=2):
        if abs(h1[p, q]) > 1e-12:
            for s in (0, 1):
                acc.append(h1[p, q] * _standin_fop(("+", so(p, s)), ("-", so(q, s))))
    # 1/2 Σ (pq|rs) a†_{pσ} a†_{rτ} a_{sτ} a_{qσ}
    for p, q, r, s_ in itertools.product(range(N_ORB), repeat=4):
        v = eri[p, q, r, s_]
        if abs(v) < 1e-12:
            continue
        for si, ti in itertools.product((0, 1), repeat=2):
            P, Q, R, S = so(p, si), so(q, si), so(r, ti), so(s_, ti)
            if P != R and Q != S:
                acc.append(0.5 * v * _standin_fop(("+", P), ("+", R), ("-", S), ("-", Q)))
    op = SparsePauliOp.sum(acc).simplify(atol=1e-10)
    # PySCF usa hilos y los coeficientes pueden cambiar en ~1e-14 entre corridas; redondear a 1e-12 Ha deja el
    # Hamiltoniano idéntico bit a bit y hace reproducible todo lo que sigue (ajuste de θ, reparto de shots).
    op = SparsePauliOp(op.paulis, np.round(op.coeffs.real, 12))
    return op, float(mol.energy_nuc())


def get_references(d):
    """references(d) del contrato (#1); si todavía no existe, el stand-in de PySCF."""
    try:
        return references(d)
    except NotImplementedError:
        _notice_standin("references", "references(d) de #1 no está: uso _standin_references (PySCF RHF/CCSD/FCI).")
        return _standin_references(d)


def get_hamiltonian(d):
    """qubit_hamiltonian(d) del contrato (#2); si todavía no existe, el stand-in de Jordan-Wigner."""
    try:
        return qubit_hamiltonian(d)
    except NotImplementedError:
        _notice_standin("qubit_hamiltonian", "qubit_hamiltonian(d) de #2 no está: uso _standin_qubit_hamiltonian (JW propio).")
        return _standin_qubit_hamiltonian(d)


# Índices del sector N=4, Sz=0 (2 α en q0-q3, 2 β en q4-q7): 36 determinantes.
_SECTOR = [b for b in range(2 ** N_QUBITS)
           if bin(b).count("1") == N_ALPHA + N_BETA and bin(b & 0b1111).count("1") == N_ALPHA]

_t = time.time()
REF, HAM = {}, {}
for d in D:
    REF[d] = get_references(d)
    HAM[d] = get_hamiltonian(d)
    # Control (barato) de que el Hamiltoniano usado reproduce FCI en el sector físico: vale para el real y para el stand-in.
    op, e_nuc = HAM[d]
    e0 = np.linalg.eigvalsh(op.to_matrix()[np.ix_(_SECTOR, _SECTOR)].real)[0] + e_nuc
    assert abs(e0 - REF[d]["e_fci"]) < 1e-6, f"el Hamiltoniano no reproduce FCI en d={d}: {e0} vs {REF[d]['e_fci']}"

_n_terms = {len(HAM[d][0]) for d in D}
print(f"Hamiltonianos y referencias para {len(D)} distancias en {time.time() - _t:.1f} s; "
      f"términos de Pauli por punto: {sorted(_n_terms)}; FCI reproducida en el sector N=4, Sz=0 (|ΔE| < 1e-6 Ha).")

In [ ]:
# --- 4.2.2 Ansatz y θ*: build_ansatz() de #4 y params de #5, o stand-in ---------------------------
# ===================== BLOQUE STAND-IN (#4 y #5) =====================================================
# UCCSD de un paso sobre HF_BITSTRING. Cada excitación es exp(θ_k G_k) con G_k = T_k - T_k† (antihermítico, real).
# Como G_k^3 = -G_k, exp(θ G) = I + sin θ G + (1 - cos θ) G^2: eso permite simular rapidísimo en numpy y sacar el
# gradiente exacto en modo reverso para ajustar θ por statevector (permitido: es BUSCAR parámetros, no reportar).
_OCC_A, _VIR_A = [alpha_qubit(p) for p in (0, 1)], [alpha_qubit(p) for p in (2, 3)]
_OCC_B, _VIR_B = [beta_qubit(p) for p in (0, 1)], [beta_qubit(p) for p in (2, 3)]


def _standin_excitations():
    singles = [((i,), (a,)) for occ, vir in ((_OCC_A, _VIR_A), (_OCC_B, _VIR_B)) for i in occ for a in vir]
    doubles = [(tuple(_OCC_A), tuple(_VIR_A)), (tuple(_OCC_B), tuple(_VIR_B))]
    doubles += [((i, j), (a, b)) for i in _OCC_A for j in _OCC_B for a in _VIR_A for b in _VIR_B]
    return doubles + singles  # 18 dobles + 8 simples = 26


def _standin_generators():
    """Lista de H_k = i (T_k - T_k†) (hermíticos) como SparsePauliOp: exp(-i θ H_k) = exp(θ G_k)."""
    gens = []
    for occ, vir in _standin_excitations():
        T = _standin_fop(*([("+", a) for a in vir] + [("-", i) for i in reversed(occ)]))
        Hk = (1j * (T - T.adjoint())).simplify(atol=1e-12)
        assert np.allclose(Hk.coeffs.imag, 0)
        gens.append(SparsePauliOp(Hk.paulis, Hk.coeffs.real))
    return gens


_STANDIN_GENS = _standin_generators()


def _standin_build_ansatz():
    """STAND-IN de #4: UCCSD de un paso (26 parámetros), conserva N_α y N_β, arranca en HF_BITSTRING."""
    theta = ParameterVector("θ", len(_STANDIN_GENS))
    qc = QuantumCircuit(N_QUBITS, name="uccsd_standin")
    for q, bit in enumerate(reversed(HF_BITSTRING)):
        if bit == "1":
            qc.x(q)
    for k, Hk in enumerate(_STANDIN_GENS):
        qc.append(PauliEvolutionGate(Hk, time=theta[k]), range(N_QUBITS))
    return qc


def _standin_fit_uccsd(ops, e_nucs):
    """STAND-IN de #5: θ* por punto con statevector numpy + gradiente analítico (L-BFGS-B), arranque en caliente."""
    from scipy.optimize import minimize
    # Todo conserva N_α y N_β: alcanza con simular en el sector de 36 determinantes (matrices 36x36).
    sec = np.array(_SECTOR)
    G = [Hk.to_matrix().imag[np.ix_(sec, sec)] for Hk in _STANDIN_GENS]  # H_k = i G_k, G_k real antisim. -> G_k = Im(H_k)
    G2 = [g @ g for g in G]
    psi0 = (sec == int(HF_BITSTRING, 2)).astype(float)

    def energy_grad(th, Hm):
        states = [psi0]
        for k, t in enumerate(th):
            s = states[-1]
            states.append(s + np.sin(t) * (G[k] @ s) + (1 - np.cos(t)) * (G2[k] @ s))
        psi = states[-1]
        lam = Hm @ psi
        e = psi @ lam
        grad = np.zeros(len(th))
        for k in range(len(th) - 1, -1, -1):
            s_prev = states[k]
            dU = np.cos(th[k]) * (G[k] @ s_prev) + np.sin(th[k]) * (G2[k] @ s_prev)
            grad[k] = 2 * lam @ dU
            # lam <- U_k^T lam = U_k(-θ) lam
            lam = lam - np.sin(th[k]) * (G[k] @ lam) + (1 - np.cos(th[k])) * (G2[k] @ lam)
        return e, grad

    out, e_fit, prev = {}, {}, np.zeros(len(G))
    for d in D:
        Hm = ops[d].to_matrix().real[np.ix_(sec, sec)]
        best = None
        for x0 in (prev, np.zeros(len(G))):
            r = minimize(energy_grad, x0, args=(Hm,), jac=True, method="L-BFGS-B", options={"maxiter": 2000, "gtol": 1e-9})
            if best is None or r.fun < best.fun:
                best = r
        # θ* redondeado a 1e-6 rad: el ajuste en caliente arrastra diferencias de ~1e-12 entre máquinas/corridas, que
        # pueden cambiar el redondeo del reparto de shots y romper la reproducibilidad bit a bit (en un mínimo, el cambio
        # de energía por redondear es ~1e-12 Ha).
        theta = np.round(best.x, 6)
        out[d], e_fit[d] = theta, float(energy_grad(theta, Hm)[0]) + e_nucs[d]
        prev = best.x
    return out, e_fit
# =================== FIN BLOQUE STAND-IN (#4 y #5) ===================================================


def _fit_generic_statevector(qc, ops):
    """STAND-IN de #5 para un ansatz real sin params todavía: θ* por statevector (genérico, más lento)."""
    from scipy.optimize import minimize
    out, prev = {}, np.zeros(qc.num_parameters)
    for d in D:
        Hm = ops[d].to_matrix()

        def f(x):
            sv = Statevector(qc.assign_parameters(x)).data
            return float(np.real(np.vdot(sv, Hm @ sv)))

        r = minimize(f, prev, method="L-BFGS-B", options={"maxiter": 500})
        out[d] = prev = r.x
    return out


try:
    ansatz = build_ansatz()
except NotImplementedError:
    _notice_standin("build_ansatz", "build_ansatz() de #4 no está: uso _standin_build_ansatz (UCCSD de un paso, 26 parámetros).")
    ansatz = _standin_build_ansatz()

_params_ok = ("params" in globals() and isinstance(globals()["params"], dict)
              and all(d in globals()["params"] and len(globals()["params"][d]) == ansatz.num_parameters for d in D))
_t = time.time()
if _params_ok:
    THETA = {d: np.asarray(globals()["params"][d], dtype=float) for d in D}
elif "build_ansatz" in STANDIN_USED:
    _notice_standin("params", "params de #5 no está: θ* ajustado por statevector dentro del bloque stand-in (no llega a FCI).")
    THETA, _E_FIT = _standin_fit_uccsd({d: HAM[d][0] for d in D}, {d: HAM[d][1] for d in D})
else:
    _notice_standin("params", "params de #5 no está: θ* ajustado por statevector (genérico) sobre el ansatz de #4.")
    THETA = _fit_generic_statevector(ansatz, {d: HAM[d][0] for d in D})

# Circuito que se mide en cada punto: ansatz con θ* asignado, transpilado UNA vez a la base de Aer (más liviano que
# transpile_line, que se usa solo para contar cz). Su energía statevector (lo que #5 reporta como óptimo) sirve solo
# para VALIDAR la medición: nunca va a la tabla.
_AER = AerSimulator()
CIRCUIT, E_SV = {}, {}
for d in D:
    op, e_nuc = HAM[d]
    CIRCUIT[d] = transpile(ansatz.assign_parameters(THETA[d]), _AER, optimization_level=1, seed_transpiler=0)
    E_SV[d] = float(Statevector(CIRCUIT[d]).expectation_value(op).real) + e_nuc
if "_E_FIT" in globals() and "params" in STANDIN_USED and "build_ansatz" in STANDIN_USED:
    # el simulador numpy del ajuste y el circuito de Qiskit tienen que ser el mismo estado
    assert max(abs(_E_FIT[d] - E_SV[d]) for d in D) < 1e-8, "el circuito stand-in no reproduce el ajuste numpy"

CZ_ANSATZ = cz_count(ansatz)  # contrato: transpile_line (cz, sx, x, rz; acoplamiento lineal) sobre el ansatz
print(f"Ansatz: {ansatz.num_parameters} parámetros, {CZ_ANSATZ} cz (transpile_line). θ* listo en {time.time() - _t:.1f} s.")
print(pd.DataFrame({"d": D,
                    "E_sv - E_FCI (mHa)": [(E_SV[d] - REF[d]["e_fci"]) * 1e3 for d in D],
                    "E_CCSD - E_FCI (mHa)": [(REF[d]["e_ccsd"] - REF[d]["e_fci"]) * 1e3 for d in D]}).round(3).to_string(index=False))

#### Cómo medimos: grupos QWC, valor por shot del grupo y reparto óptimo

1. **Grupos.** Los términos de Pauli no identidad se ordenan por $|c_k|$ decreciente y se insertan en el primer grupo con
   el que conmutan qubit a qubit (*sorted insertion*). Un grupo QWC tiene una base de medición por qubit (X, Y o Z): se rota
   con `H` (X) o `S†H` (Y) y se mide todo en Z con un solo circuito.
2. **Valor por shot del grupo.** Cada bitstring $x$ da un número $h_g(x) = \sum_{k\in g} c_k (-1)^{\text{paridad}(x\,\&\,\text{soporte}_k)}$.
   Promediamos $h_g$ sobre los shots y tomamos su varianza muestral $\operatorname{Var}_g$. Como es la varianza del grupo
   **entero**, ya incluye las covarianzas entre los términos que se miden juntos (sumarlas término a término las ignoraría).
3. **Error.** Los grupos se miden con shots independientes, así que $\sigma^2 = \sum_g \operatorname{Var}_g / N_g$.
4. **Reparto.** A total $N$ fijo, $\sigma^2$ es mínima con $N_g \propto \sqrt{\operatorname{Var}_g}$, y entonces
   $\sigma = K/\sqrt{N}$ con $K = \sum_g \sqrt{\operatorname{Var}_g}$. Las $\operatorname{Var}_g$ salen de una **corrida piloto**
   corta (500 shots por grupo) que solo decide el reparto: la energía y la σ salen de la corrida principal, con shots
   nuevos, para que el reparto no sesgue la estimación. Los shots del piloto **sí** se cuentan en `shots` (costo real).
5. **Cómo se corre.** Un solo circuito por punto: ansatz con $\theta^*$ + una rotación $U(\theta_q, 0, \lambda_q)$ parametrizada en cada
   qubit + medición. Cada grupo es una *asignación* de esas rotaciones dentro de **un** PUB de `SamplerV2`, repetida
   $m_g$ veces con el mismo número de shots, así $N_g = m_g \cdot n_\text{paquete}$. Razón principal: Aer **corre en jobs
   separados los PUBs con distinto número de shots, todos con la misma semilla**, y los grupos quedaban correlacionados y la σ
   subestimada (lo vimos: dispersión entre semillas ≈ 1,4 σ). Dentro de un PUB cada asignación usa su propia corriente
   aleatoria. Con esto mismo, las **repeticiones** de la validación son bindings extra del mismo PUB.
6. **Costo.** Lo que cuesta en Aer es el *binding* (~2 ms: reescribe y fusiona las ~1000 compuertas del ansatz en cada uno), no
   los shots ni el job. Por eso usamos `method="statevector"` explícito (con `automatic` es 3 veces más lento), bindings en
   paralelo y ~2 paquetes por grupo. Aer se queda sin memoria con miles de bindings en un job (`bad allocation`), así que se
   parten en jobs de a lo sumo 600.

In [ ]:
# --- 4.2.3 measure_energy: la función reutilizable ------------------------------------------------
_PAULI_CODE = {(False, False): 0, (True, False): 1, (True, True): 2, (False, True): 3}  # (x, z) -> I, X, Y, Z
_GROUP_CACHE = {}


def qwc_groups(op):
    """Agrupa los términos no identidad de `op` en conjuntos QWC (sorted insertion por |c| decreciente).

    Devuelve (c_identidad, grupos) con cada grupo = dict(basis: array (n,) con 0=I,1=X,2=Y,3=Z,
    terms: índices, table: array (2^n,) con el valor h_g(x) para cada bitstring entero x).
    """
    key = (tuple(op.paulis.to_labels()), np.asarray(op.coeffs).tobytes())
    if key in _GROUP_CACHE:
        return _GROUP_CACHE[key]
    coeffs = np.asarray(op.coeffs)
    assert np.allclose(coeffs.imag, 0, atol=1e-10), "el observable tiene que ser hermítico (coeficientes reales)"
    coeffs = coeffs.real
    n = op.num_qubits
    xs, zs = op.paulis.x, op.paulis.z
    codes = np.array([[_PAULI_CODE[(bool(x), bool(z))] for x, z in zip(xr, zr)] for xr, zr in zip(xs, zs)])
    c_id = float(coeffs[(codes == 0).all(axis=1)].sum())
    groups = []
    for k in np.argsort(-np.abs(coeffs), kind="stable"):
        if (codes[k] == 0).all():
            continue
        for g in groups:
            if np.all((codes[k] == 0) | (g["basis"] == 0) | (codes[k] == g["basis"])):
                g["basis"] = np.where(g["basis"] == 0, codes[k], g["basis"])
                g["terms"].append(int(k))
                break
        else:
            groups.append({"basis": codes[k].copy(), "terms": [int(k)]})
    xs_all = np.arange(2 ** n)
    for g in groups:
        table = np.zeros(2 ** n)
        for k in g["terms"]:
            mask = int(sum(1 << q for q in range(n) if codes[k, q] != 0))
            parity = np.array([bin(v).count("1") & 1 for v in (xs_all & mask)])
            table += coeffs[k] * (1 - 2 * parity)
        g["table"] = table
    _GROUP_CACHE[key] = (c_id, groups)
    return c_id, groups


_ROTATION = {0: (0.0, 0.0), 3: (0.0, 0.0), 1: (np.pi / 2, np.pi), 2: (np.pi / 2, np.pi / 2)}  # U(θ,0,λ): I, H, H·S†
UNITS_PER_GROUP = 2  # granularidad del reparto: en promedio cada grupo recibe ~2 "paquetes" de shots (cada paquete = 1 binding)
# Costo de Aer = ~2 ms por binding (reescribe y fusiona las ~1000 compuertas del ansatz en cada uno), casi independiente de los
# shots. Por eso: método statevector explícito (el "automatic" gasta 3x en elegir método por binding) y bindings en paralelo.
# Las semillas de Aer son por binding: el resultado es idéntico bit a bit con cualquier número de hilos (lo verificamos).
MAX_BINDINGS_PER_JOB = 600
_AER_OPTIONS = {"method": "statevector", "max_parallel_experiments": os.cpu_count() or 1}


def _measurement_template(circuit_bound):
    """circuito + U(θ_q, 0, λ_q) en cada qubit + medición de todo en Z.

    Cada grupo QWC es una ASIGNACIÓN de (θ, λ): I/Z -> U(0,0,0), X -> U(π/2,0,π) = H, Y -> U(π/2,0,π/2) = H·S†.
    Así Aer compila el circuito una sola vez por job y cada grupo es un binding del mismo PUB.
    """
    names = set(_AER.target.operation_names)
    base = circuit_bound
    if any(inst.operation.name not in names for inst in base.data):
        base = transpile(base, _AER, optimization_level=1, seed_transpiler=0)
    n = base.num_qubits
    th, la = ParameterVector("_rot_theta", n), ParameterVector("_rot_lambda", n)
    qc = base.copy()
    for q in range(n):
        qc.u(th[q], 0.0, la[q], q)
    qc.measure_all()
    order = {p: i for i, p in enumerate(qc.parameters)}  # orden en que SamplerV2 espera los valores
    return qc, [order[p] for p in th], [order[p] for p in la]


def _sample(template, groups, bindings_per_group, shots_per_binding, seed):
    """UN job de SamplerV2: un PUB con sum(bindings) asignaciones y `shots_per_binding` shots explícitos cada una.

    bindings_per_group : (G,) o (R, G): R estimaciones independientes (repeticiones) dentro del MISMO job.
    Aer usa una corriente aleatoria distinta para cada binding del PUB, así que grupos y repeticiones son independientes.
    (Ojo: PUBs con distinto número de shots van en jobs separados con la MISMA semilla y quedan correlacionados;
    por eso no usamos un PUB por grupo ni uno por repetición con shots distintos.)
    Devuelve (medias, varianzas por shot, shots) con la forma de bindings_per_group.
    """
    qc, i_th, i_la = template
    n = qc.num_qubits
    one_d = np.ndim(bindings_per_group) == 1
    mm = np.atleast_2d(np.asarray(bindings_per_group, dtype=int))  # (R, G)
    R, G = mm.shape
    base = []
    for g in groups:
        v = np.zeros(qc.num_parameters)
        for q, b in enumerate(g["basis"]):
            v[i_th[q]], v[i_la[q]] = _ROTATION[int(b)]
        base.append(v)
    idx = np.repeat(np.arange(R * G), mm.ravel())     # (repetición, grupo) de cada binding
    rows = np.array(base)[idx % G]
    # Aer expande cada binding en un experimento completo en memoria: con miles de bindings en un job se queda sin memoria
    # ("bad allocation"). Por eso los bindings se parten en jobs de a lo sumo MAX_BINDINGS_PER_JOB, cada uno con su semilla.
    ints = []
    for k, lo in enumerate(range(0, len(rows), MAX_BINDINGS_PER_JOB)):
        seed_k = int(np.random.SeedSequence([int(seed), k]).generate_state(1)[0]) % (2 ** 31 - 2 ** 20)
        sampler = SamplerV2(seed=seed_k, options={"backend_options": _AER_OPTIONS})
        res = sampler.run([(qc, rows[lo:lo + MAX_BINDINGS_PER_JOB], int(shots_per_binding))]).result()[0]
        arr = res.data.meas.array.astype(np.int64)  # (bindings, shots, bytes), big-endian por byte
        x = np.zeros(arr.shape[:2], dtype=np.int64)
        for byte in range(arr.shape[2]):
            x = (x << 8) | arr[:, :, byte]
        ints.append(x)
    ints = np.concatenate(ints)
    counts = np.zeros((R * G, 2 ** n))
    for i, o in enumerate(idx):
        counts[o] += np.bincount(ints[i], minlength=2 ** n)
    counts = counts.reshape(R, G, 2 ** n)
    shots = counts.sum(axis=2)
    tables = np.array([g["table"] for g in groups])
    means = (counts * tables).sum(axis=2) / shots
    vars_ = (counts * (tables[None] - means[:, :, None]) ** 2).sum(axis=2) / (shots - 1)
    shots = shots.astype(int)
    return (means[0], vars_[0], shots[0]) if one_d else (means, vars_, shots)


def measure_energy_many(circuit_bound, op_electronic, e_nuc=0.0, total_shots=None, seed=0, *, n_repeats=1, target_sigma=None,
                        pilot_shots_per_group=PILOT_SHOTS_PER_GROUP, min_group_shots=MIN_GROUP_SHOTS,
                        max_shots=MAX_SHOTS_PER_POINT, return_details=False):
    """`n_repeats` estimaciones independientes (piloto + principal cada una) con solo DOS jobs de SamplerV2 en total.

    Todas las repeticiones comparten circuito, observable y presupuesto: las repeticiones son bindings extra del mismo PUB.
    Con n_repeats > 1 hay que pasar total_shots (con target_sigma cada repetición tendría un n_unit distinto). Devuelve una lista.
    """
    if circuit_bound.num_parameters:
        raise ValueError("measure_energy necesita un circuito con todos los parámetros asignados")
    if (total_shots is None) == (target_sigma is None):
        raise ValueError("pasar exactamente uno de total_shots o target_sigma")
    if n_repeats > 1 and total_shots is None:
        raise ValueError("con n_repeats > 1 hay que pasar total_shots")
    R = int(n_repeats)
    c_id, groups = qwc_groups(op_electronic)
    template = _measurement_template(circuit_bound)
    G = len(groups)
    seed_pilot, seed_main = (int(x) % (2 ** 31 - 2 ** 20) for x in np.random.SeedSequence(int(seed)).generate_state(2))

    # Piloto: mismo número de shots por grupo, SOLO para estimar Var_g (no entra en E ni en σ).
    _, var_pilot, n_pilot = _sample(template, groups, np.ones((R, G), int), max(int(pilot_shots_per_group), 20), seed_pilot)
    sq = np.sqrt(np.maximum(var_pilot, 0.0))                 # (R, G)
    sq[sq.sum(axis=1) == 0] = 1.0

    capped = False
    if total_shots is None:
        total_shots = int(np.ceil((sq[0].sum() / target_sigma) ** 2))
    if max_shots is not None and total_shots > max_shots:
        total_shots, capped = int(max_shots), True
    # Reparto N_g ∝ sqrt(Var_g) en paquetes de n_unit shots (n_unit >= min_group_shots: ningún grupo queda sin medir).
    n_unit = max(int(min_group_shots), int(total_shots) // (UNITS_PER_GROUP * G))
    m = np.maximum(1, np.round(total_shots * sq / sq.sum(axis=1, keepdims=True) / n_unit)).astype(int)

    means, var_main, n_main = _sample(template, groups, m, n_unit, seed_main)
    outs = []
    for r in range(R):
        out = dict(energy=float(e_nuc + c_id + means[r].sum()), sigma=float(np.sqrt(np.sum(var_main[r] / n_main[r]))),
                   shots=int(n_pilot[r].sum() + n_main[r].sum()), shots_pilot=int(n_pilot[r].sum()),
                   shots_main=int(n_main[r].sum()), groups=G, K=float(np.sqrt(var_main[r]).sum()), capped=capped)
        if return_details:
            out["details"] = pd.DataFrame({"basis (q7..q0)": ["".join("IXYZ"[b] for b in g["basis"][::-1]) for g in groups],
                                           "n_terms": [len(g["terms"]) for g in groups], "shots": n_main[r],
                                           "mean": means[r], "var_per_shot": var_main[r], "var_pilot": var_pilot[r]})
        outs.append(out)
    return outs


def measure_energy(circuit_bound, op_electronic, e_nuc=0.0, total_shots=None, seed=0, *, target_sigma=None,
                   pilot_shots_per_group=PILOT_SHOTS_PER_GROUP, min_group_shots=MIN_GROUP_SHOTS,
                   max_shots=MAX_SHOTS_PER_POINT, return_details=False):
    """Estima <H> = e_nuc + <op_electronic> midiendo `circuit_bound` con qiskit_aer SamplerV2 (shots explícitos).

    circuit_bound : circuito SIN parámetros libres y sin mediciones (si hace falta, se transpila para Aer adentro).
    total_shots   : shots de la corrida principal, repartidos N_g ∝ sqrt(Var_g de la piloto). O bien...
    target_sigma  : ...σ objetivo en Ha: la corrida principal usa N = (K_piloto / target_sigma)^2 (con tope max_shots).
    seed          : semilla entera; de ella salen las semillas de SamplerV2 de la piloto y de la principal (reproducible).

    Devuelve dict(energy, sigma, shots, shots_pilot, shots_main, groups, K, capped[, details]).
    `shots` cuenta TODOS los shots ejecutados (piloto + principal); `sigma` es el error estándar de `energy`.
    """
    return measure_energy_many(circuit_bound, op_electronic, e_nuc, total_shots, seed, n_repeats=1, target_sigma=target_sigma,
                               pilot_shots_per_group=pilot_shots_per_group, min_group_shots=min_group_shots,
                               max_shots=max_shots, return_details=return_details)[0]


_c_id, _groups18 = qwc_groups(HAM[STRETCHED][0])
print(f"d = {STRETCHED} Å: {len(HAM[STRETCHED][0])} términos -> c_I + {len(_groups18)} grupos QWC "
      f"(group_commuting de Qiskit da {len(HAM[STRETCHED][0].group_commuting(qubit_wise=True))}).")

#### Cuántos shots por punto

El criterio central pide $|E_\text{VQE} - E_\text{FCI}| + \sigma < |E_\text{CCSD} - E_\text{FCI}|$ en $d \ge 1{,}8$ Å. El margen más
chico es justo 1,8 Å (≈ 9,1 mHa), y desde 2,2 Å supera los 20 mHa. Regla:

$$\sigma_\text{obj}(d) = \begin{cases} \min\big(\tfrac15\,|E_\text{CCSD}-E_\text{FCI}|,\ 3\ \text{mHa}\big) & d \ge 1{,}8\ \text{Å} \\ 3\ \text{mHa} & d < 1{,}8\ \text{Å}\end{cases}$$

Así σ se come como mucho un quinto del margen y deja el resto para el sesgo del ansatz. En la práctica: ≈ 1,8 mHa a 1,8 Å y
3 mHa en el resto. El total por punto sale de la piloto: $N = (K/\sigma_\text{obj})^2$ con $K = \sum_g\sqrt{\operatorname{Var}_g}$,
así que cada punto recibe los shots que **su** estado necesita (con $K \approx 1{,}5$ Ha son ~$7\times10^5$ shots a 1,8 Å y
~$3\times10^5$ en el resto). `SHOTS_SCALE` escala todo para pruebas rápidas.

In [ ]:
# --- 4.2.4 Medición de toda la curva -> tabla del contrato `vqe` ------------------------------------


def sigma_target(d):
    """σ objetivo (Ha) en el punto d, a partir del margen de CCSD (ver la regla arriba)."""
    if d >= STRETCHED - 1e-9:
        margin = abs(REF[d]["e_ccsd"] - REF[d]["e_fci"])
        return min(SIGMA_MARGIN_FRACTION * margin, SIGMA_DEFAULT)
    return SIGMA_DEFAULT


def _scaled(n, floor=1_000):
    """Presupuesto escalado por SHOTS_SCALE (con piso, para que las pruebas rápidas sigan teniendo sentido)."""
    return max(int(round(n * SHOTS_SCALE)), floor)


PILOT_SHOTS = _scaled(PILOT_SHOTS_PER_GROUP, floor=200)  # con escala chica la piloto no baja de 200 shots por grupo


_t = time.time()
_rows, MEAS = [], {}
for i, d in enumerate(D):
    op, e_nuc = HAM[d]
    # Con SHOTS_SCALE < 1 el σ objetivo se relaja en 1/sqrt(escala): los shots bajan en proporción.
    m = measure_energy(CIRCUIT[d], op, e_nuc, target_sigma=sigma_target(d) / np.sqrt(SHOTS_SCALE), seed=SEED + 100 * i,
                       pilot_shots_per_group=PILOT_SHOTS)
    MEAS[d] = m
    _rows.append(dict(d=float(d), energy=m["energy"], sigma=m["sigma"], shots=m["shots"], cz=CZ_ANSATZ))
    if m["capped"]:
        print(f"  aviso: d={d} llegó al tope de {MAX_SHOTS_PER_POINT} shots; σ queda por encima del objetivo")

vqe = pd.DataFrame(_rows, columns=["d", "energy", "sigma", "shots", "cz"])  # tabla del contrato (#6)
print(f"Curva medida con SamplerV2 en {time.time() - _t:.1f} s; shots totales = {vqe.shots.sum():,} "
      f"(piloto {sum(MEAS[d]['shots_pilot'] for d in D):,}).")

_show = vqe.assign(
    sigma_mHa=vqe.sigma * 1e3,
    sigma_obj_mHa=[sigma_target(d) * 1e3 / np.sqrt(SHOTS_SCALE) for d in D],
    grupos=[MEAS[d]["groups"] for d in D],
    K_Ha=[MEAS[d]["K"] for d in D],
    E_sv=[E_SV[d] for d in D],
    z=[(MEAS[d]["energy"] - E_SV[d]) / MEAS[d]["sigma"] for d in D],
)
print(_show.drop(columns="sigma").to_string(index=False, float_format=lambda x: f"{x:.6f}" if abs(x) > 50 or abs(x) < 1 else f"{x:.2f}"))

#### Validación de la medición

Tres pruebas, todas con `SamplerV2`:

1. **Medido vs. statevector en cada punto.** $z = (E_\text{medida} - E_\text{sv})/\sigma$ tiene que ser $|z| \lesssim 2$. Con 19
   puntos independientes es esperable que alguno caiga entre 2 y 3 (la probabilidad de que los 19 queden dentro de 2σ es
   ≈ 0,41), por eso además miramos $\chi^2 = \sum z^2$ contra 19 grados de libertad.
2. **σ es la dispersión real.** Repetimos la estimación completa (piloto + principal) 40 veces en tres distancias y
   comparamos el desvío estándar de las 40 energías con la σ que reporta cada corrida (si σ es correcta, el cociente cae en
   [0,82, 1,28] con 95 % de probabilidad). También miramos si la media de las 40 se aleja de $E_\text{sv}$.
3. **σ ∝ 1/√shots.** A 1,8 Å barremos el presupuesto en dos órdenes de magnitud y ajustamos la pendiente en log-log.

In [ ]:
# --- 4.2.5 Validación 1: medido vs. statevector en todos los puntos ----------------------------------
from scipy import stats

z = np.array([(MEAS[d]["energy"] - E_SV[d]) / MEAS[d]["sigma"] for d in D])
chi2 = float(np.sum(z ** 2))
p_chi2 = float(stats.chi2.sf(chi2, len(D)))
print(f"|z| <= 2 en {np.sum(np.abs(z) <= 2)}/{len(D)} puntos; |z| <= 3 en {np.sum(np.abs(z) <= 3)}/{len(D)}; "
      f"max |z| = {np.abs(z).max():.2f}; chi2 = {chi2:.1f} con {len(D)} gl (p = {p_chi2:.2f}).")
assert np.all(np.abs(z) < 3.5), "una energía medida está a más de 3.5σ del statevector: revisar la medición"

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.8))
ax1.errorbar(D, [(MEAS[d]["energy"] - E_SV[d]) * 1e3 for d in D], yerr=[MEAS[d]["sigma"] * 1e3 for d in D],
             fmt="o", ms=4, capsize=3, label="SamplerV2 − statevector (±σ)")
ax1.fill_between(D, [-2 * MEAS[d]["sigma"] * 1e3 for d in D], [2 * MEAS[d]["sigma"] * 1e3 for d in D],
                 color="0.85", label="±2σ")
ax1.axhline(0, color="k", lw=0.8)
ax1.axvline(STRETCHED, color="tab:red", ls=":", lw=1)
ax1.set(xlabel="d (Å)", ylabel="mHa", title="Energía medida − energía statevector")
ax1.legend(fontsize=8, loc="lower left")
ax2.plot(D, [MEAS[d]["sigma"] * 1e3 for d in D], "o-", ms=4, label="σ medida")
ax2.plot(D, [abs(REF[d]["e_ccsd"] - REF[d]["e_fci"]) * 1e3 for d in D], "s--", ms=4, label="|E_CCSD − E_FCI|")
ax2.axvline(STRETCHED, color="tab:red", ls=":", lw=1)
ax2.set(xlabel="d (Å)", ylabel="mHa", yscale="log", title="σ por punto vs. margen de CCSD")
ax2.legend(fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# --- 4.2.6 Validación 2: 50 repeticiones -> la dispersión coincide con σ ------------------------------
N_REPEATS = 40
REPEAT_SHOTS = _scaled(20_000, floor=10_000)  # el test es dispersión/σ: no necesita σ chica
_t = time.time()
rep_rows, REPEATS = [], {}
for j, d in enumerate((0.9, STRETCHED, DISSOCIATED)):
    d = float(D[np.argmin(np.abs(D - d))])
    op, e_nuc = HAM[d]
    # las 40 estimaciones completas (piloto + principal) salen de DOS jobs de SamplerV2: son bindings extra del mismo PUB
    runs = measure_energy_many(CIRCUIT[d], op, e_nuc, total_shots=REPEAT_SHOTS, seed=SEED + 10_000 + j, n_repeats=N_REPEATS,
                               pilot_shots_per_group=PILOT_SHOTS)
    E = np.array([r["energy"] for r in runs])
    S = np.array([r["sigma"] for r in runs])
    REPEATS[d] = (E, S)
    spread = E.std(ddof=1)
    lo, hi = np.sqrt((N_REPEATS - 1) / stats.chi2.ppf([0.975, 0.025], N_REPEATS - 1))  # IC 95% de spread/σ si σ es correcta
    rep_rows.append({"d": d, "shots por corrida": runs[0]["shots"], "σ media (mHa)": S.mean() * 1e3,
                     "dispersión (mHa)": spread * 1e3, "dispersión/σ": spread / S.mean(),
                     "IC95 esperado": f"[{lo:.2f}, {hi:.2f}]",
                     "(media − E_sv)/(σ/√N)": (E.mean() - E_SV[d]) / (S.mean() / np.sqrt(N_REPEATS))})
rep = pd.DataFrame(rep_rows)
print(f"{N_REPEATS} repeticiones x 3 distancias (2 jobs por distancia) en {time.time() - _t:.1f} s")
print(rep.round(3).to_string(index=False))
print("La última columna es ~N(0, 1) si no hay sesgo: con 3 puntos es normal que alguna llegue a ±2-3.")
assert all(0.6 < r < 1.6 for r in rep["dispersión/σ"]), "la dispersión entre semillas no coincide con σ"

fig, axs = plt.subplots(1, len(REPEATS), figsize=(11, 3.2))
for ax, (d, (E, S)) in zip(axs, REPEATS.items()):
    x = (E - E_SV[d]) * 1e3
    ax.hist(x, bins=10, density=True, color="tab:blue", alpha=0.6, label=f"{N_REPEATS} semillas")
    grid = np.linspace(-4, 4, 200) * S.mean() * 1e3
    ax.plot(grid, stats.norm.pdf(grid, 0, S.mean() * 1e3), "k-", label="N(0, σ reportada)")
    ax.set(title=f"d = {d} Å", xlabel="E − E_sv (mHa)")
axs[0].legend(fontsize=7)
plt.tight_layout()
plt.show()

In [ ]:
# --- 4.2.7 Validación 3: σ ∝ 1/sqrt(shots) a 1,8 Å ------------------------------------------------
d = float(STRETCHED)
op, e_nuc = HAM[d]
_k = max(SHOTS_SCALE, 0.1)  # el barrido no baja de 1/10 para que los pisos por grupo no lo distorsionen
budgets = [int(n * _k) for n in (20_000, 60_000, 200_000, 600_000)]
scan = [measure_energy(CIRCUIT[d], op, e_nuc, total_shots=n, seed=SEED + 20_000 + j, pilot_shots_per_group=PILOT_SHOTS)
        for j, n in enumerate(budgets)]
Nm = np.array([r["shots_main"] for r in scan])
Sg = np.array([r["sigma"] for r in scan])
slope = np.polyfit(np.log(Nm), np.log(Sg), 1)[0]
print(pd.DataFrame({"shots principal": Nm, "σ (mHa)": Sg * 1e3, "σ·sqrt(N) = K (Ha)": Sg * np.sqrt(Nm),
                    "E − E_sv (mHa)": [(r["energy"] - E_SV[d]) * 1e3 for r in scan]}).round(3).to_string(index=False))
print(f"pendiente log σ vs log N = {slope:.3f} (esperado −0.5)")
assert abs(slope + 0.5) < 0.08

plt.figure(figsize=(5, 3.5))
plt.loglog(Nm, Sg * 1e3, "o", label="σ medida (SamplerV2)")
plt.loglog(Nm, Sg[0] * np.sqrt(Nm[0] / Nm) * 1e3, "k--", lw=1, label="∝ 1/√N")
plt.errorbar(Nm, np.abs([(r["energy"] - E_SV[d]) * 1e3 for r in scan]), fmt="x", color="tab:orange", label="|E − E_sv|")
plt.axhline(abs(REF[d]["e_ccsd"] - REF[d]["e_fci"]) * 1e3, color="tab:red", ls=":", label="margen CCSD a 1,8 Å")
plt.xlabel("shots (corrida principal)")
plt.ylabel("mHa")
plt.title("σ ∝ 1/√shots a d = 1,8 Å")
plt.legend(fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# --- 4.2.8 Control cruzado con EstimatorV2 (NO es un resultado) ---------------------------------
# EstimatorV2 de qiskit-aer NO muestrea: calcula el valor exacto y, si precision > 0, le suma ruido gaussiano de ese ancho;
# `stds` devuelve exactamente `precision`. Lo mostramos solo para confirmar que el circuito y el observable son los mismos.
est = EstimatorV2(options={"run_options": {"seed_simulator": SEED}})
_x = []
for d in (0.9, STRETCHED, DISSOCIATED):
    d = float(D[np.argmin(np.abs(D - d))])
    op, e_nuc = HAM[d]
    r0 = est.run([(CIRCUIT[d], op)], precision=0.0).result()[0]
    r1 = est.run([(CIRCUIT[d], op)], precision=MEAS[d]["sigma"]).result()[0]
    _x.append({"d": d, "SamplerV2 (reportada)": MEAS[d]["energy"], "σ SamplerV2": MEAS[d]["sigma"],
               "Estimator precision=0": float(r0.data.evs) + e_nuc, "statevector": E_SV[d],
               "Estimator precision=σ": float(r1.data.evs) + e_nuc, "stds devuelto": float(r1.data.stds)})
print("CONTROL CRUZADO (no reportado): EstimatorV2 no usa shots; con precision=0 es el statevector exacto.")
print(pd.DataFrame(_x).to_string(index=False, float_format=lambda v: f"{v:.6f}"))
assert all(abs(r["Estimator precision=0"] - r["statevector"]) < 1e-8 for r in _x)

print(f"\nStand-ins usados en esta corrida: {list(STANDIN_USED) or 'ninguno (todas las piezas reales)'}")
print(f"Tiempo total de la sección 4.2: {time.time() - _T0_MEDICION:.0f} s")
vqe

#### Qué decidimos, por qué y qué no funcionó (#6)

**Qué decidimos**

- **Las energías reportadas salen de `SamplerV2` con shots explícitos**, y E y σ se calculan desde los bitstrings.
  `EstimatorV2` de qiskit-aer aparece solo como control cruzado: no muestrea (valor exacto + ruido gaussiano de ancho
  `precision`, `stds = precision`). Con `precision=0` coincide con el statevector hasta $10^{-8}$ Ha, o sea que el chequeo
  "medido ≈ exacto" hecho con Estimator no prueba nada.
- **Grupos QWC por *sorted insertion*** (términos ordenados por $|c_k|$): 37–41 grupos para los 184 términos no identidad
  (`group_commuting(qubit_wise=True)` de Qiskit da 46). El greedy propio es determinista y nos deja precomputar la tabla
  $h_g(x)$ de cada grupo para los $2^8$ bitstrings.
- **Valor por shot del grupo entero**: su varianza muestral ya incluye las covarianzas entre los términos del grupo; grupos
  distintos usan shots independientes, así que $\sigma^2 = \sum_g \operatorname{Var}_g/N_g$.
- **Reparto óptimo $N_g \propto \sqrt{\operatorname{Var}_g}$ (con $\operatorname{Var}_g$ la varianza del grupo entero, que incluye covarianzas; no $\sum|c_k|\sigma_k$ por término) con piloto separada** (500 shots por grupo, 3–8 % del total por
  punto). La piloto solo decide el reparto y el total; E y σ salen de shots nuevos, así el reparto no se correlaciona con la
  estimación. Los shots de la piloto se cuentan en `shots` porque son costo real. Piso de 100 shots por grupo.
- **Shots por punto desde el margen de CCSD**: $\sigma_\text{obj} = \min(\tfrac15|E_\text{CCSD}-E_\text{FCI}|,\ 3\text{ mHa})$ en
  $d\ge1{,}8$ Å y 3 mHa en el resto; el total sale de $N=(K/\sigma_\text{obj})^2$ con $K$ de la piloto. Resultado esperado a escala 1 (de una corrida previa a estos cambios; no re-medido): σ ≈ 1,85 mHa
  con ~7 × 10⁵ shots a 1,8 Å (margen 9,1 mHa) y σ ≈ 3 mHa con 2,4–3,4 × 10⁵ shots en los demás puntos; **≈ 6,0 × 10⁶ shots
  en toda la curva**. $K$ medido: 1,41–1,67 Ha, igual que lo que predice `docs/analysis/` (1,49 Ha a 1,8 Å).
- **Un PUB por estimación, cada grupo como asignación de rotaciones parametrizadas** $U(\theta_q,0,\lambda_q)$, repetida $m_g$
  veces con el mismo número de shots. Motivo principal: `SamplerV2` de Aer agrupa los PUBs por número de shots y corre cada
  grupo en un job con **la misma semilla**, así que PUBs con distinto número de shots salen correlacionados (medimos
  correlación 0,40 entre las medias de dos PUBs con 1000 y 1001 shots a lo largo de 200 semillas; 0,01 con los mismos shots).
  Dentro de un PUB cada asignación usa su propia corriente aleatoria (correlación ≈ 0). Bonus: Aer compila el circuito una
  sola vez por job.
- **`cz`** = `cz_count(ansatz)` del contrato (transpile_line) sobre el ansatz sin asignar, igual para todos los puntos. Para
  simular, el circuito se transpila a la base de Aer (más liviano que `transpile_line`).
- **Semillas**: cada estimación recibe una semilla entera (`SEED + 100·i` en la curva); de ella salen con `SeedSequence` dos
  semillas grandes, una para la piloto y otra para la principal (y una por cada job de 600 bindings). Todo es reproducible bit a bit
  (ver Reproducibilidad abajo: hubo que fijar PySCF a un hilo).
- **Stand-ins que se apagan solos** para #1, #2, #4 y #5 (avisan con una línea `[stand-in]`). El ansatz stand-in es UCCSD de un
  paso (26 parámetros, 1274 cz con transpile_line) ajustado por statevector en el sector de 36 determinantes: **no llega a FCI**
  (3,3 mHa arriba a 1,8 Å y 3,5 mHa a 2,0 Å). Sirve para construir y validar la medición, no es el resultado. El control
  "el Hamiltoniano reproduce FCI en el sector N=4, Sz=0" corre igual con el Hamiltoniano real.

**Validación** (corrida con `SHOTS_SCALE = 0,05`, la única que corrimos: 4,2 × 10⁵ shots en la curva, σ ≈ 13 mHa; a escala 1
σ baja ≈ 4,5 veces y el chequeo es más exigente; no la corrimos): $|z| \le 2$ en los 19 puntos, $\chi^2 = 13{,}4$ con 19 gl
($p = 0{,}82$). Con 40 repeticiones por punto, dispersión/σ = 0,91 (0,9 Å), 0,98 (1,8 Å) y 0,819 (3,0 Å): las dos
primeras dentro del intervalo del 95 % [0,82, 1,28] y la tercera pegada al borde inferior (una de tres a ese borde es esperable;
sin señal de σ subestimada, que daría > 1). Con 30 repeticiones y semillas fijas, 1,8 Å había salido en 0,76: fue una fluctuación, no
un defecto de σ. Pendiente de log σ vs. log N = −0,54 con $\sigma\sqrt N$ ≈ 1,54 Ha.

**Sesgo.** La columna $(\bar E - E_\text{sv})/(\sigma/\sqrt N)$ de la tabla de repeticiones salió alta algunas veces (hasta
2,8 con 40 repeticiones; 2,9 y 3,7 en corridas anteriores con 30). Para descartar un sesgo del estimador lo medimos aparte
(fuera del notebook) a 0,9 Å con 150 repeticiones y 4 semillas distintas: z = +0,6, −1,3, +0,7 y −0,8, sin signo
sistemático y con sesgo máximo de 1,1 mHa (σ ≈ 17 mHa por corrida; error estándar ≈ 1,4 mHa), compatible con 0. Con 3
distancias por corrida es esperable que alguna pase 2σ; la conclusión «sin sesgo» se apoya en esa prueba extendida y en el
$\chi^2$ de la curva, no solo en este último.

**Reproducibilidad.** Dos corridas seguidas con las mismas semillas dan tablas idénticas bit a bit (incluidos shots totales,
energías y σ). Antes no: el reparto de shots cambiaba entre corridas. La causa era PySCF: sus sumas en paralelo (OpenMP)
cambian de orden y los coeficientes de $H$ varían ~1e-14; el ajuste de θ lo amplifica a ~1e-9 Ha y eso mueve el redondeo del
reparto y el piloto. Se arregló con `pyscf.lib.num_threads(1)` (para H4 en STO-3G un hilo no es más lento). Los
redondeos de $H$ y θ* que ya había quedan como segunda defensa. Aer no es fuente de aleatoriedad: sus semillas son por
binding y da el mismo resultado con cualquier número de hilos (verificamos el hash de los bitstrings). En `references`,
`qubit_hamiltonian` y `params` reales de los compañeros hay que mantener esto (un hilo, o redondear) si quieren la misma garantía.
**Velocidad:** la sección tarda ≈ 50 s a escala 0,05 en una máquina sin carga (96–140 s con otros procesos corriendo; antes
≈ 116–162 s). Quedan fijos ≈ 9 s de PySCF, ≈ 9 s del ajuste de θ y ≈ 6 s de la curva. A escala 1 estimamos un tiempo parecido
porque el costo es por binding y no por shot (estimación, no medición).

**Qué no funcionó o descartamos**

- *Un PUB por grupo con shots distintos en un mismo `run`*: es lo natural, pero por lo de la semilla compartida la dispersión
  entre semillas daba ≈ 1,3–1,4 σ y $\chi^2$ salía mal ($p < 0{,}01$). Lo detectó la validación con repeticiones; por eso la
  dejamos en el notebook.
- *Un `run` de `SamplerV2` separado por grupo con semillas distintas*: arregla la correlación, pero cada `run` recompila el
  circuito (~35 ms por cada 1000 compuertas) y una estimación tardaba ~3–4 s.
- *Una repetición por llamada de `measure_energy` (30 × 3 distancias, 180 jobs)*: ~100 s. Lo que pesa es el binding, no el job,
  así que juntar las repeticiones en un PUB no alcanzó por sí solo: lo que bajó el costo fue `method="statevector"` (3×),
  bindings en paralelo (2,4×) y 2 paquetes por grupo en vez de 4. Con 100 repeticiones de golpe Aer falló por memoria
  (`bad allocation`); de ahí el corte en jobs de 600 bindings.
- *EstimatorV2 con `precision`*: descartado como fuente de números reportados (ver arriba).
- *Sumar varianzas término a término*: ignora covarianzas dentro del grupo y da una σ incorrecta.
- *Reusar los shots de la piloto en la estimación*: ahorra 3–8 %, pero acopla reparto y estimación; preferimos la versión limpia.
- *Mediciones más baratas* (rotación de base / factorización doble, ~6× menos shots a 1,8 Å según `docs/analysis/`): quedan
  como mejora posible; con QWC y reparto óptimo ya llegamos a σ = 1,8 mHa a 1,8 Å con pocos minutos de simulación.

## 5. Resultados: curva, $R_e$, $D_e$ y tabla (energía, $\sigma$, shots, `cz`)

Todo lo de esta sección se **recalcula** desde las tablas del contrato: `vqe` (energía medida con `SamplerV2`, σ, shots, cz; #6) y las
referencias CCSD/FCI de #1 (vía `get_references`). No hay números pegados a mano. Convención: energías **totales** en Ha, errores en mHa,
$\text{err} = |E - E_\text{FCI}|$. FCI y CCSD son solo para comparar; el resultado reportado es siempre el de VQE.

In [ ]:
# --- 5.1 Tabla por punto: refs + vqe -> `resultados` ------------------------------------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

MHA = 1e3  # Ha -> mHa
_refs_df = pd.DataFrame([get_references(float(d)) for d in vqe.d])[["d", "e_ccsd", "e_fci"]]
resultados = vqe.merge(_refs_df, on="d", how="left", validate="one_to_one")
assert resultados[["e_ccsd", "e_fci"]].notna().all().all(), "faltan referencias para algún d de vqe"
resultados["err_vqe"] = (resultados.energy - resultados.e_fci).abs() * MHA
resultados["err_ccsd"] = (resultados.e_ccsd - resultados.e_fci).abs() * MHA
resultados["sigma_mHa"] = resultados.sigma * MHA
# gana_a_ccsd = err_vqe + sigma < err_ccsd, solo para d >= 1,8 Å (en el resto queda <NA>)
_g = (resultados.err_vqe + resultados.sigma_mHa < resultados.err_ccsd)
resultados["gana_a_ccsd"] = pd.array(np.where(resultados.d >= STRETCHED - 1e-9, _g, pd.NA), dtype="boolean")

_cols = ["d", "energy", "sigma", "shots", "cz", "e_ccsd", "e_fci", "err_vqe", "err_ccsd", "gana_a_ccsd"]
tabla_resultados = resultados[_cols]
print(tabla_resultados.to_string(index=False, float_format=lambda x: f"{x:.6f}" if abs(x) < 1e-1 or abs(x) > 5 else f"{x:.3f}"))
_est = resultados[resultados.d >= STRETCHED - 1e-9]
print(f"\nPuntos con d >= {STRETCHED} Å donde VQE le gana a CCSD (err_vqe + sigma < err_ccsd): "
      f"{int(_est.gana_a_ccsd.sum())} de {len(_est)}")

In [ ]:
# --- 5.2 Gráfico: E(d) ± σ vs CCSD y FCI, y panel de errores en escala log ----------------------------
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12.5, 4.6), constrained_layout=True)
ax1.plot(resultados.d, resultados.e_fci, "-", color="k", lw=1.6, label="FCI")
ax1.plot(resultados.d, resultados.e_ccsd, "--", color="tab:orange", lw=1.6, label="CCSD")
ax1.errorbar(resultados.d, resultados.energy, yerr=resultados.sigma, fmt="o", ms=4, capsize=3, color="tab:blue",
             label=r"VQE ($\pm\sigma$, SamplerV2)")
ax1.axvline(STRETCHED, color="gray", ls=":", lw=1)
ax1.set(xlabel="d (Å)", ylabel="E total (Ha)", title="H$_4$ lineal, STO-3G: energía total")
ax1.legend()

_floor = 1e-3  # mHa: piso para que el log no se rompa si err_vqe ~ 0
ax2.errorbar(resultados.d, np.maximum(resultados.err_vqe, _floor), yerr=resultados.sigma_mHa, fmt="o-", ms=4, capsize=3,
             color="tab:blue", label=r"VQE: $|E-E_{FCI}|\ \pm\ \sigma$")
ax2.plot(resultados.d, np.maximum(resultados.err_ccsd, _floor), "s--", ms=4, color="tab:orange", label=r"CCSD: $|E-E_{FCI}|$")
ax2.axhline(1.6, color="crimson", lw=1, label="1,6 mHa (precisión química)")
ax2.axvline(STRETCHED, color="gray", ls=":", lw=1, label=f"d = {STRETCHED} Å")
ax2.set(xlabel="d (Å)", ylabel="error (mHa)", yscale="log", title="Error respecto de FCI")
ax2.legend(fontsize=8)
plt.show()

### 5.3 `R_e` y `D_e`

**`R_e`**: ajustamos un polinomio a las energías cerca del mínimo (con la grilla de 0,1 Å cae entre 0,8 y 1,0 Å) y tomamos el mínimo del ajuste.
Los ajustes de VQE van ponderados por $1/\sigma$. **Incertidumbre**: bootstrap paramétrico, re-muestreando cada energía como
$\mathcal N(E_i,\sigma_i^2)$ y repitiendo el ajuste; además reportamos la **sensibilidad** (rango al variar grado y ventana), que es la que domina para FCI/CCSD,
que no tienen ruido de medición (su incertidumbre estadística es 0 por construcción).

**`D_e = E(3,0 Å) − E(R_e)`**, con $E(R_e)$ el valor del ajuste en su mínimo y $E(3{,}0)$ el punto medido (con su σ). Para FCI y CCSD el mismo cálculo con sus energías.

*Aclaración sobre `D_e`*: la definición es la del enunciado, pero a 3,0 Å la cadena **todavía no está del todo disociada**. Abajo calculamos la energía de 4 átomos de H aislados
(UHF STO-3G) y verificamos que la FCI a 3,0 Å queda ~1 mHa por debajo de ella, es decir, este $D_e$ **subestima** la energía de atomización completa en esa cantidad.

In [ ]:
# --- 5.3 R_e por ajuste local + bootstrap + sensibilidad; D_e con incertidumbre ------------------------
D_FIN = 3.0
d_all = resultados.d.to_numpy()
BOOT_N = 2000
_rng = np.random.default_rng(SEED + 7)

# Configuración por defecto (la de las cifras reportadas) y barrido de sensibilidad
DEG0, WIN0 = 4, (0.6, 1.4)
DEGS = (2, 3, 4, 5)
WINS = ((0.7, 1.2), (0.6, 1.3), (0.6, 1.4), (0.6, 1.6))


def _min_fit(x, y, w, deg, win):
    """Mínimo del polinomio de grado `deg` ajustado dentro de `win` (acotado a la ventana). Devuelve (R_e, E(R_e))."""
    xs = np.linspace(win[0], win[1], 2001)
    c = np.polyfit(x, y, deg, w=w)
    ys = np.polyval(c, xs)
    i = int(np.argmin(ys))
    return xs[i], ys[i]


def _fit_batch(E, sig, deg, win, nboot=0, rng=None):
    """Ajuste en `win` a E (vector sobre toda la grilla). Con nboot>0 re-muestrea E ~ N(E, sig^2). Devuelve R_e, E(R_e), E(3.0)."""
    m = (d_all >= win[0] - 1e-9) & (d_all <= win[1] + 1e-9)
    x = d_all[m]
    if len(x) < deg + 2:
        return None
    w = None if sig is None else 1.0 / sig[m]
    i3 = int(np.argmin(np.abs(d_all - D_FIN)))
    if nboot == 0:
        r, e = _min_fit(x, E[m], w, deg, win)
        return r, e, E[i3]
    out = []
    for _ in range(nboot):
        Eb = E + rng.normal(0.0, 1.0, size=E.shape) * sig
        r, e = _min_fit(x, Eb[m], w, deg, win)
        out.append((r, e, Eb[i3]))
    return np.array(out)


SERIES = {  # nombre -> (energías, σ o None)
    "VQE": (resultados.energy.to_numpy(), resultados.sigma.to_numpy()),
    "FCI": (resultados.e_fci.to_numpy(), None),
    "CCSD": (resultados.e_ccsd.to_numpy(), None),
}

# Barrido de sensibilidad (sin bootstrap): R_e y D_e para cada (grado, ventana)
sens_rows = []
for name, (E, sig) in SERIES.items():
    for deg in DEGS:
        for win in WINS:
            f = _fit_batch(E, sig, deg, win)
            if f is None:
                continue
            r, e, e3 = f
            sens_rows.append(dict(metodo=name, grado=deg, ventana=f"{win[0]}-{win[1]}", R_e=r, D_e_mHa=(e3 - e) * MHA))
sens = pd.DataFrame(sens_rows)

# Resultados por defecto + bootstrap (solo VQE tiene σ de medición)
res_rows = []
for name, (E, sig) in SERIES.items():
    r0, e0, e3 = _fit_batch(E, sig, DEG0, WIN0)
    De0 = (e3 - e0) * MHA
    if sig is not None:
        bs = _fit_batch(E, sig, DEG0, WIN0, nboot=BOOT_N, rng=_rng)
        r_boot, De_boot = bs[:, 0].std(ddof=1), ((bs[:, 2] - bs[:, 1]) * MHA).std(ddof=1)
    else:
        r_boot = De_boot = 0.0
    s = sens[sens.metodo == name]
    r_sens = (s.R_e.max() - s.R_e.min()) / 2
    D_sens = (s.D_e_mHa.max() - s.D_e_mHa.min()) / 2
    res_rows.append(dict(metodo=name, R_e_A=r0, R_e_boot=r_boot, R_e_sens=r_sens,
                         D_e_mHa=De0, D_e_boot=De_boot, D_e_sens=D_sens))
RED = pd.DataFrame(res_rows)
RED["R_e_tot"] = np.hypot(RED.R_e_boot, RED.R_e_sens)   # incertidumbre combinada en cuadratura
RED["D_e_tot"] = np.hypot(RED.D_e_boot, RED.D_e_sens)
print(f"Ajuste por defecto: grado {DEG0}, ventana {WIN0[0]}-{WIN0[1]} Å, bootstrap de {BOOT_N} muestras.")
print(RED.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

print("\nSensibilidad de R_e (Å) al grado y a la ventana:")
print(sens.pivot_table(index=["metodo", "grado"], columns="ventana", values="R_e").to_string(float_format=lambda x: f"{x:.4f}"))
print("\nSensibilidad de D_e (mHa):")
print(sens.pivot_table(index=["metodo", "grado"], columns="ventana", values="D_e_mHa").to_string(float_format=lambda x: f"{x:.2f}"))

In [ ]:
# --- 5.4 Tablita final R_e / D_e y chequeo de la disociación a 3,0 Å ----------------------------------
tablita = pd.DataFrame({
    "metodo": RED.metodo,
    "R_e (Å)": [f"{r:.3f} ± {u:.3f}" for r, u in zip(RED.R_e_A, RED.R_e_tot)],
    "D_e (mHa)": [f"{x:.1f} ± {u:.1f}" for x, u in zip(RED.D_e_mHa, RED.D_e_tot)],
    "D_e (eV)": [f"{x * 0.0272114:.3f} ± {u * 0.0272114:.3f}" for x, u in zip(RED.D_e_mHa, RED.D_e_tot)],
    "± estad. R_e": [f"{u:.3f}" for u in RED.R_e_boot], "± sens. R_e": [f"{u:.3f}" for u in RED.R_e_sens],
    "± estad. D_e": [f"{u:.2f}" for u in RED.D_e_boot], "± sens. D_e": [f"{u:.2f}" for u in RED.D_e_sens],
})
print(tablita.to_string(index=False))
_ref = RED.set_index("metodo")
print(f"\nVQE - FCI: R_e = {_ref.R_e_A['VQE'] - _ref.R_e_A['FCI']:+.3f} Å, D_e = {_ref.D_e_mHa['VQE'] - _ref.D_e_mHa['FCI']:+.2f} mHa;  "
      f"CCSD - FCI: R_e = {_ref.R_e_A['CCSD'] - _ref.R_e_A['FCI']:+.3f} Å, D_e = {_ref.D_e_mHa['CCSD'] - _ref.D_e_mHa['FCI']:+.2f} mHa")

# Cuatro H aislados (UHF STO-3G, doblete): cota del límite de disociación completa
_h = gto.M(atom="H 0 0 0", basis="sto-3g", spin=1, verbose=0)
_e_h = scf.UHF(_h).run().e_tot
_gap = (resultados.set_index("d").e_fci[D_FIN] - 4 * _e_h) * MHA
print(f"\nCuatro H aislados (UHF STO-3G): {4 * _e_h:.6f} Ha; FCI(3,0 Å) queda {abs(_gap):.2f} mHa "
      f"{'por debajo' if _gap < 0 else 'por encima'} de ese límite -> a 3,0 Å la cadena no está del todo disociada "
      f"y D_e(3,0) subestima la energía de atomización completa en ~{abs(_gap):.1f} mHa.")

#### Qué decidimos, por qué y qué no funcionó (#7)

**Qué decidimos**

- **Todo sale de `vqe` + referencias** (`get_references`, que usa `references(d)` de #1 y cae al stand-in de PySCF solo si falta). `err_vqe = |E_medida - E_FCI|` usa la
  energía *medida* (con su ruido); por eso la columna `gana_a_ccsd = err_vqe + σ < err_ccsd` es conservadora: exige ganar aun sumando una σ. Se evalúa solo para $d \ge 1{,}8$ Å (`<NA>` antes).
- **`R_e`**: polinomio de grado 4 en 0,6–1,4 Å, mínimo buscado en una malla fina dentro de la ventana. VQE ajustado con pesos $1/\sigma$. Se eligió una ventana que incluye varios puntos a cada lado del mínimo
  (que cae entre 0,8 y 1,0 Å) porque la curva es muy asimétrica y una parábola sola lo sesga.
- **Incertidumbres en dos partes**: *estadística* (bootstrap paramétrico $\mathcal N(E_i,\sigma_i^2)$, 2000 muestras, semilla fija) y *de sensibilidad* (semi-rango al variar grado 2–5 y cuatro ventanas), combinadas en cuadratura.
  Para FCI y CCSD no hay ruido de medición, así que solo cuenta la sensibilidad, que además mide el error sistemático del método de ajuste con la grilla de 0,1 Å (se ve mejor comparando FCI con VQE).
- **`D_e = E(3,0) − E(R_e)`** con $E(R_e)$ el mínimo del ajuste y $E(3{,}0)$ el punto medido; en el bootstrap ambos se re-muestrean juntos, así se propaga la correlación entre ellos. En FCI/CCSD, mismo cálculo.
- **Disociación incompleta**: la FCI a 3,0 Å se compara con cuatro H aislados (UHF STO-3G) calculados en la propia celda; la diferencia (~1 mHa) se imprime y se aclara en el texto.
- **Gráfico**: el panel de errores usa un piso de $10^{-3}$ mHa para que la escala log no falle si un error sale ~0, y barras $\pm\sigma$ sobre el error de VQE.

**Qué no funcionó o descartamos**

- *Parábola sola*: con una ventana ancha da un `R_e` sesgado (la curva es asimétrica); queda en la tabla de sensibilidad como contraste, no como resultado.
- *Spline cúbico*: con σ de ~1–3 mHa por punto interpola el ruido y el mínimo salta; preferimos un ajuste ponderado de grado bajo, que lo suaviza.
- *Ajustar con todos los puntos hasta 3,0 Å*: los puntos lejanos (casi planos) tiran del polinomio y ensanchan la dependencia con el grado.
- *Reportar solo la incertidumbre del bootstrap*: subestima el error, porque no ve el sesgo del modelo de ajuste; por eso se suma la sensibilidad.
- Mientras #4/#5 sean stand-ins, el ansatz no llega a FCI (3–4 mHa arriba a 1,8–2,0 Å): `gana_a_ccsd` no es un resultado final, solo muestra que el pipeline funciona.

## 6. Zona estirada ($d \geq 1{,}8$ Å) vs. CCSD

## Resultados clave

Resumen de una pantalla, recalculado desde las tablas del contrato `refs` (#1) y `vqe` (#6), sin números pegados a mano: $R_e$ y $D_e$ (VQE contra FCI y CCSD), error máximo en la zona estirada, si le ganamos a CCSD con la barra de error incluida, precisión química, `cz` por circuito y shots totales.

- $R_e$ sale de un ajuste cúbico local (5 puntos de la grilla alrededor del mínimo; para VQE, pesado con $1/\sigma$) y $D_e = E(3{,}0\ \text{Å}) - E(R_e)$ con $E(R_e)$ del mismo ajuste. Es una estimación rápida, igual para los tres métodos; los valores con incertidumbre (bootstrap) son los de la sección 5.
- "Le ganamos a CCSD" se evalúa punto por punto para $d \geq 1{,}8$ Å con el criterio del enunciado: $|E_\text{VQE} - E_\text{FCI}| + \sigma < |E_\text{CCSD} - E_\text{FCI}|$.
- Si `refs` o `vqe` todavía no existen, están incompletas o son un *stand-in* (marcado con `tabla.attrs["mock"] = True`), la celda lo dice y no calcula nada: nunca muestra números de mentira como si fueran resultados.

In [ ]:
# Resultados clave, recalculados desde `refs` (#1) y `vqe` (#6). Si faltan, avisa en vez de fallar.
from IPython.display import Markdown, display

_COLS = {"refs": ["d", "e_hf", "e_ccsd", "e_fci"], "vqe": ["d", "energy", "sigma", "shots", "cz"]}


def contract_table(name: str) -> tuple[pd.DataFrame | None, str | None]:
    """(tabla, None) si `name` cumple el contrato 0.1 en toda la grilla D; si no, (None, motivo)."""
    t = globals().get(name)
    if t is None:
        return None, f"`{name}` todavía no está definida"
    if getattr(t, "attrs", {}).get("mock"):
        return None, f"`{name}` es un stand-in (`attrs['mock']`), no un resultado"
    try:
        t = pd.DataFrame(t)
    except (TypeError, ValueError):
        return None, f"`{name}` no se puede leer como tabla"
    missing = [c for c in _COLS[name] if c not in t.columns]
    if missing:
        return None, f"a `{name}` le faltan las columnas {missing}"
    t = t[_COLS[name]].assign(d=lambda x: x.d.astype(float).round(2))
    absent = sorted(set(D) - set(t.d))
    if absent:
        return None, f"a `{name}` le faltan las distancias {', '.join(f'{x:g}' for x in absent)} Å"
    t = t[t.d.isin(D)].drop_duplicates("d").sort_values("d").reset_index(drop=True)
    if t.isna().any().any():
        return None, f"`{name}` tiene valores vacíos (NaN)"
    return t, None


def local_minimum(d: np.ndarray, e: np.ndarray, sigma: np.ndarray | None = None) -> tuple[float, float]:
    """(R_e, E(R_e)): mínimo de un ajuste cúbico a los 5 puntos de la grilla más cercanos al mínimo."""
    near = np.argsort(np.abs(d - d[np.argmin(e)]))[:5]
    w = None if sigma is None else 1 / np.maximum(sigma[near], 1e-12)
    poly = np.poly1d(np.polyfit(d[near], e[near], 3, w=w))
    x = np.linspace(d[near].min(), d[near].max(), 4001)
    return x[np.argmin(poly(x))], poly(x).min()


def key_results(refs: pd.DataFrame, vqe: pd.DataFrame) -> None:
    t = refs.merge(vqe, on="d")
    t["err_vqe"] = (t.energy - t.e_fci).abs() * 1e3   # mHa
    t["err_ccsd"] = (t.e_ccsd - t.e_fci).abs() * 1e3  # mHa
    t["sigma_mHa"] = t.sigma * 1e3
    t["gana_a_ccsd"] = t.err_vqe + t.sigma_mHa < t.err_ccsd
    s = t[t.d >= STRETCHED]

    d = t.d.to_numpy()
    e_end = t.set_index("d")
    curves = {"VQE": (t.energy.to_numpy(), t.sigma.to_numpy()), "FCI": (t.e_fci.to_numpy(), None),
              "CCSD": (t.e_ccsd.to_numpy(), None)}
    col = {"VQE": "energy", "FCI": "e_fci", "CCSD": "e_ccsd"}
    rows = {}
    for name, (e, sig) in curves.items():
        r_e, e_min = local_minimum(d, e, sig)
        rows[name] = {"R_e (Å)": r_e, "D_e (mHa)": (e_end.loc[DISSOCIATED, col[name]] - e_min) * 1e3}
    display(pd.DataFrame(rows).T.round({"R_e (Å)": 3, "D_e (mHa)": 1}))

    worst = s.loc[s.err_vqe.idxmax()]
    margin = s.loc[(s.err_ccsd - s.err_vqe - s.sigma_mHa).idxmin()]
    yes = lambda ok: "**sí**" if ok else "**no**"
    cz = sorted(t.cz.astype(int).unique())
    lo, hi = int(t.shots.min()), int(t.shots.max())
    shots_range = f"{lo:,}" if lo == hi else f"{lo:,}–{hi:,}"
    display(Markdown("\n".join([
        f"- **Error máximo en la zona estirada** ($d \\geq {STRETCHED}$ Å): {worst.err_vqe:.2f} mHa a {worst.d} Å "
        f"($\\sigma$ = {worst.sigma_mHa:.2f} mHa); CCSD ahí erra {worst.err_ccsd:.1f} mHa.",
        f"- **¿Le ganamos a CCSD?** {yes(s.gana_a_ccsd.all())}: en {int(s.gana_a_ccsd.sum())} de {len(s)} puntos "
        f"$|E_\\text{{VQE}} - E_\\text{{FCI}}| + \\sigma < |E_\\text{{CCSD}} - E_\\text{{FCI}}|$. Margen más chico: "
        f"{margin.err_ccsd - margin.err_vqe - margin.sigma_mHa:.2f} mHa a {margin.d} Å.",
        f"- **¿Precisión química** ($< {CHEM_ACC * 1e3:g}$ mHa)? Zona estirada: {yes((s.err_vqe < CHEM_ACC * 1e3).all())} "
        f"({int((s.err_vqe < CHEM_ACC * 1e3).sum())} de {len(s)} puntos); toda la curva: "
        f"{int((t.err_vqe < CHEM_ACC * 1e3).sum())} de {len(t)} puntos.",
        f"- **`cz` por circuito:** {', '.join(map(str, cz))} (transpilado a `cz, sx, x, rz` en línea).",
        (f"- **Shots:** {int(t.shots.sum()):,} en total para {len(t)} puntos ({shots_range} por punto)."
         .replace(",", ".")),
    ])))


_tables = {name: contract_table(name) for name in _COLS}
_pending = [why for _, why in _tables.values() if why]
if _pending:
    display(Markdown("**Resultados pendientes.** " + "; ".join(_pending) + ". "
                     "Esta celda se completa sola cuando las secciones 1 y 4 llenen `refs` y `vqe`."))
else:
    key_results(_tables["refs"][0], _tables["vqe"][0])

## 7. Decisiones, qué no funcionó y limitaciones

Esta sección junta lo que documentó cada sección (el detalle, con tablas y pruebas, está en el markdown de decisiones de cada una; acá va el resumen y el enlace) y analiza hasta dónde funciona el método, dónde falla y por qué. Las decisiones de infraestructura también están como ADR en `docs/adr/` del repo.

### 7.1 Decisiones

| Etapa | Decisión | Por qué | Detalle |
|---|---|---|---|
| Referencias (#1) | RHF (no UHF) con chequeo de estabilidad interna, CCSD y FCI singlete con PySCF sobre la grilla `D` → `refs`. | Queremos un singlete con orbitales restringidos, como el Hamiltoniano de qubits; HF/CCSD/FCI solo sirven para comparar y elegir puntos de partida (regla del desafío). | Sección 1 |
| Hamiltoniano (#2) | `qubit_hamiltonian(d) -> (op, e_nuc)`; Jordan-Wigner escrito a mano, genérico; orbitales RHF con signo fijo; caché que devuelve copias. | Sin dependencias nuevas y con control exacto del orden de qubits; el signo fijo hace que los coeficientes de $\hat H$ varíen suave con $d$ (si no, el *warm start* arranca lejos); RHF en toda la grilla para que la referencia siga siendo un singlete y $\langle S^2\rangle$ sirva de diagnóstico. | Sección 2, *Decisiones (#2)*; [ADR-0001](https://github.com/Santiago-Garrote/QJ/blob/main/docs/adr/0001-firma-qubit-hamiltonian.md), [ADR-0002](https://github.com/Santiago-Garrote/QJ/blob/main/docs/adr/0002-jordan-wigner-a-mano.md), [ADR-0003](https://github.com/Santiago-Garrote/QJ/blob/main/docs/adr/0003-signo-orbitales-y-cache.md) |
| Ansatz (#4) | 2-QUpCCGSD podado: 19 excitaciones de qubit (pares generalizados + simples con el mismo ángulo en α y β) en 2 capas, como rotaciones de Givens; **465 `cz`**, 19 parámetros. | Llega a FCI en toda la grilla (≤ 0,03 mHa con statevector), donde UCCSD se traba (3,3 mHa a 1,8 Å) por la misma razón que CCSD; conserva $N$ y $S_z$; Givens con multiplexor Gray en vez de `PauliEvolutionGate` (837 → 590 `cz`) y poda de las excitaciones de menor ángulo (590 → 465). | Sección 3, *Decisión (#4)* |
| Optimización (#5) | BFGS con gradiente adjunto exacto en el sector de 36 estados; *warm start* en las dos direcciones con varios arranques por punto y rondas de pulido; se minimiza $E + 0{,}05\,\langle S^2\rangle$. | La regla de *parameter-shift* de 2 términos no vale para excitaciones dobles; los mínimos locales obligan a multi-start; la penalidad evita elegir estados con triplete casi degenerados en energía. Error ≤ 0,003 mHa en toda la grilla. | Sección 4 |
| Medición (#6) | *Pendiente:* `SamplerV2`, agrupamiento de Paulis, reparto de shots, cálculo y validación de $\sigma$. | | Sección 4 |
| Resultados (#7) | *Pendiente:* ajuste de $R_e$, incertidumbre de $R_e$ y $D_e$. | | Sección 5 |
| Confiabilidad (#8) | *Pendiente:* evidencias sin FCI ($\langle S^2\rangle$, varianza/Temple, arranques múltiples, $\sigma$ vs. statevector). | | Sección 6 |

### 7.2 Qué no funcionó

- **`A.compose(B)` para multiplicar operadores (#2).** En Qiskit es $B \cdot A$: daba el Hamiltoniano con partículas y huecos intercambiados, que a medio llenado tiene *el mismo espectro* en el sector $N = 4$. La prueba contra FCI pasaba igual; solo la prueba de HF lo detectó. Lección: hacen falta las dos pruebas. Se usa `@`.
- **Orbitales sin signo fijo (#2, #4).** Entre 76 y 84 términos de Pauli cambiaban de signo de golpe entre distancias vecinas; con un Hamiltoniano provisorio así, el *warm start* del ansatz caía en otros mínimos y aparecía contaminación de triplete ($\langle S^2\rangle$ hasta 3e-2) desde 2,0 Å.
- **UCCSD y 1-UpCCGSD (#4):** no llegan a precisión química en la zona estirada (3,3 y 11 mHa a 1,8 Å).
- **Hardware-efficient (#4):** 42–56 `cz`, pero optimizando la energía queda a 23 mHa de FCI a 1,8 Å; puede representar el estado (optimizando contra el vector FCI, cosa imposible en un VQE real), así que el problema es el paisaje de optimización. Además no conserva $N$ ni $S_z$.
- **QEB-ADAPT-VQE (#4):** recomendado por el análisis previo; llega a FCI pero necesita 461–573 `cz` en línea (sus dobles tocan qubits lejanos) y un circuito distinto por distancia. No mejora al elegido.
- **`EstimatorV2` de `qiskit_aer` para reportar:** no muestrea; devuelve el valor exacto más ruido gaussiano opcional. Equivale a leer el statevector, así que se descartó para los números reportados (#6).
- **Elegir el mejor arranque solo por energía (#5, #16).** En la zona estirada a veces ganaba un estado con $\langle S^2\rangle \approx 3\cdot 10^{-2}$ a solo 0,02 mHa de FCI, y cuál ganaba cambiaba entre corridas por diferencias numéricas de ~1e-13 Ha. Se agregó la penalidad en $\langle S^2\rangle$.
- **Arrancar desde HF o al azar (#5).** Desde $\theta = 0$ el optimizador queda a 150–290 mHa en la zona estirada; con arranques al azar la mediana queda a 0,1–3 mHa. Hace falta el *warm start*; un solo sentido del barrido deja hasta 0,1 mHa, los dos más el pulido bajan a 0,003 mHa.
- *Pendiente (#6–#8):* lo que no funcionó en medición, ajuste de $R_e$ y confiabilidad.

### 7.3 Limitaciones: hasta dónde funciona y dónde falla

**Dónde funciona.** Sin ruido, en este sistema, el ansatz representa el fundamental en toda la grilla (≤ 0,03 mHa con statevector, singlete con $\langle S^2\rangle \leq 3\cdot 10^{-7}$) y el error que queda en la energía reportada es estadístico: lo fija el presupuesto de shots, no el método. Ganarle a CCSD en la zona estirada es holgado (el margen más chico es 9,1 mHa a 1,8 Å); lo exigente es la precisión química, que depende de $\sigma$.

**Dónde falla o deja de valer, y por qué:**

- **Base mínima (STO-3G).** La curva es la del *modelo* $\mathrm{H_4}$/STO-3G: "exacto" quiere decir FCI en esta base, no el $\mathrm{H_4}$ real. $R_e$ y $D_e$ cambian bastante en una base más grande, que además necesitaría muchos más qubits. Y a 3,0 Å la cadena todavía no está del todo disociada: FCI queda ~1 mHa por debajo de cuatro átomos de H aislados, así que $D_e$ (definida por el enunciado con 3,0 Å) está levemente subestimada respecto del límite de disociación.
- **Solo 8 qubits.** El sector relevante ($N = 4$, $S_z = 0$) tiene 36 determinantes: FCI es trivial clásicamente y por eso podemos validar todo contra él. El método no demuestra ventaja cuántica; demuestra el procedimiento. Al crecer el sistema crecen la cantidad de excitaciones (y de `cz`), los términos del Hamiltoniano y los shots necesarios para un $\sigma$ dado.
- **Búsqueda de parámetros con statevector.** Está permitida, pero cuesta exponencial en el número de qubits y aprovecha que el ansatz vive en el sector de 36 estados. En hardware real habría que optimizar con energías y gradientes medidos (con shots y ruido), mucho más lento e inestable.
- **El ansatz se eligió y se podó mirando FCI.** Comparar contra FCI está permitido, pero en un sistema donde FCI no se puede calcular la poda tendría que guiarse por otro criterio (gradientes al estilo ADAPT, varianza de la energía). Por eso la sección 6 arma el argumento de confiabilidad sin FCI.
- **Mínimos locales.** El paisaje tiene mínimos locales: con un solo arranque algunas distancias quedan a 1 mHa o a decenas de mHa de FCI. Usamos varios arranques y *warm start*, pero no hay garantía de mínimo global; la curva suave y los arranques coincidentes son evidencia, no prueba.
- **Contaminación de espín.** El ansatz conserva $N$ y $S_z$ pero no $S^2$ (las excitaciones de qubit no están adaptadas al espín). En la zona estirada el triplete está casi degenerado (0,46 mHa a 3,0 Å), así que una mezcla singlete-triplete casi no cambia la energía ni la varianza: solo $\langle S^2\rangle$ la delata. Hay que medirlo en cada punto juzgado (sección 6).
- **Hartree-Fock deja de ser buena referencia.** RHF es inestable hacia UHF desde ~1,1 Å y su peso en el estado exacto cae a 0,27 a 3,0 Å. Mantenemos orbitales RHF (solo definen la base), pero el estado inicial `HF_BITSTRING` está lejos del fundamental estirado y el ansatz tiene que hacer todo el trabajo multirreferencia.
- **Costo de medición.** $\sigma \propto 1/\sqrt{N_\text{shots}}$: con grupos QWC y reparto óptimo, $\sigma = 1$ mHa cuesta ~2 × 10⁶ shots por punto (`docs/analysis/`). Bajar $\sigma$ a la mitad cuesta 4 veces más shots.
- **Ruido: con 465 `cz` el circuito no sobrevive en hardware sin mitigación.** Con 0,1 % de error por `cz` (modelo de `ruido.py`), la probabilidad de al menos una falla por shot es $1 - 0{,}999^{465} \approx 37\,\%$, sin contar las compuertas de un qubit ni el *readout*. En `FakeFez` circuitos de este tamaño (520–640 `cz`) quedan 600–800 mHa lejos de FCI, peor que Hartree-Fock (`docs/analysis/state-of-the-art.md`, sección 5). Los resultados del desafío central son **sin ruido**; con ruido harían falta un ansatz más corto (la versión de 17 excitaciones tiene 411 `cz`), menos qubits por simetrías y mitigación (extras).
- *Pendiente (#6–#8):* limitaciones que aparezcan en optimización, medición y confiabilidad (p. ej. puntos donde $\sigma$ no alcance para precisión química, o donde $\langle S^2\rangle$ se aleje de 0).

## 8. Extras

Análisis complementarios (ruido, hardware simulado, esquemas de medición). No son parte del resultado reportado: todas las energías reportadas salen de `SamplerV2` con shots reales (#6).

### 8.1 Ruido (ruido.py)

Extra de #11. Repetimos la medición de #6 en cuatro distancias representativas (0,9 / 1,8 / 2,4 / 3,0 Å) sobre un simulador
**con ruido**: error despolarizante de 0,1 % por `cz` y 0,005 % por compuerta de un qubit (`sx`, `x`, `rz`), conectividad en
línea (el modelo común del enunciado, `ruido.py`). Medimos con la misma `measure_energy` de #6 (`SamplerV2` de `qiskit_aer`,
shots reales), con los mismos $\theta^*$ y el circuito de `transpile_line`, y comparamos **con ruido vs. sin ruido vs. FCI**.
Como el ansatz exacto tiene cientos de `cz`, agregamos **versiones cortas** (pocos operadores) para ver el compromiso
entre expresividad (sesgo del ansatz) y ruido (crece con los `cz`).

`ruido.py` todavía no está en el repo: si existe `notebooks/ruido.py` (o `ruido.py` junto al notebook) se usa; si no, un
**stand-in** con la especificación del enunciado, que se apaga solo cuando llegue el archivo oficial.

In [ ]:
# --- 8.1.1 Modelo de ruido: ruido.py oficial o stand-in con la especificación del enunciado ---------------------
import importlib.util
from contextlib import contextmanager
from qiskit_aer.noise import NoiseModel, depolarizing_error

P_CZ, P_1Q = 1e-3, 5e-5          # especificación del enunciado
_N_U_DECOMP = 5                  # u(θ,φ,λ) = rz sx rz sx rz: la rotación de base de la medición cuesta 5 compuertas de 1 qubit


def _standin_noise_model():
    """STAND-IN de ruido.py: despolarizante 1e-3 en cz y 5e-5 en sx/x/rz. El `u` de rotación de base de la medición no
    está en la base del modelo; se le asigna el error de su descomposición (5 compuertas de 1 qubit)."""
    nm = NoiseModel(basis_gates=["cz", "sx", "x", "rz", "u"])
    nm.add_all_qubit_quantum_error(depolarizing_error(P_CZ, 2), ["cz"])
    nm.add_all_qubit_quantum_error(depolarizing_error(P_1Q, 1), ["sx", "x", "rz"])
    nm.add_all_qubit_quantum_error(depolarizing_error(1 - (1 - P_1Q) ** _N_U_DECOMP, 1), ["u"])
    return nm


def _load_ruido_py():
    """Intenta cargar ruido.py (nombres de función probables); devuelve un NoiseModel o None. API oficial a confirmar (#3)."""
    for path in ("notebooks/ruido.py", "ruido.py"):
        if os.path.exists(path):
            spec = importlib.util.spec_from_file_location("ruido", path)
            mod = importlib.util.module_from_spec(spec)
            spec.loader.exec_module(mod)
            for name in ("get_noise_model", "build_noise_model", "noise_model", "make_noise_model", "NOISE_MODEL"):
                obj = getattr(mod, name, None)
                if obj is not None:
                    return obj() if callable(obj) else obj
    return None


NOISE_MODEL = _load_ruido_py()
if NOISE_MODEL is None:
    _notice_standin("ruido", "ruido.py (#3) no está: uso _standin_noise_model (despolarizante 1e-3 cz, 5e-5 sx/x/rz).")
    NOISE_MODEL = _standin_noise_model()
print("Modelo de ruido:", "stand-in del enunciado" if "ruido" in STANDIN_USED else "ruido.py oficial",
      "| compuertas con error:", sorted(NOISE_MODEL.noise_instructions))


@contextmanager
def con_ruido(nm=None):
    """Dentro del bloque, el SamplerV2 que usa measure_energy (de #6) simula con el modelo de ruido `nm`: se cambian las
    opciones de backend de Aer (`_AER_OPTIONS` de #6). Matriz densidad: el circuito se evoluciona una vez por binding y los
    shots se muestrean del estado final."""
    nm = NOISE_MODEL if nm is None else nm
    g = globals()
    original = g["_AER_OPTIONS"]
    g["_AER_OPTIONS"] = {**original, "method": "density_matrix", "noise_model": nm}
    try:
        yield
    finally:
        g["_AER_OPTIONS"] = original


@contextmanager
def _con_generadores(subset):
    """Cambia temporalmente los generadores del stand-in de #4 (para armar/ajustar una versión corta del ansatz)."""
    g = globals()
    original = g["_STANDIN_GENS"]
    g["_STANDIN_GENS"] = [original[i] for i in subset]
    try:
        yield
    finally:
        g["_STANDIN_GENS"] = original

In [ ]:
# --- 8.1.2 Versiones del ansatz: completo + cortas (pocos operadores), todas con transpile_line -------------------
DIST_RUIDO = [0.9, 1.8, 2.4, 3.0]
K_CORTAS = [2, 12]                      # operadores de la versión corta (el completo se agrega aparte)
SEED_RUIDO = SEED + 77_000
_t = time.time()

if "build_ansatz" in STANDIN_USED:
    # Ranking de operadores por tamaño del ángulo óptimo (máximo sobre la curva); las cortas conservan los más importantes
    # y se REAJUSTAN por statevector (buscar parámetros vale de cualquier forma).
    _rank = list(np.argsort(-np.max([np.abs(THETA[d]) for d in D], axis=0)))
    VERSIONES = {}
    for k in K_CORTAS + [len(_rank)]:
        sub = sorted(int(i) for i in _rank[:k])
        with _con_generadores(sub):
            qc_k = _standin_build_ansatz()
            if k < len(_rank):
                th_k, _ = _standin_fit_uccsd({d: HAM[d][0] for d in D}, {d: HAM[d][1] for d in D})
            else:
                th_k = THETA
        VERSIONES[k] = (qc_k, th_k)
else:
    # Ansatz real de #4: versión corta = los k operadores de mayor |θ*| (máximo sobre la curva) con el resto en 0 (el
    # transpilador elimina las rotaciones nulas), y los k ángulos REAJUSTADOS por statevector en el sector N=4, Sz=0
    # (`sector_ansatz` y `H_SECTOR` de la sección 4; buscar parámetros vale de cualquier forma).
    from scipy.optimize import minimize as _minimize
    _n = ansatz.num_parameters
    _rank = list(np.argsort(-np.max([np.abs(THETA[d]) for d in D], axis=0)))

    def _theta_corto(sub, d):
        Hs = H_SECTOR[float(d)][0]

        def fg(y):
            x = np.zeros(_n)
            x[sub] = y
            e, g = sector_ansatz.energy_grad(x, Hs)
            return e, g[sub]

        best = min((_minimize(fg, y0, jac=True, method="BFGS", options={"gtol": 1e-8}) for y0 in (THETA[d][sub], np.zeros(len(sub)))),
                   key=lambda r: r.fun)
        x = np.zeros(_n)
        x[sub] = np.round(best.x, 6)
        return x

    VERSIONES = {}
    for k in K_CORTAS + [_n]:
        sub = sorted(int(i) for i in _rank[:k])
        VERSIONES[k] = (ansatz, THETA if k == _n else {d: _theta_corto(sub, d) for d in D})

def _observable_fisico(circ, op):
    """transpile_line elige un layout y puede rutear con swaps: el qubit lógico i termina en el físico f[i]. Para medir el MISMO
    observable sobre el circuito transpilado se lleva cada Pauli del qubit lógico i al físico f[i]."""
    f = circ.layout.final_index_layout(filter_ancillas=True) if circ.layout is not None else list(range(circ.num_qubits))
    return op.apply_layout(list(f), num_qubits=circ.num_qubits)


CIRC_LINEA, CZ_V = {}, {}
for k, (qc_k, th_k) in VERSIONES.items():
    for d in DIST_RUIDO:
        CIRC_LINEA[(k, d)] = transpile_line(qc_k.assign_parameters(th_k[d]))   # cz, sx, x, rz sobre línea
        if k == max(VERSIONES):   # control: el circuito de línea + observable permutado reproduce la energía del ansatz de #6
            _e = float(Statevector(CIRC_LINEA[(k, d)]).expectation_value(_observable_fisico(CIRC_LINEA[(k, d)], HAM[d][0])).real) + HAM[d][1]
            assert abs(_e - E_SV[d]) < 1e-6, f"layout de transpile_line mal manejado en d={d}: {_e} vs {E_SV[d]}"
    CZ_V[k] = CIRC_LINEA[(k, DIST_RUIDO[0])].count_ops().get("cz", 0)
print(f"Versiones del ansatz transpiladas en {time.time() - _t:.1f} s:",
      {f"{k} op": f"{CZ_V[k]} cz" for k in VERSIONES})

In [ ]:
# --- 8.1.3 Medición: sin ruido y con ruido (SamplerV2 de qiskit_aer), mismos θ*, mismo circuito ---------------------
SIGMA_RUIDO = 3e-3 / np.sqrt(SHOTS_SCALE)   # el ruido sesga mucho más que 3 mHa: no hace falta más precisión
PILOT_RUIDO = _scaled(PILOT_SHOTS_PER_GROUP, floor=200)
# Con ruido cada binding (grupo QWC) es una evolución completa de la matriz densidad: lo caro son los bindings, no los shots.
# Un piso alto de shots por binding deja ~1 binding por grupo (en vez de ~4) y acelera ~3x sin cambiar σ.
MIN_SHOTS_BINDING = 5_000
_t = time.time()
_rows = []
for k in VERSIONES:
    for j, d in enumerate(DIST_RUIDO):
        circ = CIRC_LINEA[(k, d)]
        op, e_nuc = _observable_fisico(circ, HAM[d][0]), HAM[d][1]
        seed = SEED_RUIDO + 1000 * k + j
        m0 = measure_energy(circ, op, e_nuc, target_sigma=SIGMA_RUIDO, seed=seed, pilot_shots_per_group=PILOT_RUIDO, min_group_shots=MIN_SHOTS_BINDING)
        with con_ruido():
            m1 = measure_energy(circ, op, e_nuc, target_sigma=SIGMA_RUIDO, seed=seed, pilot_shots_per_group=PILOT_RUIDO, min_group_shots=MIN_SHOTS_BINDING)
        _rows.append(dict(ops=k, d=d, cz=CZ_V[k], n1q=sum(v for g_, v in circ.count_ops().items() if g_ != "cz"),
                          E_sin=m0["energy"], s_sin=m0["sigma"], E_con=m1["energy"], s_con=m1["sigma"],
                          shots=m0["shots"] + m1["shots"], E_fci=REF[d]["e_fci"], E_ccsd=REF[d]["e_ccsd"],
                          E_sv=float(Statevector(circ).expectation_value(op).real) + e_nuc))
RUIDO = pd.DataFrame(_rows)
print(f"{len(RUIDO)} puntos (cada uno sin y con ruido) en {time.time() - _t:.1f} s; shots totales {RUIDO.shots.sum():,}")

In [ ]:
# --- 8.1.4 Tabla: ruido vs. sin ruido vs. FCI (y vs. CCSD) ------------------------------------------------------------
RUIDO["err_sin_mHa"] = (RUIDO.E_sin - RUIDO.E_fci) * 1e3
RUIDO["err_con_mHa"] = (RUIDO.E_con - RUIDO.E_fci) * 1e3
RUIDO["ruido_mHa"] = (RUIDO.E_con - RUIDO.E_sin) * 1e3
RUIDO["sigma_mHa"] = np.hypot(RUIDO.s_sin, RUIDO.s_con) * 1e3
RUIDO["ccsd_mHa"] = (RUIDO.E_ccsd - RUIDO.E_fci).abs() * 1e3   # |E_CCSD - E_FCI|, como el criterio del desafío
print("Errores respecto de FCI (mHa); ruido = E_con - E_sin; sigma = σ de esa diferencia; ccsd = error de CCSD:")
print(RUIDO[["ops", "d", "cz", "err_sin_mHa", "err_con_mHa", "ruido_mHa", "sigma_mHa", "ccsd_mHa"]]
      .to_string(index=False, float_format=lambda v: f"{v:.1f}"))

_full = max(VERSIONES)
_f = RUIDO[RUIDO.ops == _full].set_index("d")
print(f"\nAnsatz completo ({CZ_V[_full]} cz): el ruido suma entre {_f.ruido_mHa.min():.0f} y {_f.ruido_mHa.max():.0f} mHa; "
      f"¿con ruido supera el error de CCSD? " + ", ".join(f"{d}: {'sí' if _f.err_con_mHa[d] > _f.ccsd_mHa[d] else 'no'}"
                                                           for d in DIST_RUIDO if d >= STRETCHED))
_best = RUIDO.loc[RUIDO.groupby("d").err_con_mHa.idxmin(), ["d", "ops", "cz", "err_con_mHa", "ccsd_mHa"]]
print("Mejor versión con ruido por distancia (menor error vs. FCI):")
print(_best.to_string(index=False, float_format=lambda v: f"{v:.1f}"))

# Modelo global de despolarización: E_con ≈ E_sin + (1-F)(E_mezcla - E_sin), F = (1-P_CZ)^cz (1-P_1Q)^n1q y
# E_mezcla = c_I + e_nuc (energía del estado máximamente mezclado). Cota gruesa: el error real es local (2 qubits por cz).
RUIDO["F"] = (1 - P_CZ) ** RUIDO.cz * (1 - P_1Q) ** RUIDO.n1q
_Emix = np.array([float(HAM[d][0].coeffs[~(HAM[d][0].paulis.z | HAM[d][0].paulis.x).any(axis=1)].real.sum()) + HAM[d][1]
                  for d in RUIDO.d])
RUIDO["ruido_modelo_mHa"] = (1 - RUIDO.F) * (_Emix - RUIDO.E_sin) * 1e3

_cols = ["tab:blue", "tab:orange", "tab:green", "tab:red"]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.5, 4))
for d, c in zip(DIST_RUIDO, _cols):
    g = RUIDO[RUIDO.d == d].sort_values("cz")
    ax1.plot(g.cz, g.ruido_mHa, "o-", color=c, label=f"d = {d} Å")
    ax2.plot(g.cz, g.err_con_mHa, "o-", color=c, label=f"d = {d} Å, con ruido")
    ax2.plot(g.cz, g.err_sin_mHa, "x--", color=c, alpha=0.6, label=f"d = {d} Å, sin ruido")
    if d >= STRETCHED:
        ax2.axhline(float(g.ccsd_mHa.iloc[0]), color=c, ls=":", lw=1)
ax1.set(xlabel="cz del circuito", ylabel="E_con − E_sin  (mHa)", title="Costo del ruido vs. número de cz")
ax1.legend(fontsize=7)
ax2.set(xlabel="cz del circuito", ylabel="E − E_FCI  (mHa)", yscale="symlog", title="Error vs. FCI (punteado: error de CCSD)")
ax2.legend(fontsize=6, ncol=2)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(6.5, 4))
ax.plot(D, [REF[d]["e_fci"] for d in D], "k-", lw=1, label="FCI")
for k, g in RUIDO.groupby("ops"):
    ax.plot(g.d, g.E_con, "o", ms=5, label=f"con ruido, {k} op ({g.cz.iloc[0]} cz)")
g = RUIDO[RUIDO.ops == _full]
ax.plot(g.d, g.E_sin, "k^", mfc="none", ms=7, label="sin ruido, ansatz completo")
ax.set(xlabel="d (Å)", ylabel="E (Ha)", title="Con ruido (cz: 0,1 %) vs. sin ruido vs. FCI")
ax.legend(fontsize=6)
plt.tight_layout()
plt.show()

# Relación con los cz: pendiente por el origen (mHa por cz) y comparación con el modelo global.
_sl = {d: float((g.cz * g.ruido_mHa).sum() / (g.cz ** 2).sum()) for d, g in RUIDO.groupby("d")}
print("Costo del ruido por cz (pendiente por el origen, mHa/cz):", {d: round(v, 3) for d, v in _sl.items()})
print(RUIDO[RUIDO.ops == _full][["d", "cz", "F", "ruido_mHa", "ruido_modelo_mHa"]].to_string(
    index=False, float_format=lambda v: f"{v:.3f}"))
for d, g in RUIDO.groupby("d"):
    g = g.sort_values("cz")
    assert g.ruido_mHa.iloc[-1] > g.ruido_mHa.iloc[0] + 3 * g.sigma_mHa.max(), f"el ruido no crece con los cz en d={d}"
print(f"Stand-ins usados: {list(STANDIN_USED)}")
RUIDO

#### Qué decidimos, por qué y qué no funcionó (#11)

**Qué decidimos**

- **Modelo de ruido:** despolarizante de 1e-3 en `cz` y 5e-5 en `sx`, `x`, `rz`, como pide el enunciado. Se carga `ruido.py`
  si existe; si no, el stand-in. Como el circuito ya sale de `transpile_line` (base `cz, sx, x, rz`, línea), no hace falta pasar
  el mapa de acoplamiento al simulador. Sin error de lectura (el enunciado no lo menciona).
- **Misma medición que #6:** reutilizamos `measure_energy` (`SamplerV2` con shots, grupos QWC) y solo cambiamos el simulador
  dentro de `con_ruido()`. El `EstimatorV2` de Aer no muestrea, así que no sirve para medir con ruido.
- **Las rotaciones de base de la medición también cuestan:** el `u` que agrega `measure_energy` se descompone en 5 compuertas
  de 1 qubit, así que se le asignó ese error (≈ 2,5e-4 por qubit, poco frente al `cz`).
- **Matriz densidad** en Aer: con 8 qubits es mucho más rápido que trayectorias (una evolución por binding y muestreo del final).
- **Versiones cortas:** los operadores más importantes (mayor $|\theta^*|$ sobre la curva), reajustados por statevector, que
  está permitido para buscar parámetros. Se mide siempre con shots.

**Qué vimos**

- El error de ruido crece con el número de `cz`: cada `cz` fallido (1e-3) acerca el estado a uno mezclado y la energía medida a
  $E_\text{mezcla}$ (el término identidad del Hamiltoniano, muy por encima del fundamental). Con ~600-700 `cz` el ruido suma
  unos 300 mHa (con el modelo global esperaríamos 430-500: el ruido real es local y pega menos), muy por encima del margen de CCSD
  (9 mHa a 1,8 Å): con este modelo ningún ansatz que llegue a FCI le gana a CCSD; la pendiente es ≈ 0,45 mHa por `cz`.
- Las versiones cortas pagan sesgo del ansatz pero ganan en ruido; el mejor compromiso depende de la distancia (ver la tabla
  de "mejor versión con ruido por distancia").

**Qué no funcionó / límites**

- **Transpilar en línea cambia el layout:** `transpile_line` elige layout y mete swaps, así que el qubit lógico i termina en otro
  físico; hay que llevar el observable a la misma permutación (`_observable_fisico`), verificado contra la energía de #6.
  Sin eso, las energías salían cientos de mHa mal sin ruido.
- El modelo de despolarización **global** da solo el orden de magnitud: el ruido es local.
- No re-optimizamos con ruido (opcional del issue): con cientos de `cz` cada evaluación de energía es cara.
- El nombre de la función que expone `ruido.py` es una suposición; ajustar `_load_ruido_py` cuando llegue el archivo oficial (#3).

### 8.2 Ruido de hardware (FakeFez)

Repetimos **un** punto estirado, $d = 2{,}4$ Å, con el ruido completo de `FakeFez` (backend simulado de IBM, 156 qubits Heron r2):
errores de puertas por qubit/par, T1/T2 y **error de lectura**. Receta:

1. `FakeFez` viene de `qiskit_ibm_runtime.fake_provider`: es una **dependencia nueva** (`qiskit-ibm-runtime`, ver el `%pip` de la celda 0).
2. El circuito se transpila **al backend** (base `cz, sx, x, rz, id`, conectividad heavy-hex, con elección de layout y SWAPs) y recién ahí se cuentan `cz` y profundidad.
3. Se mide con `SamplerV2` de `qiskit_aer` con `NoiseModel.from_backend(FakeFez())`, con la lectura incluida. Es el mismo esquema que #6
   (grupos QWC, σ por varianza muestral): se **reusa** `measure_energy`, cambiando solo cómo se arma y se corre el circuito.
4. No usamos `EstimatorV2` para el número reportado: el de Aer calcula la esperanza exacta sin pasar por la medición, así que **esconde** el error de lectura (lo comprobamos abajo con el estado HF).

In [ ]:
# --- 8.2.0 Backend, modelo de ruido y puente hacia measure_energy ---------------------------------------
import contextlib
from qiskit import ClassicalRegister
from qiskit_aer.noise import NoiseModel
from qiskit_aer.primitives import SamplerV2 as _AerSamplerV2
from qiskit_ibm_runtime.fake_provider import FakeFez

D_FEZ = float(D[int(np.argmin(np.abs(np.asarray(D) - 2.4)))])  # 2,4 Å
SEED_FEZ = SEED + 12_000
FEZ = FakeFez()
FEZ_NOISE = NoiseModel.from_backend(FEZ)  # incluye errores de puerta, relajación térmica y lectura
# method="density_matrix": el circuito usa 8 de los 156 qubits (Aer descarta los inactivos) y 8 qubits en matriz densidad
# son baratos; así los shots salen de la distribución exacta del circuito ruidoso, sin trayectorias.
FEZ_BACKEND_OPTIONS = {"method": "density_matrix", "noise_model": FEZ_NOISE}

_t = FEZ.target
_med = lambda name: float(np.median([p.error for p in _t[name].values() if p is not None and p.error is not None]))
print(f"FakeFez: {FEZ.num_qubits} qubits, base {sorted(_t.operation_names)}.")
print(f"Errores medianos del backend: cz {_med('cz'):.2e}, sx {_med('sx'):.2e}, lectura {_med('measure'):.2e}.")


def to_backend(circ, seeds=(0, 1, 2)):
    """Transpila `circ` (8 qubits) a FakeFez (opt. 3, layout incluido). Devuelve (circuito, cz, profundidad, seed).

    Elige la mejor de varias semillas por número de cz (el transpilador es estocástico)."""
    best = None
    for s in seeds:
        tc = transpile(circ, FEZ, optimization_level=3, seed_transpiler=s)
        cz = tc.count_ops().get("cz", 0)
        if best is None or cz < best[1]:
            best = (tc, cz, tc.depth(), s)
    return best


class _Proxy:
    """Plantilla para measure_energy: expone 8 qubits lógicos pero lleva el circuito real (156 qubits físicos)."""
    def __init__(self, qc):
        self.qc, self.num_qubits, self.num_parameters = qc, 8, qc.num_parameters


class _NoisySampler:
    """Reemplazo de SamplerV2 con el ruido de FakeFez; desenvuelve la plantilla y deja el resto igual."""
    def __init__(self, seed=None, options=None):  # `options` (statevector sin ruido de #6) se ignora: acá va el ruido de FakeFez
        self._s = _AerSamplerV2(seed=seed, options={"backend_options": FEZ_BACKEND_OPTIONS})

    def run(self, pubs):
        (proxy, vals, shots), = pubs
        return self._s.run([(proxy.qc, vals, shots)])


_FEZ_SEED = {}  # semilla de transpilación elegida para cada circuito (se calcula una vez y se reusa en la plantilla)


def _fez_template(circuit_bound):
    """Plantilla de medición en el backend: circuito + U(θ_q,0,λ_q) por qubit (la base de cada grupo) + lectura de 8 bits."""
    key = id(circuit_bound)
    if key not in _FEZ_SEED:
        _FEZ_SEED[key] = to_backend(circuit_bound)[3]
    th, la = ParameterVector("_rot_theta", 8), ParameterVector("_rot_lambda", 8)
    qc = QuantumCircuit(8)
    qc.compose(circuit_bound, inplace=True)
    for q in range(8):
        qc.u(th[q], 0.0, la[q], q)
    qc.add_register(ClassicalRegister(8, "meas"))
    qc.measure(range(8), range(8))
    tc = transpile(qc, FEZ, optimization_level=3, seed_transpiler=_FEZ_SEED[key])
    order = {p: i for i, p in enumerate(tc.parameters)}
    return _Proxy(tc), [order[p] for p in th], [order[p] for p in la]


@contextlib.contextmanager
def con_ruido_fez():
    """Dentro del bloque, measure_energy (de #6) transpila al backend y muestrea con el ruido de FakeFez."""
    g = globals()
    old = g["_measurement_template"], g["SamplerV2"], g["UNITS_PER_GROUP"]
    # Con matriz densidad cada asignación del PUB es una simulación completa del circuito ruidoso (los shots son casi gratis):
    # 1 paquete por grupo en vez de ~2 reduce 2 veces el costo sin cambiar la estadística (σ sigue saliendo de la varianza por grupo).
    g["_measurement_template"], g["SamplerV2"], g["UNITS_PER_GROUP"] = _fez_template, _NoisySampler, 1
    try:
        yield
    finally:
        g["_measurement_template"], g["SamplerV2"], g["UNITS_PER_GROUP"] = old


def medir_fez(circ, op, e_nuc, sigma, seed):
    """Energía ± σ con SamplerV2 + FakeFez (lectura incluida). σ objetivo relajado por SHOTS_SCALE como en #6."""
    with con_ruido_fez():
        return measure_energy(circ, op, e_nuc, target_sigma=sigma / np.sqrt(SHOTS_SCALE), seed=seed,
                              pilot_shots_per_group=PILOT_SHOTS, max_shots=4_000_000)

#### Circuitos: ansatz completo, versiones cortas (1 y 2 operadores) y HF

El ansatz completo tiene cientos de `cz`: con ~4 × 10⁻³ por `cz` el circuito pierde casi toda la señal. Por eso armamos versiones cortas:
nos quedamos con los 1 y 2 parámetros de mayor $|\theta^*|$ (los operadores que más aportan) y los **re-ajustamos por statevector sin ruido**
(buscar parámetros vale de cualquier forma; la energía reportada igual sale de medir). El estado HF (0 operadores, 0 `cz`) aísla el efecto
de la **lectura**. Nota: `restringir` conserva el prefijo sin parámetros (los `X` del HF) y las puertas parametrizadas elegidas.

In [ ]:
# --- 8.2.1 Circuitos del punto d = 2,4 Å -----------------------------------------------------------------
from scipy.optimize import minimize

op_f, e_nuc_f = HAM[D_FEZ]
th_full = THETA[D_FEZ]
_P = list(ansatz.parameters)


def _params_de(inst):
    ps = set()
    for p in inst.operation.params:
        ps |= set(getattr(p, "parameters", ()))
    t = getattr(inst.operation, "time", None)
    return ps | set(getattr(t, "parameters", ()))


def restringir(qc, keep, values):
    """Circuito con solo los parámetros `keep` (ya asignados a `values`); el resto de las puertas parametrizadas se descarta.

    Las puertas fijas (X inicial, los CX de cada rotación de Givens) se conservan: sin sus rotaciones, los CX de cada
    excitación descartada se cancelan entre sí (identidad) y el transpilador los elimina."""
    keep = set(keep)
    out = QuantumCircuit(*qc.qregs, name="corto")
    for inst in qc.data:
        ps = _params_de(inst)
        if ps and not ps <= keep:
            continue
        out.append(inst.operation, inst.qubits, inst.clbits)
    return out.assign_parameters(dict(zip(sorted(keep, key=lambda p: _P.index(p)), values)))


def circuito_corto(n_ops):
    idx = sorted(np.argsort(-np.abs(th_full))[:n_ops])
    keep = [_P[i] for i in idx]
    Hm = op_f.to_matrix()

    def f(x):
        sv = Statevector(restringir(ansatz, keep, x)).data
        return float(np.real(np.vdot(sv, Hm @ sv)))

    r = minimize(f, th_full[idx], method="BFGS")  # statevector sin ruido: solo búsqueda de parámetros
    return restringir(ansatz, keep, np.round(r.x, 6)), idx


_t0 = time.time()
CIRCS_FEZ = {}
CIRCS_FEZ["ansatz completo"] = ansatz.assign_parameters(th_full)
for n_ops in (2, 1):
    CIRCS_FEZ[f"corto ({n_ops} op.)"], _idx = circuito_corto(n_ops)
    print(f"  corto ({n_ops} op.): parámetros {[int(i) for i in _idx]} de {len(_P)}")
hf = QuantumCircuit(8, name="HF")
for q, bit in enumerate(reversed(HF_BITSTRING)):
    if bit == "1":
        hf.x(q)
CIRCS_FEZ["HF (0 op.)"] = hf

E_SV_FEZ = {k: float(Statevector(c).expectation_value(op_f).real) + e_nuc_f for k, c in CIRCS_FEZ.items()}  # solo control
print(f"Circuitos listos en {time.time() - _t0:.1f} s.")

In [ ]:
# --- 8.2.2 Transpilación al backend: cz y profundidad ----------------------------------------------------
_t0 = time.time()
BACKEND_STATS = {}
for k, c in CIRCS_FEZ.items():
    tc, cz, depth, seed = to_backend(c)
    used = sorted({tc.find_bit(q).index for inst in tc.data for q in inst.qubits})
    BACKEND_STATS[k] = dict(cz_backend=cz, depth_backend=depth, cz_linea=cz_count(c), qubits_fisicos=used)
print(pd.DataFrame(BACKEND_STATS).T[["cz_linea", "cz_backend", "depth_backend"]].to_string())
print(f"(layout de 'ansatz completo': qubits físicos {BACKEND_STATS['ansatz completo']['qubits_fisicos']}; "
      f"transpilación en {time.time() - _t0:.1f} s)")

In [ ]:
# --- 8.2.3 Medición con SamplerV2: sin ruido y con FakeFez -----------------------------------------------
SIGMA_FEZ = 3.0e-3  # σ objetivo (Ha) en cada medición; con SHOTS_SCALE < 1 se relaja como en #6
e_fci_f = REF[D_FEZ]["e_fci"]
_t0 = time.time()
rows = []
for j, (k, c) in enumerate(CIRCS_FEZ.items()):
    m0 = measure_energy(c, op_f, e_nuc_f, target_sigma=SIGMA_FEZ / np.sqrt(SHOTS_SCALE), seed=SEED_FEZ + 10 * j,
                        pilot_shots_per_group=PILOT_SHOTS)  # SamplerV2 sin ruido (mismo esquema que #6)
    m1 = medir_fez(c, op_f, e_nuc_f, SIGMA_FEZ, SEED_FEZ + 10 * j + 1)
    rows.append(dict(circuito=k, cz=BACKEND_STATS[k]["cz_backend"], profundidad=BACKEND_STATS[k]["depth_backend"],
                     E_sv=E_SV_FEZ[k], E_sin_ruido=m0["energy"], sigma_sin=m0["sigma"],
                     E_fez=m1["energy"], sigma_fez=m1["sigma"], shots_fez=m1["shots"]))
    print(f"  {k}: {time.time() - _t0:.0f} s")
fez = pd.DataFrame(rows)
fez["err_sin_ruido_mHa"] = (fez.E_sin_ruido - e_fci_f) * 1e3
fez["err_fez_mHa"] = (fez.E_fez - e_fci_f) * 1e3
fez["ruido_mHa"] = (fez.E_fez - fez.E_sin_ruido) * 1e3
_ccsd_f = (REF[D_FEZ]["e_ccsd"] - e_fci_f) * 1e3
print(f"d = {D_FEZ} Å: E_FCI = {e_fci_f:.6f} Ha, E_CCSD - E_FCI = {_ccsd_f:.2f} mHa.")
print(fez[["circuito", "cz", "profundidad", "E_sin_ruido", "sigma_sin", "E_fez", "sigma_fez", "err_sin_ruido_mHa",
           "err_fez_mHa", "ruido_mHa", "shots_fez"]].round({"E_sin_ruido": 6, "E_fez": 6, "sigma_sin": 5, "sigma_fez": 5,
           "err_sin_ruido_mHa": 2, "err_fez_mHa": 1, "ruido_mHa": 1}).to_string(index=False))
print(f"Medición con FakeFez en {time.time() - _t0:.1f} s.")

In [ ]:
# --- 8.2.4 Control: EstimatorV2 de Aer esconde el error de lectura (solo comparación, no se reporta) --------
hf_tc, _, _, _ = to_backend(CIRCS_FEZ["HF (0 op.)"])
op_phys = op_f.apply_layout(hf_tc.layout, hf_tc.num_qubits)
est_fez = EstimatorV2(options={"backend_options": FEZ_BACKEND_OPTIONS, "run_options": {"seed": SEED_FEZ}})
e_est = float(est_fez.run([(hf_tc, op_phys)]).result()[0].data.evs) + e_nuc_f
e_smp = float(fez.loc[fez.circuito == "HF (0 op.)", "E_fez"].iloc[0])
print(f"HF con ruido FakeFez, d = {D_FEZ} Å:  EstimatorV2 -> {e_est:.5f} Ha   SamplerV2 -> {e_smp:.5f} Ha   "
      f"(sin ruido {E_SV_FEZ['HF (0 op.)']:.5f})")
print(f"EstimatorV2 mueve el HF {abs(e_est - E_SV_FEZ['HF (0 op.)']) * 1e3:.1f} mHa; SamplerV2 {abs(e_smp - E_SV_FEZ['HF (0 op.)']) * 1e3:.1f} mHa.")

In [ ]:
# --- 8.2.5 Gráfico: error contra FCI, con y sin ruido, según los cz del backend --------------------------
fig, ax = plt.subplots(figsize=(7, 4))
x = np.arange(len(fez))
w = 0.38
ax.bar(x - w / 2, np.abs(fez.err_sin_ruido_mHa).clip(lower=1e-1), w, yerr=fez.sigma_sin * 1e3, label="SamplerV2 sin ruido")
ax.bar(x + w / 2, np.abs(fez.err_fez_mHa), w, yerr=fez.sigma_fez * 1e3, label="SamplerV2 + FakeFez")
ax.axhline(abs(_ccsd_f), color="k", ls="--", lw=1, label=f"|E_CCSD - E_FCI| = {abs(_ccsd_f):.1f} mHa")
ax.set_yscale("log")
ax.set_xticks(x)
ax.set_xticklabels([f"{c}\n{z} cz" for c, z in zip(fez.circuito, fez.cz)], fontsize=8)
ax.set_ylabel("|E - E_FCI| (mHa)")
ax.set_title(f"H4 a d = {D_FEZ} Å: ruido de FakeFez")
ax.legend(fontsize=8)
fig.tight_layout()
plt.show()

#### Cómo se lee, y la diferencia con `ruido.py` (#11)

* **Qué mide la tabla.** `err_sin_ruido` es el sesgo del circuito (sin ruido físico; queda el σ de los shots) y `err_fez` suma el efecto de FakeFez.
  `ruido_mHa` es la diferencia. Crece con los `cz` **del backend** y no con los de la conectividad lineal: en heavy-hex hacen falta SWAPs extra.
* **Por qué el ansatz exacto no sirve acá.** Con $\epsilon_{cz} \sim 4\times10^{-3}$ y cientos de `cz`, la fidelidad del circuito es
  $\sim e^{-N_{cz}\epsilon}$, de unas pocas decenas de por ciento o menos: el estado se parece a uno mezclado y la energía cae lejos de FCI.
  Los circuitos cortos pierden menos por ruido (menos `cz`), pero con 1-2 operadores dejan cientos de mHa de sesgo sin ruido: en este punto ninguno supera al estado HF
  medido con el mismo ruido. Con este hardware simulado el compromiso expresividad/ruido se resuelve a favor de HF; el ansatz exacto queda peor que HF.
* **Lectura.** Incluso con 0 `cz` (HF) hay error: la lectura mediana (~0,8 %) por qubit voltea bits del bitstring. Se ve con `SamplerV2`, y **no** con
  `EstimatorV2` de Aer (celda 8.2.4).
* **Diferencia con el modelo común (`ruido.py`, #11).** `ruido.py` (no está en `main` todavía, por eso no lo corrimos) es, según el enunciado, un
  modelo **sintético y uniforme**: depolarizante $10^{-3}$ en `cz` y $5\times10^{-5}$ en `sx`, `x`, `rz`, conectividad lineal, sin T1/T2 ni lectura.
  FakeFez es el ruido **calibrado** de un chip real: `cz` ~4× peor (y distinto en cada par), relajación durante las esperas, error de lectura
  por qubit y conectividad heavy-hex (más `cz` por SWAPs). Cuando llegue #11 hay que correr el mismo circuito con los dos modelos y poner los números lado a lado.

### Qué decidimos, por qué y qué no funcionó (8.2)

* **SamplerV2 con matriz densidad.** Los shots salen de la distribución exacta del circuito ruidoso (`method="density_matrix"`); es más rápido que trayectorias
  y no cambia la estadística de muestreo. σ es el error estándar de los shots, no incluye variación entre calibraciones.
* **Reuso de `measure_energy` (#6).** En vez de copiarlo, dentro de `con_ruido_fez()` se reemplazan la plantilla de medición (se transpila a FakeFez con la mejor
  de 3 semillas y se mide solo en 8 bits) y el sampler. Así los grupos QWC, el reparto de shots y σ son los de #6.
* **cz y profundidad.** Se reportan del circuito **sin** rotaciones de medición, transpilado al backend con opt. 3 y la mejor de 3 semillas.
* **Circuitos cortos.** Se eligen por $|\theta^*|$ y se reajustan sin ruido por statevector. Más simple que ADAPT; si #4 cambia la estructura del ansatz hay que revisar `restringir`.
* **EstimatorV2.** Solo como control con HF: esconde la lectura y por eso queda sub-estimado el error.
* **No funcionó / límites.** No probamos mitigación (extrapolación a ruido cero, TREX), que sería el siguiente paso.

### 8.3 Esquemas de medición (#19)

La energía reportada sale de #6: grupos QWC (*sorted insertion*) medidos con `SamplerV2`. Acá buscamos **gastar menos
shots para la misma σ** y lo cuantificamos en tres distancias estiradas (1,8 / 2,4 / 3,0 Å) sobre el mismo estado
$|\psi(\theta^*)\rangle$ que mide #6. Comparamos cuatro esquemas:

| Esquema | Qué es | Circuito de medición |
|---|---|---|
| **QWC #6** | línea de base: *sorted insertion* por $\lvert c_k\rvert$ | `H` / `S†H` por qubit (0 `cz`) |
| **(a) QWC mejorado** | mismo tipo de grupos, pero elegimos la partición que minimiza $K=\sum_g\sqrt{\operatorname{Var}_g}$ entre muchas candidatas (órdenes aleatorios + coloreo de grafos) | igual que #6 (0 `cz`) |
| **(b) Conmutantes (GC)** | grupos de Paulis que conmutan (no necesariamente qubit a qubit) | Clifford que diagonaliza el grupo (sí suma `cz`) |
| **(c) Rotación de base (DF)** | factorización doble (Huggins et al., npj QI 7, 23 (2021)): $H = F_0 + \sum_\ell \tfrac{\lambda_\ell}{2}\big(\sum_{pq} L^{\ell}_{pq} E_{pq}\big)^2$; cada fragmento es diagonal en ocupaciones tras una rotación de orbitales | red de Givens por espín |

**Cómo comparamos.** Para un esquema con fragmentos $g$ (cada uno: un circuito de cambio de base $V_g$ y una tabla
$h_g(x)$ con el valor por shot), con reparto óptimo $N_g\propto\sqrt{\operatorname{Var}_g}$ vale $\sigma = K/\sqrt N$, así que
los **shots para σ = 1 mHa** son $N = (K/10^{-3})^2$. $\operatorname{Var}_g$ es la varianza del valor por shot del fragmento
entero (incluye covarianzas), igual que en #6. Para *diseñar* (elegir particiones, repartir shots y llenar la columna de
shots) usamos las varianzas exactas sobre el statevector: es una decisión de diseño, no una energía. La **energía** de
cada esquema sale de medir con `SamplerV2` y se compara con la de #6.

**Cómo validamos cada esquema.** No confiamos en convenciones de signos: para cada fragmento construimos la matriz
$V_g O_g V_g^\dagger$ (256×256) y exigimos que sea diagonal (error $<10^{-9}$); la tabla $h_g(x)$ es esa diagonal. Además
$\sum_g O_g$ tiene que reproducir el Hamiltoniano de #2 como operador.

In [ ]:
# --- 8.3.1 Datos comunes: distancias, estado, términos de Pauli -------------------------------------
import time
import numpy as np
import pandas as pd
import scipy.linalg
import rustworkx as rx
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit.circuit.library import UnitaryGate
from qiskit.quantum_info import Operator, SparsePauliOp, StabilizerState, Statevector
from qiskit_aer.primitives import SamplerV2
from pyscf import ao2mo, gto, scf

_T0_MS = time.time()
MS_D = [float(D[np.argmin(np.abs(D - x))]) for x in (1.8, 2.4, 3.0)]  # distancias estiradas de la grilla
MS_SIGMA = 1.0e-3            # Ha: σ de referencia para la columna "shots para σ = 1 mHa"
MS_SIGMA_RUN = 2.0e-3        # Ha: σ objetivo de la corrida con SamplerV2 (a escala completa; se relaja con SHOTS_SCALE)
MS_RANDOM_ORDERS = 100       # órdenes aleatorios probados por (a) y (b)
MS_SEED = SEED + 19_000
_MS_DKEY = {d: D[np.argmin(np.abs(D - d))] for d in MS_D}  # clave exacta en HAM/CIRCUIT/MEAS
_NQ = N_QUBITS
_DIM = 2 ** _NQ
_MS_SECTOR_MASK = np.zeros(_DIM, bool)
_MS_SECTOR_MASK[_SECTOR] = True  # N = 4, Sz = 0 (de #6)
_MS_NA = np.array([sum(x >> alpha_qubit(p) & 1 for p in range(N_QUBITS // 2)) for x in range(_DIM)])  # N_α por bitstring
_MS_NB = np.array([sum(x >> beta_qubit(p) & 1 for p in range(N_QUBITS // 2)) for x in range(_DIM)])


def _ms_pauli_data(d):
    """Términos no identidad de H(d): códigos IXYZ, matrices de compatibilidad QWC/GC y momentos exactos sobre ψ(θ*)."""
    op, e_nuc = HAM[_MS_DKEY[d]]
    coeffs = np.asarray(op.coeffs).real
    xs, zs = op.paulis.x, op.paulis.z
    keep = ~((~xs) & (~zs)).all(axis=1)
    c_id = float(coeffs[~keep].sum())
    idx = np.flatnonzero(keep)
    x, z = xs[idx].astype(int), zs[idx].astype(int)
    codes = np.where(x & z, 2, np.where(x == 1, 1, np.where(z == 1, 3, 0)))  # 0=I 1=X 2=Y 3=Z (como #6)
    qwc = ((codes[:, None, :] == 0) | (codes[None, :, :] == 0) | (codes[:, None, :] == codes[None, :, :])).all(axis=2)
    anti = (x @ z.T + z @ x.T) % 2
    comm = anti == 0
    psi = Statevector(CIRCUIT[_MS_DKEY[d]]).data
    phi = np.array([op.paulis[k].to_matrix(sparse=True) @ psi for k in idx])  # P_k |ψ>
    C = (phi.conj() @ phi.T).real            # <P_j P_k> (real para pares que conmutan)
    m = (phi.conj() @ psi).real              # <P_k>
    return dict(op=op, e_nuc=e_nuc, c_id=c_id, idx=idx, coeffs=coeffs[idx], codes=codes, qwc=qwc, comm=comm,
                C=C, m=m, psi=psi)


def _ms_K(P, groups):
    """K = sum_g sqrt(Var_g) exacto (covarianzas incluidas) para una partición en índices locales."""
    K = 0.0
    for g in groups:
        c = P["coeffs"][g]
        K += np.sqrt(max(c @ P["C"][np.ix_(g, g)] @ c - (c @ P["m"][g]) ** 2, 0.0))
    return K


def _ms_insertion(compat, order):
    """Inserción greedy: cada término va al primer grupo con el que es compatible con todos sus miembros."""
    groups, allowed = [], []  # allowed[i][k]: k es compatible con todos los miembros del grupo i
    for k in order:
        for g, ok in zip(groups, allowed):
            if ok[k]:
                g.append(int(k))
                ok &= compat[k]
                break
        else:
            groups.append([int(k)])
            allowed.append(compat[k].copy())
    return groups


def _ms_coloring(compat, strategy):
    """Partición por coloreo greedy del grafo de conflictos (arista = términos incompatibles) con rustworkx."""
    n = compat.shape[0]
    graph = rx.PyGraph()
    graph.add_nodes_from(range(n))
    ii, jj = np.nonzero(np.triu(~compat, 1))
    graph.add_edges_from_no_data(list(zip(ii.tolist(), jj.tolist())))
    col = rx.graph_greedy_color(graph, strategy=strategy)
    groups = {}
    for node, cidx in col.items():
        groups.setdefault(cidx, []).append(int(node))
    return list(groups.values())


def _ms_best_partition(P, compat, seed):
    """Candidatas: sorted insertion, coloreos de rustworkx y órdenes aleatorios. Devuelve (mejor, tabla de candidatas)."""
    rng = np.random.default_rng(seed)
    cands = {"sorted insertion": _ms_insertion(compat, np.argsort(-np.abs(P["coeffs"]), kind="stable"))}
    for name in ("Degree", "Saturation", "IndependentSet"):
        if hasattr(rx.ColoringStrategy, name):
            cands[f"coloreo {name}"] = _ms_coloring(compat, getattr(rx.ColoringStrategy, name))
    best_rand = None
    for _ in range(MS_RANDOM_ORDERS):
        g = _ms_insertion(compat, rng.permutation(len(P["coeffs"])))
        k = _ms_K(P, g)
        if best_rand is None or k < best_rand[0]:
            best_rand = (k, g)
    cands[f"mejor de {MS_RANDOM_ORDERS} órdenes aleatorios"] = best_rand[1]
    rows = [(name, len(g), _ms_K(P, g)) for name, g in cands.items()]
    name = min(rows, key=lambda r: r[2])[0]
    return cands[name], name, rows


_t = time.time()
MS_P = {d: _ms_pauli_data(d) for d in MS_D}
print(f"Datos de Pauli y momentos exactos en {MS_D} Å: {len(MS_P[MS_D[0]]['coeffs'])} términos no identidad "
      f"({time.time() - _t:.1f} s).")

In [ ]:
# --- 8.3.2 Fragmentos medibles: (circuito de cambio de base, tabla por shot) y su validación ----------


def _ms_fragment(circ, O, label):
    """Fragmento medible: V = circ, tabla = diag(V O V†). Exige que V O V† sea diagonal."""
    Vm = Operator(circ).data
    M = Vm @ O @ Vm.conj().T
    off = np.abs(M - np.diag(np.diag(M))).max()
    assert off < 1e-9, f"{label}: V O V† no es diagonal (off = {off:.2e})"
    # ¿El circuito de medición conserva N_α y N_β? Entonces cada shot sigue en el sector N=4, Sz=0 y se puede post-seleccionar.
    keeps = all(np.abs(Vm * (n[:, None] - n[None, :])).max() < 1e-9 for n in (_MS_NA, _MS_NB))
    return dict(circ=circ, table=np.diag(M).real.copy(), cz=cz_count(circ) if circ.size() else 0, keeps_N=keeps)


def _ms_pauli_sum_matrix(P, g):
    return SparsePauliOp(P["op"].paulis[P["idx"][g]], P["coeffs"][g]).to_matrix()


def ms_qwc_fragments(P, groups):
    """Grupos QWC -> fragmentos con rotaciones de un qubit (X: H, Y: S†H)."""
    frags = []
    for g in groups:
        basis = P["codes"][g].max(axis=0)  # en un grupo QWC cada qubit tiene a lo sumo un Pauli no trivial
        circ = QuantumCircuit(_NQ)
        for q, b in enumerate(basis):
            if b == 2:
                circ.sdg(q)
            if b in (1, 2):
                circ.h(q)
        frags.append(_ms_fragment(circ, _ms_pauli_sum_matrix(P, g), "QWC"))
    return frags


def _ms_independent(P, g):
    """Subconjunto independiente (sobre GF(2), vectores simplécticos) de un grupo conmutante."""
    op = P["op"].paulis[P["idx"][g]]
    rows = np.hstack([op.x, op.z]).astype(np.uint8)
    basis, chosen = [], []
    for i, r in enumerate(rows):
        v = r.copy()
        for piv, b in basis:
            if v[piv]:
                v ^= b
        if v.any():
            piv = int(np.flatnonzero(v)[0])
            basis = [(p, b ^ v if b[piv] else b) for p, b in basis] + [(piv, v)]
            chosen.append(i)
    return [op[i].to_label().lstrip("-i") for i in chosen]


def ms_gc_fragments(P, groups):
    """Grupos conmutantes -> Clifford C cuyo estado estabilizador tiene a los generadores del grupo; medimos tras C†."""
    frags = []
    for g in groups:
        gens = _ms_independent(P, g)
        cliff = StabilizerState.from_stabilizer_list(gens, allow_underconstrained=True).clifford
        circ = cliff.to_circuit().inverse()
        frags.append(_ms_fragment(circ, _ms_pauli_sum_matrix(P, g), "GC"))
    return frags


# ---- (c) Factorización doble ----
def _ms_annihilators():
    """a_j densos (256×256) en Jordan-Wigner, qubit j = bit j del índice (little endian, como Qiskit)."""
    out = []
    for j in range(_NQ):
        a = np.zeros((_DIM, _DIM))
        for x in range(_DIM):
            if x >> j & 1:
                a[x ^ (1 << j), x] = (-1) ** bin(x & ((1 << j) - 1)).count("1")
        out.append(a)
    return out


_MS_A = _ms_annihilators()
_MS_NORB = N_QUBITS // 2
# E_pq = sum_σ a†_pσ a_qσ con la convención del contrato (α: alpha_qubit(p), β: beta_qubit(p))
_MS_E = np.array([[sum(_MS_A[f(p)].T @ _MS_A[f(q)] for f in (alpha_qubit, beta_qubit))
                   for q in range(_MS_NORB)] for p in range(_MS_NORB)])


def _ms_one_body(A):
    return np.tensordot(A, _MS_E, axes=([0, 1], [0, 1]))


def _ms_integrals(d):
    """h_pq y (pq|rs) en la base MO de RHF/STO-3G (PySCF), con los signos de MO elegidos para reproducir H de #2."""
    mol = gto.M(atom=h4_geometry(d), basis=BASIS, unit="Angstrom", verbose=0)
    mf = scf.RHF(mol)
    mf.conv_tol = 1e-10
    mf.run()
    H_ref = HAM[_MS_DKEY[d]][0].to_matrix().real
    for signs in np.array(np.meshgrid(*[[1, -1]] * (_MS_NORB - 1))).T.reshape(-1, _MS_NORB - 1):
        Cmo = mf.mo_coeff * np.concatenate([[1], signs])
        h = Cmo.T @ mf.get_hcore() @ Cmo
        eri = ao2mo.restore(1, ao2mo.kernel(mol, Cmo), _MS_NORB)
        Hf = _ms_one_body(h) - 0.5 * _ms_one_body(np.einsum("pqqs->ps", eri))
        for p in range(_MS_NORB):
            for q in range(_MS_NORB):
                Hf += 0.5 * _MS_E[p, q] @ _ms_one_body(eri[p, q])
        if np.abs(Hf - H_ref).max() < 1e-8:
            return h, eri
    print(f"AVISO d={d}: las integrales de PySCF no reproducen el Hamiltoniano de #2 (otra convención); "
          "salteamos DF en esta distancia.")
    return None


def _ms_givens_gate(theta):
    """exp(θ (a†_lo a_hi − a†_hi a_lo)) sobre dos modos adyacentes en JW (sin cuerda de Z entre ellos)."""
    a_lo = np.array([[0, 1], [0, 0]])
    lo, hi = np.kron(np.eye(2), a_lo), np.kron(a_lo, np.diag([1, -1]))  # índice = 2*b_hi + b_lo
    return UnitaryGate(scipy.linalg.expm(theta * (lo.T @ hi - hi.T @ lo)), label="G")


def _ms_orbital_rotation(U):
    """Circuito de R(U) (R a†_q R† = sum_p U_pq a†_p) para cada espín: red de Givens adyacentes + signos (Z)."""
    n = U.shape[0]
    W, factors = U.copy(), []
    for j in range(n - 1):
        for i in range(n - 1, j, -1):
            a, b = W[i - 1, j], W[i, j]
            r = np.hypot(a, b)
            if r < 1e-14:
                continue
            c, s = a / r, b / r
            G = np.eye(n)
            G[i - 1, i - 1], G[i - 1, i], G[i, i - 1], G[i, i] = c, s, -s, c
            W = G @ W
            factors.append((i - 1, np.arctan2(-s, c)))  # G^T = exp(θ A_{i-1,i})
    diag = np.sign(np.diag(W))  # U = G_1^T ... G_m^T D
    circ = QuantumCircuit(_NQ)
    for f in (alpha_qubit, beta_qubit):
        for p in range(n):
            if diag[p] < 0:
                circ.z(f(p))
        for p, th in reversed(factors):
            circ.append(_ms_givens_gate(th), [f(p), f(p + 1)])
    return circ


def ms_df_fragments(d):
    """Factorización doble exacta (sin truncar): F_0 = e_nuc + h', F_ℓ = λ_ℓ/2 (Σ L^ℓ_pq E_pq)^2."""
    ints = _ms_integrals(d)
    if ints is None:
        return None
    h, eri = ints
    e_nuc = HAM[_MS_DKEY[d]][1]
    n = _MS_NORB
    h1 = h - 0.5 * np.einsum("pqqs->ps", eri)
    lam, vec = np.linalg.eigh(eri.reshape(n * n, n * n))
    keep = lam > 1e-10
    parts = [(h1, None)] + [(vec[:, l].reshape(n, n), lam[l]) for l in np.flatnonzero(keep)]
    frags, total = [], np.zeros((_DIM, _DIM))
    for A, lam_l in parts:
        A = 0.5 * (A + A.T)
        eps, U = np.linalg.eigh(A)
        O1 = _ms_one_body(A)
        O = O1 + e_nuc * np.eye(_DIM) if lam_l is None else 0.5 * lam_l * O1 @ O1
        total += O
        circ = _ms_orbital_rotation(U).inverse()  # V = R(U)†: lleva Σ ε_k b†_k b_k a Σ ε_k n_k
        frags.append(_ms_fragment(circ, O, "DF"))
    H_full = HAM[_MS_DKEY[d]][0].to_matrix().real + e_nuc * np.eye(_DIM)
    assert np.abs(total - H_full).max() < 1e-8, "la suma de fragmentos DF no reproduce H"
    return frags

In [ ]:
# --- 8.3.3 Construcción de los cuatro esquemas en 1,8 / 2,4 / 3,0 Å ----------------------------------
_t = time.time()
MS_SCHEMES, MS_CANDS = {}, []
for i, d in enumerate(MS_D):
    P = MS_P[d]
    pos = {int(k): j for j, k in enumerate(P["idx"])}
    base_groups = [[pos[k] for k in g["terms"]] for g in qwc_groups(P["op"])[1]]  # la partición exacta de #6
    qwc_best, qwc_name, rows_a = _ms_best_partition(P, P["qwc"], MS_SEED + 10 * i)
    gc_best, gc_name, rows_b = _ms_best_partition(P, P["comm"], MS_SEED + 10 * i + 1)
    MS_CANDS += [dict(d=d, esquema="(a) QWC", candidata=r[0], grupos=r[1], K_Ha=r[2]) for r in rows_a]
    MS_CANDS += [dict(d=d, esquema="(b) GC", candidata=r[0], grupos=r[1], K_Ha=r[2]) for r in rows_b]
    MS_SCHEMES[d] = {
        "QWC #6": (P["e_nuc"] + P["c_id"], ms_qwc_fragments(P, base_groups), "—"),
        "(a) QWC mejorado": (P["e_nuc"] + P["c_id"], ms_qwc_fragments(P, qwc_best), qwc_name),
        "(b) conmutantes + Clifford": (P["e_nuc"] + P["c_id"], ms_gc_fragments(P, gc_best), gc_name),
    }
    _df = ms_df_fragments(d)
    if _df is not None:
        MS_SCHEMES[d]["(c) rotación de base (DF)"] = (0.0, _df, "factorización doble exacta")
print(f"Esquemas construidos y validados (V O V† diagonal, suma = H) en {time.time() - _t:.1f} s.")
print(pd.DataFrame(MS_CANDS).assign(K_Ha=lambda t: t.K_Ha.round(4)).to_string(index=False))

In [ ]:
# --- 8.3.4 Shots para σ = 1 mHa (exacto) y energía medida con SamplerV2 -------------------------------


def _ms_exact(psi, frags):
    """(media, varianza por shot) exactas de cada fragmento sobre ψ, y fracción de probabilidad en el sector N=4, Sz=0."""
    out = []
    for f in frags:
        p = np.abs(Statevector(psi).evolve(f["circ"]).data) ** 2
        mu = p @ f["table"]
        out.append((mu, p @ (f["table"] - mu) ** 2))
    return np.array(out)


def ms_measure(circuit_bound, c0, frags, var_exact, total_shots, seed, postselect=False):
    """Energía con SamplerV2: N_g ∝ sqrt(Var_g) shots para el fragmento g, una llamada de SamplerV2 por fragmento con
    semilla propia (sacada de SeedSequence) para que los fragmentos sean independientes. Con postselect descarta shots
    fuera de N=4, Sz=0 (solo tiene sentido si el circuito de medición conserva N_α y N_β, como en DF)."""
    sq = np.sqrt(np.maximum(var_exact, 1e-16))
    n_g = np.maximum(MIN_GROUP_SHOTS, np.round(total_shots * sq / sq.sum())).astype(int)
    seeds = np.random.SeedSequence(int(seed)).generate_state(len(frags)) % (2 ** 31 - 1)
    counts = np.zeros((len(frags), _DIM))
    for g, f in enumerate(frags):
        qc = circuit_bound.compose(f["circ"])
        qc.measure_all()  # sin transpilar: Aer ejecuta h, s, cx, swap y unitary nativamente
        arr = SamplerV2(seed=int(seeds[g])).run([(qc, None, int(n_g[g]))]).result()[0].data.meas.array.astype(np.int64)
        ints = np.zeros(arr.shape[0], dtype=np.int64)
        for byte in range(arr.shape[1]):
            ints = (ints << 8) | arr[:, byte]
        counts[g] = np.bincount(ints, minlength=_DIM)
    raw = counts.sum()
    if postselect:
        counts[:, ~_MS_SECTOR_MASK] = 0
    n = counts.sum(axis=1)
    tables = np.array([f["table"] for f in frags])
    means = (counts * tables).sum(axis=1) / n
    vars_ = (counts * (tables - means[:, None]) ** 2).sum(axis=1) / (n - 1)
    return dict(energy=float(c0 + means.sum()), sigma=float(np.sqrt((vars_ / n).sum())), shots=int(raw),
                kept=float(n.sum() / raw))


_t = time.time()
_sig_run = MS_SIGMA_RUN / np.sqrt(SHOTS_SCALE)
MS_ROWS = []
for i, d in enumerate(MS_D):
    dk = _MS_DKEY[d]
    for j, (name, (c0, frags, _)) in enumerate(MS_SCHEMES[d].items()):
        ex = _ms_exact(MS_P[d]["psi"], frags)
        K = float(np.sqrt(ex[:, 1]).sum())
        e_exact = c0 + ex[:, 0].sum()
        assert abs(e_exact - E_SV[dk]) < 1e-8, f"{name} d={d}: la suma de fragmentos no da <H>"
        n_keep = sum(f["keeps_N"] for f in frags)
        if name == "QWC #6":  # la línea de base ya está medida en #6: no la repetimos
            r = dict(energy=MEAS[dk]["energy"], sigma=MEAS[dk]["sigma"], shots=MEAS[dk]["shots"], kept=np.nan)
        else:
            r = ms_measure(CIRCUIT[dk], c0, frags, ex[:, 1], int(np.ceil((K / _sig_run) ** 2)), MS_SEED + 100 * i + j,
                           postselect=n_keep == len(frags))
        MS_ROWS.append(dict(
            d=d, esquema=name, circuitos=len(frags), K_Ha=K, shots_1mHa=(K / MS_SIGMA) ** 2,
            cz_med_max=max(f["cz"] for f in frags), cz_med_total=sum(f["cz"] for f in frags),
            postsel_N_Sz="sí (todos)" if n_keep == len(frags) else f"{n_keep}/{len(frags)} circuitos",
            E=r["energy"], sigma=r["sigma"], shots=r["shots"], kept=r["kept"],
            z_vs_6=(r["energy"] - MEAS[dk]["energy"]) / np.hypot(r["sigma"], MEAS[dk]["sigma"]),
            z_vs_sv=(r["energy"] - E_SV[dk]) / r["sigma"]))
ms = pd.DataFrame(MS_ROWS)
ms["ahorro_vs_6"] = [ms.shots_1mHa[(ms.d == d) & (ms.esquema == "QWC #6")].iloc[0] / s for d, s in zip(ms.d, ms.shots_1mHa)]
print(f"Corridas con SamplerV2 en {time.time() - _t:.1f} s (σ objetivo de la corrida = {_sig_run * 1e3:.2f} mHa).")
print(ms[["d", "esquema", "circuitos", "shots_1mHa", "ahorro_vs_6", "cz_med_max", "cz_med_total", "postsel_N_Sz"]]
      .assign(shots_1mHa=lambda t: t.shots_1mHa.map(lambda v: f"{v:.2e}"), ahorro_vs_6=lambda t: t.ahorro_vs_6.round(2))
      .to_string(index=False))
print()
print(ms[["d", "esquema", "E", "sigma", "shots", "kept", "z_vs_6", "z_vs_sv"]]
      .assign(sigma_mHa=lambda t: (t.sigma * 1e3).round(3), E=lambda t: t.E.round(5),
              z_vs_6=lambda t: t.z_vs_6.round(2), z_vs_sv=lambda t: t.z_vs_sv.round(2))
      .drop(columns="sigma").to_string(index=False))
print(f"#6 en esas distancias: " + ", ".join(f"{d}: {MEAS[_MS_DKEY[d]]['energy']:.5f} ± {MEAS[_MS_DKEY[d]]['sigma'] * 1e3:.2f} mHa"
                                            for d in MS_D))
assert (ms.z_vs_6.abs() < 3).all(), "algún esquema se aparta de #6 en más de 3σ"

In [ ]:
# --- 8.3.5 Figura: shots para σ = 1 mHa por esquema ---------------------------------------------------
fig, ax = plt.subplots(figsize=(7, 3.6))
names = list(dict.fromkeys(n for d in MS_D for n in MS_SCHEMES[d]))
w = 0.8 / len(names)
for j, name in enumerate(names):
    vals = [ms.shots_1mHa[(ms.d == d) & (ms.esquema == name)].max() for d in MS_D]  # NaN si el esquema falta
    ax.bar(np.arange(len(MS_D)) + (j - (len(names) - 1) / 2) * w, vals, w, label=name)
ax.set_yscale("log")
ax.set_xticks(range(len(MS_D)), [f"{d} Å" for d in MS_D])
ax.set_ylabel("shots para σ = 1 mHa")
ax.set_title("Esquemas de medición sobre ψ(θ*)")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()
print(f"Tiempo total de la sección 8.3: {time.time() - _T0_MS:.0f} s")

#### Conclusión

Sobre el mismo estado que mide #6 (los shots para σ = 1 mHa no dependen de `SHOTS_SCALE`):

- **(c) Rotación de base (DF)** es el gran ganador: 10 circuitos y $3{,}7\times10^5$ shots a 1,8 Å (**6×** menos que los
  $2{,}3\times10^6$ de QWC #6; coincide con los ~$3{,}4\times10^5$ del análisis de la rama de referencia), y el ahorro crece
  al estirar (~37× a 2,4 Å, ~300× a 3,0 Å). Cuesta como mucho **24 `cz`** extra por circuito (red de Givens, 6 por espín) y,
  como la rotación conserva $N_\alpha$ y $N_\beta$, **permite post-seleccionar** por N y $S_z$ (útil para #13).
- **(b) Conmutantes + Clifford**: 9–13 circuitos y 2,2–3,4× menos shots, pero hasta 69–81 `cz` por circuito y sin
  post-selección. Con ruido no conviene frente a DF.
- **(a) QWC mejorado**: entre ~100 particiones candidatas la mejor apenas gana 0–5 %: *sorted insertion* ya está cerca del óptimo.
- Las energías medidas con (a), (b) y (c) coinciden con la de #6 dentro de 3σ combinada (los valores de $z$
  de la tabla de 8.3.4 cambian con `SHOTS_SCALE` y la semilla; los apartamientos más grandes vinieron del propio punto de #6) y todas
  están a < 2σ de $E_\text{sv}$.

**Recomendación:** proponer DF para el resultado principal de #6 (mismo σ con 6× menos shots en 1,8 Å, donde el margen
es más chico). Ojo: sus `cz` de medición hay que reportarlos aparte de los del ansatz, y con ruido pesan.

#### Qué decidimos, por qué y qué no funcionó

- **Varianzas exactas para diseñar, shots para medir.** La partición de (a)/(b), el reparto $N_g\propto\sqrt{\operatorname{Var}_g}$
  y la columna "shots para σ = 1 mHa" usan la varianza exacta del valor por shot de cada fragmento sobre el statevector
  (es diseño, como buscar parámetros). La energía de cada esquema sale de `SamplerV2` con shots reales, y la σ medida se
  estima con las varianzas muestrales.
- **Validación sin convenciones.** Cada fragmento se acepta solo si $V_g O_g V_g^\dagger$ es diagonal; la tabla por shot es
  esa diagonal. Así los signos de Pauli, el orden de bits y la dirección de la rotación de orbitales quedan verificados
  numéricamente (además, $\sum_g O_g = H$ y $\sum_g \langle O_g\rangle = E_\text{sv}$).
- **Integrales para DF.** Las calculamos con PySCF (RHF/STO-3G) y elegimos los signos de los orbitales que reproducen el
  Hamiltoniano de #2 como operador; si #2 usa otra convención, la búsqueda de signos lo absorbe (y si no alcanza, avisa
  y saltea DF en esa distancia en lugar de cortar el notebook). La factorización es exacta (no truncamos autovalores > 1e-10), así que no hay sesgo.
- **Rotaciones de Givens.** Descomponemos $U$ en rotaciones de modos adyacentes (que en JW no tienen cuerda de Z dentro
  del bloque α o β) más signos (`Z`); cada rotación es un `UnitaryGate` de 2 qubits que `transpile_line` deja en 2 `cz`.
- **Diagonalización Clifford.** Generadores independientes del grupo (eliminación en GF(2)) → `StabilizerState.from_stabilizer_list`
  (completa los estabilizadores que faltan) → medimos tras $C^\dagger$. Es la síntesis genérica de Qiskit: no optimizamos
  esos `cz` (una síntesis a medida bajaría algo, pero no cambia la conclusión frente a DF).
- **Una llamada de `SamplerV2` por fragmento, con semilla propia.** Juntar todos los circuitos en un job (como #6 con los
  bindings) obligaba a repetir PUBs y costaba ~40 ms por PUB; con PUBs de distinto número de shots Aer reutiliza la
  semilla y correlaciona los fragmentos (lo mismo que vio #6). Tampoco transpilamos para Aer: armar el `target` del
  backend en cada `transpile` dominaba el tiempo.
- **σ de la corrida = 2 mHa** (a escala completa) y no 1 mHa, para que la sección sea rápida: la comprobación "dentro de 2σ
  de #6" no necesita más, y la relación $\sigma = K/\sqrt N$ ya la validó #6.
- **No funcionó / no ayudó:** los coloreos greedy de grafos (Degree, Saturation, IndependentSet) dan menos grupos en
  algunos casos pero **más** $K$ que *sorted insertion*: minimizar el número de grupos no es minimizar los shots.
  La línea de base QWC #6 no se vuelve a medir: usamos directamente la energía y σ de #6.